# Expanded all-HYGD AQPR endpoints

**Public role:** Core expanded reliability-endpoint notebook.

**Relation to the manuscript:** Supports saliency sanity, insertion/deletion, perturbation, matched-random sensitivity, and patch-occlusion safeguards.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `15_aqpr_expanded_core_audit_all_hygd_v36_robustness_reviewfix.ipynb`


# Notebook 15 — Expanded AQPR core audit on HYGD — v3.6 reviewer-robustness-ready

This version replaces the earlier reviewer-efficient 60-image audit with an expanded, resumable audit designed to address the most recent reviewer concerns.

Main changes:

1. Uses **all eligible HYGD strict-valid images** as the primary audit cohort whenever the required image and OD/OC proxy inputs are available.
2. Keeps execution **resumable** for long saliency, deletion/insertion, and perturbation runs.
3. Adds **image-level and patient-level bootstrap confidence intervals** for saliency sanity, deletion/insertion faithfulness, and perturbation sensitivity.
4. Adds **multi-seed balanced subset stability** computed from the expanded all-HYGD outputs, so the original 60-image concern is no longer the primary evidence base.
5. Adds **insertion/deletion AUC summaries** and additional perturbation controls to test whether conclusions depend on the blur-only design.
6. Produces reviewer-facing, manuscript-facing, and supplement-facing CSV outputs without mentioning notebooks in the manuscript.

Run this after the finalized HYGD/SMDG forensic-quality audit has generated the unified HYGD manifest.


## v3.6 update

This version keeps the expanded all-HYGD audit as the primary evidence and adds reviewer-requested robustness outputs:
matched-random mask setting ablations, full insertion/deletion curve exports, a secondary occlusion-based attribution check, and a final Notebook 16 handoff.


In [ ]:
# Cell 1: Setup, Drive mount, constants, and output folders
#
# This cell is intended for Google Colab. For a local runtime, set:
#   os.environ["AQPR_PROJECT_ROOT"] = "/path/to/AI_Glaucoma/journal_pipeline"

from pathlib import Path
import os
import re
import sys
import json
import math
import time
import random
import shutil
import warnings
import subprocess
import hashlib
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------------------------------------------------------
# Mount Google Drive explicitly when running in Colab.
# -------------------------------------------------------------------------
IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=True)
except ModuleNotFoundError:
    IN_COLAB = False
    print("google.colab is not available. Assuming a local runtime.")

if IN_COLAB:
    PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
else:
    PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT was not found: {PROJECT_ROOT}\n"
        "If you are in Colab, check that Drive is mounted and the folder is exactly "
        "MyDrive/AI_Glaucoma/journal_pipeline. If you are local, set AQPR_PROJECT_ROOT."
    )

# -------------------------------------------------------------------------
# Constants
# -------------------------------------------------------------------------
NOTEBOOK_NAME = "15_aqpr_expanded_core_audit_all_hygd_v31"
SEED = 411

PRIMARY_MODEL_NAME = "EfficientNetB0"
SECONDARY_MODEL_NAME = "ConvNeXtTiny"
MODEL_NAMES = [PRIMARY_MODEL_NAME, SECONDARY_MODEL_NAME]

# Expanded reviewer-ready audit: use all eligible HYGD strict-valid images as primary cohort.
AUDIT_COHORT_MODE = "all_hygd_strict_valid"
MAX_IMAGES_PER_LABEL_QUALITY = None
SUBSET_STABILITY_SEEDS = [411, 512, 613, 714, 815]
SUBSET_STABILITY_IMAGES_PER_LABEL_QUALITY = 10

# Increased IG steps for the expanded audit.
# Raise to 24 if runtime allows; 16 is a pragmatic all-HYGD default.
IG_STEPS = 16
BOOTSTRAP_REPEATS = 5000
BOOTSTRAP_SEED = 411

# Execution switches.
RUN_SALIENCY_SANITY = True
RUN_DELETION_FAITHFULNESS = True
RUN_PERTURBATION_SENSITIVITY = True
RUN_OUTSIDE_OD_CONSERVATIVE_INFERENCE = False

# Runtime / image settings.
IMAGE_SIZE = 224
SAVE_EVERY_N_TASKS = 25
RANDOMIZATION_CONDITIONS = [
    "original",
    "randomized_classifier_head",
    "randomized_last_block_plus_head",
]
DELETION_FRACTIONS = [0.00, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
INSERTION_FRACTIONS = DELETION_FRACTIONS.copy()
PERTURBATION_GRID = [
    {"setting_id": "sigma4_dil0",  "sigma": 4,  "dilation": 0},
    {"setting_id": "sigma4_dil7",  "sigma": 4,  "dilation": 7},
    {"setting_id": "sigma4_dil14", "sigma": 4,  "dilation": 14},
    {"setting_id": "sigma8_dil0",  "sigma": 8,  "dilation": 0},
    {"setting_id": "sigma8_dil7",  "sigma": 8,  "dilation": 7},
    {"setting_id": "sigma8_dil14", "sigma": 8,  "dilation": 14},
    {"setting_id": "sigma12_dil0", "sigma": 12, "dilation": 0},
    {"setting_id": "sigma12_dil7", "sigma": 12, "dilation": 7},
    {"setting_id": "sigma12_dil14", "sigma": 12, "dilation": 14},
]
PERTURBATION_METHODS = ["blur", "mean_fill", "gaussian_noise"]
QUALITY_ORDER = ["low", "medium", "high"]

# Conservative mismatch guard for checkpoint loading.
MAX_ALLOWED_MISSING_KEYS = 30
MAX_ALLOWED_UNEXPECTED_KEYS = 30

RESULTS_ROOT = PROJECT_ROOT / "results"
RESULTS_15_DIR = RESULTS_ROOT / NOTEBOOK_NAME
TABLE_DIR = RESULTS_15_DIR / "tables"
FIGURE_DIR = RESULTS_15_DIR / "figures"
FIGURE_DATA_DIR = RESULTS_15_DIR / "figure_data"
ARRAY_DIR = RESULTS_15_DIR / "arrays"
QC_DIR = RESULTS_15_DIR / "qc"

for d in [RESULTS_15_DIR, TABLE_DIR, FIGURE_DIR, FIGURE_DATA_DIR, ARRAY_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

# -------------------------------------------------------------------------
# Dependencies
# -------------------------------------------------------------------------
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except Exception as exc:
    TORCH_AVAILABLE = False
    DEVICE = "cpu"
    raise RuntimeError(f"PyTorch is required for Notebook 15: {type(exc).__name__}: {exc}")

try:
    import timm
    TIMM_AVAILABLE = True
except Exception:
    print("timm was not found. Installing timm quietly...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
    import timm
    TIMM_AVAILABLE = True

try:
    from PIL import Image, ImageFilter
    PIL_AVAILABLE = True
except Exception as exc:
    PIL_AVAILABLE = False
    raise RuntimeError(f"PIL is required for Notebook 15: {type(exc).__name__}: {exc}")

try:
    from torchvision import transforms
    TORCHVISION_AVAILABLE = True
except Exception as exc:
    TORCHVISION_AVAILABLE = False
    raise RuntimeError(f"torchvision is required for Notebook 15: {type(exc).__name__}: {exc}")

try:
    from scipy.ndimage import binary_dilation
    SCIPY_NDIMAGE_AVAILABLE = True
except Exception:
    SCIPY_NDIMAGE_AVAILABLE = False

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 1",
    "status": "PASS",
    "in_colab": bool(IN_COLAB),
    "project_root": str(PROJECT_ROOT),
    "results_root": str(RESULTS_ROOT),
    "results_15_dir": str(RESULTS_15_DIR),
    "device": str(DEVICE),
    "torch_available": bool(TORCH_AVAILABLE),
    "timm_available": bool(TIMM_AVAILABLE),
    "pil_available": bool(PIL_AVAILABLE),
    "seed": int(SEED),
    "model_names_requested": MODEL_NAMES,
    "audit_cohort_mode": AUDIT_COHORT_MODE,
    "max_images_per_label_quality": (
        None if MAX_IMAGES_PER_LABEL_QUALITY is None else int(MAX_IMAGES_PER_LABEL_QUALITY)
    ),
    "subset_stability_seeds": SUBSET_STABILITY_SEEDS,
    "subset_stability_images_per_label_quality": int(SUBSET_STABILITY_IMAGES_PER_LABEL_QUALITY),
    "ig_steps": int(IG_STEPS),
}

with open(RESULTS_15_DIR / "notebook15_cell01_setup_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


In [ ]:
# Cell 2: Registry-first source discovery and input audit
#
# Critical correction:
# - Checkpoints are selected from notebook08_aqpr_model_registry_for_xai.csv first.
# - Global checkpoint search is only a last-resort fallback.

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

def norm_text(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")

def canonical_model_name(x):
    s = norm_text(x)
    aliases = {
        "efficientnetb0": "EfficientNetB0",
        "efficientnet_b0": "EfficientNetB0",
        "tf_efficientnet_b0": "EfficientNetB0",
        "convnexttiny": "ConvNeXtTiny",
        "convnext_tiny": "ConvNeXtTiny",
        "densenet121": "DenseNet121",
        "dense_net_121": "DenseNet121",
        "swintiny": "SwinTiny",
        "swin_tiny": "SwinTiny",
        "swin_tiny_patch4_window7_224": "SwinTiny",
        "resnet18": "ResNet18",
        "resnet_18": "ResNet18",
    }
    return aliases.get(s, str(x))

def safe_read_csv(path_like):
    if path_like is None:
        return pd.DataFrame()
    path = Path(path_like)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as exc:
        print(f"Could not read {path}: {type(exc).__name__}: {exc}")
        return pd.DataFrame()

def find_column(df, candidates=None, contains_any=None, contains_all=None):
    if df is None or df.empty:
        return None
    candidates = candidates or []
    norm_to_col = {norm_text(c): c for c in df.columns}
    for cand in candidates:
        key = norm_text(cand)
        if key in norm_to_col:
            return norm_to_col[key]
    contains_any = [norm_text(x) for x in (contains_any or [])]
    contains_all = [norm_text(x) for x in (contains_all or [])]
    for c in df.columns:
        nc = norm_text(c)
        ok_any = any(tok in nc for tok in contains_any) if contains_any else True
        ok_all = all(tok in nc for tok in contains_all) if contains_all else True
        if ok_any and ok_all:
            return c
    return None

def first_existing(patterns, root=RESULTS_ROOT):
    hits = []
    for pattern in patterns:
        hits.extend(sorted(root.glob(pattern)))
    hits = [p for p in hits if p.exists() and p.is_file()]
    return hits[0] if hits else None

def all_existing(patterns, root=RESULTS_ROOT, max_hits=1000):
    hits = []
    for pattern in patterns:
        hits.extend(sorted(root.glob(pattern)))
    out = []
    seen = set()
    for p in hits:
        if p.exists() and p.is_file() and str(p) not in seen:
            out.append(p)
            seen.add(str(p))
    return out[:max_hits]

def path_from_value(x):
    if pd.isna(x):
        return None
    p = Path(str(x))
    return p if p.exists() else None

def score_checkpoint_fallback(path, model_name):
    # Fallback only. Strongly penalize unrelated model names.
    s = norm_text(str(path))
    model_s = norm_text(model_name)
    score = 0
    if model_s in s:
        score += 100
    if canonical_model_name(model_name) == "EfficientNetB0" and ("effb0" in s or "efficient" in s):
        score += 80
    if canonical_model_name(model_name) == "ConvNeXtTiny" and ("convnext" in s or "conv_next" in s):
        score += 80
    if "best" in s:
        score += 20
    if "checkpoint" in s or "ckpt" in s:
        score += 10
    if "notebook08" in s or "xai" in s:
        score += 5
    return score

SOURCE_PATTERNS = {
    "hygd_unified_base": [
        "14_hygd_smdg_domain_shift_audit/tables/notebook14_hygd_unified_aqpr_base_manifest.csv",
        "**/notebook14_hygd_unified_aqpr_base_manifest.csv",
    ],
    "hygd_quality_summary": [
        "14_hygd_smdg_domain_shift_audit/tables/notebook14_hygd_quality_stratified_aqpr_summary.csv",
        "**/notebook14_hygd_quality_stratified_aqpr_summary.csv",
    ],
    "outside_od_conservative_manifest": [
        "14_hygd_smdg_domain_shift_audit/tables/notebook14_outside_od_conservative_mask_sensitivity_manifest.csv",
        "**/notebook14_outside_od_conservative_mask_sensitivity_manifest.csv",
    ],
    "model_registry_xai": [
        "08_anatomical_explainability_validation/notebook08_aqpr_model_registry_for_xai.csv",
        "**/notebook08_aqpr_model_registry_for_xai.csv",
    ],
    "architecture_results": [
        "07_hygd_glaucoma_classifier_benchmark/notebook07_architecture_screening_results_consolidated.csv",
        "**/notebook07_architecture_screening_results_consolidated.csv",
    ],
}

registry_rows = []
selected_paths = {}
for role, patterns in SOURCE_PATTERNS.items():
    p = first_existing(patterns)
    selected_paths[role] = p
    df = safe_read_csv(p) if p else pd.DataFrame()
    registry_rows.append({
        "role": role,
        "selected": bool(p is not None),
        "selection_method": "glob_exact_or_recursive",
        "path": str(p) if p else "",
        "file_name": p.name if p else "",
        "n_rows": int(len(df)),
        "n_columns": int(df.shape[1]) if not df.empty else 0,
        "columns": ";".join(map(str, df.columns[:100])) if not df.empty else "",
    })

# Registry-first model checkpoint discovery.
xai_registry_path = selected_paths.get("model_registry_xai")
xai_registry = safe_read_csv(xai_registry_path)

checkpoint_hits = all_existing(["**/*.pt", "**/*.pth", "**/*.ckpt"], RESULTS_ROOT, max_hits=1000)

if not xai_registry.empty:
    model_col = find_column(xai_registry, ["model_name", "architecture", "model", "classifier"], contains_any=["model", "architecture"])
    ckpt_col = find_column(xai_registry, ["checkpoint_path", "best_checkpoint_path", "model_checkpoint_path", "path"], contains_any=["checkpoint", "path"])
    target_col = find_column(xai_registry, ["target_layer_name", "target_layer", "xai_target_layer"], contains_any=["target", "layer"])
else:
    model_col = ckpt_col = target_col = None

for model_name in MODEL_NAMES:
    canonical = canonical_model_name(model_name)
    chosen_path = None
    selection_method = "not_found"
    target_layer = ""

    if not xai_registry.empty and model_col is not None and ckpt_col is not None:
        tmp = xai_registry.copy()
        tmp["_canonical_model"] = tmp[model_col].apply(canonical_model_name)
        block = tmp[tmp["_canonical_model"].eq(canonical)].copy()
        if not block.empty:
            # Prefer rows whose checkpoint path exists.
            block["_checkpoint_exists_runtime"] = block[ckpt_col].apply(lambda x: path_from_value(x) is not None)
            block = block.sort_values("_checkpoint_exists_runtime", ascending=False)
            for _, r in block.iterrows():
                p = path_from_value(r.get(ckpt_col, ""))
                if p is not None:
                    chosen_path = p
                    selection_method = "notebook08_xai_registry_checkpoint_path"
                    if target_col is not None and pd.notna(r.get(target_col, np.nan)):
                        target_layer = str(r.get(target_col))
                    break

    # Last-resort fallback, only if registry failed.
    if chosen_path is None:
        scored = [(score_checkpoint_fallback(p, canonical), p) for p in checkpoint_hits]
        scored = sorted([x for x in scored if x[0] > 0], key=lambda x: (-x[0], str(x[1])))
        if scored:
            chosen_path = scored[0][1]
            selection_method = f"global_fallback_score_{scored[0][0]}"

    registry_rows.append({
        "role": f"checkpoint_{canonical}",
        "selected": bool(chosen_path is not None),
        "selection_method": selection_method,
        "path": str(chosen_path) if chosen_path else "",
        "file_name": chosen_path.name if chosen_path else "",
        "n_rows": np.nan,
        "n_columns": np.nan,
        "columns": "",
        "target_layer_name": target_layer,
    })

source_registry = pd.DataFrame(registry_rows)
SOURCE_REGISTRY_CSV = TABLE_DIR / "notebook15_source_registry.csv"
source_registry.to_csv(SOURCE_REGISTRY_CSV, index=False)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 2",
    "status": "PASS",
    "source_registry_csv": str(SOURCE_REGISTRY_CSV),
    "xai_registry_path": str(xai_registry_path) if xai_registry_path else "",
    "xai_registry_available": bool(not xai_registry.empty),
    "n_roles": int(len(source_registry)),
    "n_selected_roles": int(source_registry["selected"].sum()),
    "checkpoint_selection_policy": "registry_first_then_global_fallback",
}

with open(RESULTS_15_DIR / "notebook15_cell02_source_registry_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(source_registry.to_string(index=False))
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 3: Expanded all-HYGD strict-valid audit cohort and multi-seed subset manifest
#
# Primary reviewer-ready change:
# - The main cohort is no longer a 60-image subset.
# - It uses all eligible HYGD images with strict-valid OD/OC proxy, binary label,
#   quality tertile, and existing image path.
# - Balanced 60-image subset assignments are still generated only as stability
#   checks after the expanded cohort results are available.

def get_source_path(role):
    block = source_registry[source_registry["role"].astype(str).eq(role)].copy()
    if block.empty or not bool(block.iloc[0]["selected"]):
        return None
    p = Path(str(block.iloc[0]["path"]))
    return p if p.exists() else None

def normalize_quality(x):
    x = str(x).strip().lower()
    mapping = {
        "0": "low", "1": "medium", "2": "high",
        "low_quality": "low", "medium_quality": "medium", "high_quality": "high",
        "q1": "low", "q2": "medium", "q3": "high",
    }
    return mapping.get(x, x)

def stable_int_hash(*parts, modulo=10_000_000):
    s = "||".join(map(str, parts))
    return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:12], 16) % modulo

base_path = get_source_path("hygd_unified_base")
if base_path is None:
    raise FileNotFoundError("HYGD unified base manifest from the previous audit was not found. Run the HYGD/SMDG audit first.")

base = safe_read_csv(base_path)

image_col = find_column(base, ["image_id", "sample_id", "fundus_id"], contains_any=["image"])
patient_col = find_column(base, ["patient_id", "subject_id", "patient"], contains_any=["patient"])
label_col = find_column(base, ["gon_binary", "label", "y_true", "target"], contains_any=["gon", "label", "target"])
quality_col = find_column(base, ["quality_tertile", "quality_bin", "quality_group"], contains_any=["quality"])
path_col = find_column(base, ["image_path", "fundus_path", "path"], contains_any=["path"])

missing = [k for k, v in {
    "image_col": image_col,
    "patient_col": patient_col,
    "label_col": label_col,
    "quality_col": quality_col,
    "path_col": path_col,
}.items() if v is None]
if missing:
    raise RuntimeError(f"Missing required HYGD columns: {missing}. Available columns: {list(base.columns)}")

df = base.copy()
df["image_id"] = df[image_col].astype(str)
df["patient_id"] = df[patient_col].astype(str)
df["gon_binary"] = pd.to_numeric(df[label_col], errors="coerce")
df["quality_tertile"] = df[quality_col].apply(normalize_quality)
df["image_path"] = df[path_col].astype(str)

if "strict_anatomical_proxy_valid" in df.columns:
    df["strict_anatomical_proxy_valid"] = df["strict_anatomical_proxy_valid"].astype(str).str.lower().isin(["true", "1", "yes", "y", "t"])
else:
    df["strict_anatomical_proxy_valid"] = True

df["image_path_exists"] = df["image_path"].apply(lambda x: Path(str(x)).exists())

# Attach mask-path columns when available before filtering.
mask_candidates = {
    "od_mask_path": ["od_mask_path", "disc_mask_path", "optic_disc_mask_path"],
    "oc_mask_path": ["oc_mask_path", "cup_mask_path", "optic_cup_mask_path"],
    "od_oc_mask_path": ["od_oc_mask_path", "odoc_mask_path", "disc_cup_mask_path"],
}
for standard_name, candidates in mask_candidates.items():
    col = find_column(base, candidates, contains_any=[standard_name.replace("_path", "")])
    if col is not None:
        df[standard_name] = base[col].astype(str)
    elif standard_name not in df.columns:
        df[standard_name] = ""

eligible = df[
    df["strict_anatomical_proxy_valid"]
    & df["gon_binary"].isin([0, 1])
    & df["quality_tertile"].isin(QUALITY_ORDER)
    & df["image_path_exists"]
].copy()

if eligible.empty:
    raise RuntimeError("No eligible HYGD images for the expanded Notebook 15 audit.")

# All-HYGD primary audit cohort.
subset = eligible.sort_values(["quality_tertile", "gon_binary", "patient_id", "image_id"]).reset_index(drop=True)

SUBSET_CSV = TABLE_DIR / "notebook15_hygd_all_strict_valid_core_audit_cohort.csv"
subset.to_csv(SUBSET_CSV, index=False)

subset_summary = (
    subset.groupby(["quality_tertile", "gon_binary"])
    .agg(n_images=("image_id", "nunique"), n_patients=("patient_id", "nunique"))
    .reset_index()
)
SUBSET_SUMMARY_CSV = TABLE_DIR / "notebook15_hygd_all_strict_valid_core_audit_cohort_summary.csv"
subset_summary.to_csv(SUBSET_SUMMARY_CSV, index=False)

# Multi-seed balanced subset assignments for reviewer-requested stability analyses.
stability_rows = []
for subset_seed in SUBSET_STABILITY_SEEDS:
    for q in QUALITY_ORDER:
        for label in [0, 1]:
            block = subset[(subset["quality_tertile"].eq(q)) & (subset["gon_binary"].eq(label))].copy()
            if block.empty:
                continue
            reps = block.sample(frac=1.0, random_state=subset_seed).drop_duplicates("patient_id")
            n_take = min(SUBSET_STABILITY_IMAGES_PER_LABEL_QUALITY, len(reps))
            chosen = reps.sample(n=n_take, random_state=subset_seed) if len(reps) > n_take else reps
            for _, rr in chosen.iterrows():
                stability_rows.append({
                    "subset_seed": int(subset_seed),
                    "image_id": str(rr["image_id"]),
                    "patient_id": str(rr["patient_id"]),
                    "quality_tertile": str(rr["quality_tertile"]),
                    "gon_binary": int(rr["gon_binary"]),
                })

stability_subset = pd.DataFrame(stability_rows).drop_duplicates()
STABILITY_SUBSET_CSV = TABLE_DIR / "notebook15_balanced_subset_stability_assignments.csv"
stability_subset.to_csv(STABILITY_SUBSET_CSV, index=False)

stability_summary = (
    stability_subset.groupby(["subset_seed", "quality_tertile", "gon_binary"])
    .agg(n_images=("image_id", "nunique"), n_patients=("patient_id", "nunique"))
    .reset_index()
) if not stability_subset.empty else pd.DataFrame()
STABILITY_SUBSET_SUMMARY_CSV = TABLE_DIR / "notebook15_balanced_subset_stability_summary.csv"
stability_summary.to_csv(STABILITY_SUBSET_SUMMARY_CSV, index=False)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 3",
    "status": "PASS",
    "audit_cohort_mode": AUDIT_COHORT_MODE,
    "subset_csv": str(SUBSET_CSV),
    "subset_summary_csv": str(SUBSET_SUMMARY_CSV),
    "stability_subset_csv": str(STABILITY_SUBSET_CSV),
    "stability_subset_summary_csv": str(STABILITY_SUBSET_SUMMARY_CSV),
    "n_audit_images": int(len(subset)),
    "n_audit_patients": int(subset["patient_id"].nunique()),
    "n_stability_seeds": int(len(SUBSET_STABILITY_SEEDS)),
    "n_stability_rows": int(len(stability_subset)),
}
with open(RESULTS_15_DIR / "notebook15_cell03_expanded_cohort_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Expanded all-HYGD audit cohort summary:")
print(subset_summary.to_string(index=False))
print("\nBalanced subset stability summary:")
print(stability_summary.head(30).to_string(index=False) if not stability_summary.empty else "No stability rows.")
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 4: Image, mask, model, saliency, and perturbation utilities

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image, ImageFilter

MODEL_LOAD_AUDIT_CSV = TABLE_DIR / "notebook15_model_load_audit.csv"

# -------------------------------------------------------------------------
# Model utilities
# -------------------------------------------------------------------------

def model_name_to_timm_id(model_name):
    m = canonical_model_name(model_name)
    mapping = {
        "EfficientNetB0": "efficientnet_b0",
        "ConvNeXtTiny": "convnext_tiny",
        "DenseNet121": "densenet121",
        "SwinTiny": "swin_tiny_patch4_window7_224",
        "ResNet18": "resnet18",
    }
    return mapping.get(m, None)

def create_model(model_name, num_classes=2):
    timm_id = model_name_to_timm_id(model_name)
    if timm_id is None:
        raise ValueError(f"No timm ID mapping for model: {model_name}")
    return timm.create_model(timm_id, pretrained=False, num_classes=num_classes)

def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model", "net", "network"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
    return checkpoint if isinstance(checkpoint, dict) else {}

def clean_state_dict_keys(state_dict):
    cleaned = {}
    for k, v in state_dict.items():
        nk = str(k)
        for prefix in ["module.", "model.", "net.", "network."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v
    return cleaned

def try_load_model_guarded(model_name, checkpoint_path):
    attempts = []
    checkpoint_path = Path(checkpoint_path)
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    sd = clean_state_dict_keys(extract_state_dict(ckpt))

    for num_classes in [2, 1]:
        try:
            model = create_model(model_name, num_classes=num_classes)
            missing, unexpected = model.load_state_dict(sd, strict=False)
            attempts.append({
                "num_classes": num_classes,
                "model": model,
                "n_missing_keys": len(missing),
                "n_unexpected_keys": len(unexpected),
                "missing_preview": ";".join(list(missing)[:10]),
                "unexpected_preview": ";".join(list(unexpected)[:10]),
            })
        except Exception as exc:
            attempts.append({
                "num_classes": num_classes,
                "model": None,
                "n_missing_keys": 10**9,
                "n_unexpected_keys": 10**9,
                "error": f"{type(exc).__name__}: {exc}",
            })

    attempts_sorted = sorted(attempts, key=lambda x: (x["n_missing_keys"] + x["n_unexpected_keys"], x["n_missing_keys"]))
    best = attempts_sorted[0]

    audit = {
        "model_name": canonical_model_name(model_name),
        "checkpoint_path": str(checkpoint_path),
        "num_classes_selected": best.get("num_classes", np.nan),
        "n_missing_keys": int(best["n_missing_keys"]),
        "n_unexpected_keys": int(best["n_unexpected_keys"]),
        "missing_preview": best.get("missing_preview", ""),
        "unexpected_preview": best.get("unexpected_preview", ""),
    }

    if best["model"] is None:
        audit["load_status"] = "load_error"
        return None, audit

    if best["n_missing_keys"] > MAX_ALLOWED_MISSING_KEYS or best["n_unexpected_keys"] > MAX_ALLOWED_UNEXPECTED_KEYS:
        audit["load_status"] = "rejected_architecture_mismatch"
        return None, audit

    model = best["model"].to(DEVICE).eval()
    audit["load_status"] = "loaded_and_guard_passed"
    return model, audit

def reset_module_parameters(module):
    for child in module.children():
        reset_module_parameters(child)
    if hasattr(module, "reset_parameters"):
        try:
            module.reset_parameters()
        except Exception:
            pass

def find_classifier_module(model):
    for attr in ["classifier", "head", "fc"]:
        if hasattr(model, attr):
            return getattr(model, attr), attr
    return None, ""

def find_last_feature_module(model):
    for attr in ["blocks", "stages", "features", "layers"]:
        if hasattr(model, attr):
            module = getattr(model, attr)
            try:
                return module[-1], attr
            except Exception:
                return module, attr
    return None, ""

def randomized_model_copy(model, condition):
    m = deepcopy(model).to(DEVICE).eval()
    if condition == "original":
        return m, "no_randomization"
    if condition == "randomized_classifier_head":
        head, head_name = find_classifier_module(m)
        if head is None:
            raise RuntimeError("Classifier head was not found for randomization.")
        reset_module_parameters(head)
        return m, f"randomized_{head_name}"
    if condition == "randomized_last_block_plus_head":
        last, last_name = find_last_feature_module(m)
        head, head_name = find_classifier_module(m)
        if last is not None:
            reset_module_parameters(last)
        if head is not None:
            reset_module_parameters(head)
        return m, f"randomized_{last_name}_and_{head_name}"
    raise ValueError(f"Unsupported randomization condition: {condition}")

# -------------------------------------------------------------------------
# Image and tensor utilities
# -------------------------------------------------------------------------

def imagenet_transform(image_size=IMAGE_SIZE):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def load_image_tensor(path, image_size=IMAGE_SIZE):
    img = Image.open(path).convert("RGB")
    return imagenet_transform(image_size)(img).unsqueeze(0)

def inverse_normalize_tensor(x):
    mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(1, 3, 1, 1)
    return torch.clamp(x * std + mean, 0, 1)

def normalize_tensor_for_model(x01):
    mean = torch.tensor([0.485, 0.456, 0.406], device=x01.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=x01.device).view(1, 3, 1, 1)
    return (x01 - mean) / std

def tensor_to_pil_01(x01):
    x01 = torch.clamp(x01.detach().cpu().squeeze(0), 0, 1)
    arr = (x01.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(arr)

def load_mask_array(path, image_size=IMAGE_SIZE):
    path = Path(str(path))
    if not path.exists():
        return None
    try:
        mask = Image.open(path).convert("L").resize((image_size, image_size), Image.Resampling.NEAREST)
        return np.asarray(mask) > 0
    except Exception:
        return None

def infer_od_oc_mask(row, image_size=IMAGE_SIZE):
    masks = []
    used_paths = []
    for col in ["od_oc_mask_path", "od_mask_path", "oc_mask_path"]:
        if col in row.index and isinstance(row[col], str) and len(str(row[col])) > 0:
            m = load_mask_array(row[col], image_size=image_size)
            if m is not None:
                masks.append(m)
                used_paths.append(str(row[col]))
    if masks:
        out = np.zeros((image_size, image_size), dtype=bool)
        for m in masks:
            out |= m.astype(bool)
        return out, "mask_path", ";".join(used_paths)

    # Fallback only for execution robustness. It must be audited and not used as a main claim.
    yy, xx = np.mgrid[:image_size, :image_size]
    cx, cy = image_size // 2, image_size // 2
    radius = int(image_size * 0.13)
    fallback = ((xx - cx) ** 2 + (yy - cy) ** 2) <= radius ** 2
    return fallback, "center_circle_fallback", ""

def dilate_mask(mask, dilation):
    mask = mask.astype(bool)
    if dilation <= 0:
        return mask
    if SCIPY_NDIMAGE_AVAILABLE:
        return binary_dilation(mask, iterations=int(dilation))
    out = mask.copy()
    for _ in range(int(dilation)):
        padded = np.pad(out, 1, mode="constant", constant_values=False)
        out = padded[1:-1, 1:-1] | padded[:-2, 1:-1] | padded[2:, 1:-1] | padded[1:-1, :-2] | padded[1:-1, 2:]
    return out

def make_random_mask_like(mask, seed):
    rng = np.random.default_rng(seed)
    flat = np.zeros(mask.size, dtype=bool)
    n = int(mask.sum())
    if n <= 0:
        return flat.reshape(mask.shape)
    idx = rng.choice(mask.size, size=n, replace=False)
    flat[idx] = True
    return flat.reshape(mask.shape)

def apply_mask_blur_tensor(x_norm, mask, sigma=8):
    x01 = inverse_normalize_tensor(x_norm)
    pil = tensor_to_pil_01(x01)
    blurred = pil.filter(ImageFilter.GaussianBlur(radius=float(sigma)))
    arr_orig = np.asarray(pil).astype(np.float32) / 255.0
    arr_blur = np.asarray(blurred).astype(np.float32) / 255.0
    arr = np.where(mask.astype(bool)[..., None], arr_blur, arr_orig)
    x01_new = torch.from_numpy(arr).permute(2, 0, 1).float().unsqueeze(0).to(x_norm.device)
    return normalize_tensor_for_model(x01_new)

# -------------------------------------------------------------------------
# Prediction and saliency utilities
# -------------------------------------------------------------------------

def positive_score_from_logits(logits):
    if logits.ndim == 1:
        logits = logits.unsqueeze(0)
    if logits.shape[1] == 1:
        return logits[:, 0]
    return logits[:, 1]

def positive_probability_from_logits(logits):
    if logits.ndim == 1:
        logits = logits.unsqueeze(0)
    if logits.shape[1] == 1:
        return torch.sigmoid(logits[:, 0])
    return torch.softmax(logits, dim=1)[:, 1]

@torch.no_grad()
def predict_probability(model, x):
    model.eval()
    logits = model(x)
    return float(positive_probability_from_logits(logits).detach().cpu().item())

def integrated_gradients_saliency(model, x, steps=IG_STEPS):
    model.eval()
    baseline = torch.zeros_like(x, device=x.device)
    total_grad = torch.zeros_like(x, device=x.device)
    for alpha in torch.linspace(0, 1, int(steps), device=x.device):
        xi = baseline + alpha * (x - baseline)
        xi.requires_grad_(True)
        model.zero_grad(set_to_none=True)
        logits = model(xi)
        score = positive_score_from_logits(logits).sum()
        grad = torch.autograd.grad(score, xi, retain_graph=False, create_graph=False)[0]
        total_grad += grad.detach()
    attr = (x - baseline) * total_grad / float(steps)
    sal = attr.detach().abs().sum(dim=1).squeeze(0)
    sal = sal - sal.min()
    sal = sal / (sal.max() + 1e-8)
    return sal.detach().cpu().numpy().astype(np.float32)

def saliency_metrics(saliency, mask, seed):
    sal = np.asarray(saliency, dtype=np.float32)
    sal = sal / (sal.sum() + 1e-8)
    mask = mask.astype(bool)
    random_mask = make_random_mask_like(mask, seed=seed)
    mass_mask = float(sal[mask].sum()) if mask.any() else np.nan
    mass_random = float(sal[random_mask].sum()) if random_mask.any() else np.nan
    return {
        "saliency_mass_od_oc": mass_mask,
        "saliency_mass_random": mass_random,
        "od_oc_observed_minus_random": mass_mask - mass_random if np.isfinite(mass_mask) and np.isfinite(mass_random) else np.nan,
        "od_oc_enrichment_ratio": mass_mask / (mass_random + 1e-8) if np.isfinite(mass_mask) and np.isfinite(mass_random) else np.nan,
    }

def map_similarity(a, b):
    a = np.asarray(a, dtype=np.float32).reshape(-1)
    b = np.asarray(b, dtype=np.float32).reshape(-1)
    if a.std() < 1e-8 or b.std() < 1e-8:
        return {"pearson_similarity_vs_original": np.nan, "spearman_similarity_vs_original": np.nan}
    pearson = float(np.corrcoef(a, b)[0, 1])
    ar = pd.Series(a).rank(method="average").values
    br = pd.Series(b).rank(method="average").values
    spearman = float(np.corrcoef(ar, br)[0, 1]) if np.std(ar) > 1e-8 and np.std(br) > 1e-8 else np.nan
    return {"pearson_similarity_vs_original": pearson, "spearman_similarity_vs_original": spearman}

def saliency_top_fraction_mask(saliency, fraction):
    sal = np.asarray(saliency, dtype=np.float32)
    if fraction <= 0:
        return np.zeros_like(sal, dtype=bool)
    n = int(round(fraction * sal.size))
    n = max(1, min(n, sal.size))
    threshold = np.partition(sal.reshape(-1), -n)[-n]
    return sal >= threshold

print("Utility cell loaded.")


In [ ]:
# Cell 5: Load frozen models with architecture-mismatch guard
#
# Critical correction:
# - A model with hundreds of missing/unexpected keys is rejected.
# - This prevents ConvNeXtTiny from silently loading an EfficientNetB0 checkpoint.

model_load_rows = []
models = {}

for model_name in MODEL_NAMES:
    role = f"checkpoint_{canonical_model_name(model_name)}"
    ckpt_path = get_source_path(role)

    if ckpt_path is None:
        model_load_rows.append({
            "model_name": canonical_model_name(model_name),
            "checkpoint_path": "",
            "load_status": "checkpoint_not_found",
            "n_missing_keys": np.nan,
            "n_unexpected_keys": np.nan,
        })
        continue

    try:
        model, audit = try_load_model_guarded(model_name, ckpt_path)
        model_load_rows.append(audit)
        if model is not None:
            models[canonical_model_name(model_name)] = model
    except Exception as exc:
        model_load_rows.append({
            "model_name": canonical_model_name(model_name),
            "checkpoint_path": str(ckpt_path),
            "load_status": f"load_error:{type(exc).__name__}:{exc}",
            "n_missing_keys": np.nan,
            "n_unexpected_keys": np.nan,
        })

model_load_audit = pd.DataFrame(model_load_rows)
model_load_audit.to_csv(MODEL_LOAD_AUDIT_CSV, index=False)

if len(models) == 0:
    status = "NO_MODELS_LOADED"
else:
    status = "PASS"

# Effective model list for downstream cells.
MODEL_NAMES_EFFECTIVE = list(models.keys())

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 5",
    "status": status,
    "n_models_requested": int(len(MODEL_NAMES)),
    "n_models_loaded": int(len(models)),
    "models_loaded": MODEL_NAMES_EFFECTIVE,
    "model_load_audit_csv": str(MODEL_LOAD_AUDIT_CSV),
    "mismatch_guard": {
        "max_allowed_missing_keys": int(MAX_ALLOWED_MISSING_KEYS),
        "max_allowed_unexpected_keys": int(MAX_ALLOWED_UNEXPECTED_KEYS),
    },
}
with open(RESULTS_15_DIR / "notebook15_cell05_model_load_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(model_load_audit.to_string(index=False))
print("\nSummary:")
print(json.dumps(summary, indent=2))

if len(models) == 0:
    raise RuntimeError("No valid models were loaded. Fix checkpoint paths in notebook15_source_registry.csv before continuing.")


In [ ]:
# Cell 6: Resumable parameter-randomization saliency sanity checks on expanded HYGD cohort
#
# Reviewer concern addressed:
# - The sanity audit is no longer tied to a single 60-image subset.
# - It runs on the expanded all-HYGD strict-valid cohort from Cell 3.
# - Execution is resumable and saves saliency arrays for downstream deletion/insertion.

SALIENCY_SANITY_CSV = TABLE_DIR / "notebook15_saliency_sanity_model_randomization_metrics.csv"
SALIENCY_ARRAY_INDEX_CSV = TABLE_DIR / "notebook15_saliency_array_index.csv"

existing = safe_read_csv(SALIENCY_SANITY_CSV)
completed = set()
if not existing.empty:
    for _, r in existing.iterrows():
        err = str(r.get("error", ""))
        if err not in ["", "nan", "None"]:
            continue
        completed.add((str(r.get("model_name", "")), str(r.get("image_id", "")), str(r.get("randomization_condition", ""))))

saliency_rows = [] if existing.empty else existing.to_dict("records")
array_rows = []
if SALIENCY_ARRAY_INDEX_CSV.exists():
    old_index = safe_read_csv(SALIENCY_ARRAY_INDEX_CSV)
    if not old_index.empty:
        array_rows = old_index.to_dict("records")

subset = safe_read_csv(SUBSET_CSV)
if subset.empty:
    raise RuntimeError("Expanded audit cohort CSV is empty. Run Cell 3 first.")

# Stable task order. Randomized models are created once per model/condition, not once per image.
tasks = []
for model_name in MODEL_NAMES_EFFECTIVE:
    for condition in RANDOMIZATION_CONDITIONS:
        for idx, row in subset.iterrows():
            tasks.append((model_name, condition, int(idx), row))

todo = [t for t in tasks if (t[0], str(t[3]["image_id"]), t[1]) not in completed]
print(f"Total tasks: {len(tasks)} | already completed: {len(completed)} | remaining: {len(todo)}")
print(f"IG_STEPS={IG_STEPS}; audit images={len(subset)}; models={MODEL_NAMES_EFFECTIVE}; conditions={RANDOMIZATION_CONDITIONS}")

start_time = time.time()
processed_since_save = 0
original_saliency_cache = {}

array_index_df = pd.DataFrame(array_rows)
if not array_index_df.empty:
    original_index = array_index_df[array_index_df["randomization_condition"].astype(str).eq("original")].copy()
    original_path_map = {
        (str(r["model_name"]), str(r["image_id"])): str(r["saliency_npy_path"])
        for _, r in original_index.iterrows()
    }
else:
    original_path_map = {}

def stable_task_seed(*parts):
    return int(hashlib.md5("||".join(map(str, parts)).encode("utf-8")).hexdigest()[:8], 16) % 1_000_000

if RUN_SALIENCY_SANITY:
    # Build randomized model once per condition for better comparability and runtime.
    randomized_models = {}
    for model_name in MODEL_NAMES_EFFECTIVE:
        base_model = models[model_name]
        for condition in RANDOMIZATION_CONDITIONS:
            eval_model, randomization_description = randomized_model_copy(base_model, condition)
            randomized_models[(model_name, condition)] = (eval_model, randomization_description)

    for task_i, (model_name, condition, idx, row) in enumerate(todo, start=1):
        image_id = str(row["image_id"])
        patient_id = str(row["patient_id"])
        image_path = Path(str(row["image_path"]))

        if not image_path.exists():
            continue

        try:
            eval_model, randomization_description = randomized_models[(model_name, condition)]
            x = load_image_tensor(image_path, IMAGE_SIZE).to(DEVICE)
            odoc_mask, mask_source, mask_source_detail = infer_od_oc_mask(row, IMAGE_SIZE)
            prob = predict_probability(eval_model, x)
            sal = integrated_gradients_saliency(eval_model, x, steps=IG_STEPS)
            metrics = saliency_metrics(sal, odoc_mask, seed=SEED + stable_task_seed(model_name, image_id, condition))

            safe_image_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", image_id)
            sal_path = ARRAY_DIR / f"saliency_{model_name}_{condition}_{safe_image_id}.npy"
            np.save(sal_path, sal)

            sim = {"pearson_similarity_vs_original": np.nan, "spearman_similarity_vs_original": np.nan}
            if condition == "original":
                original_saliency_cache[(model_name, image_id)] = sal
                original_path_map[(model_name, image_id)] = str(sal_path)
            else:
                original_sal = original_saliency_cache.get((model_name, image_id), None)
                if original_sal is None:
                    p0 = original_path_map.get((model_name, image_id), "")
                    if p0 and Path(p0).exists():
                        original_sal = np.load(p0)
                if original_sal is not None:
                    sim = map_similarity(original_sal, sal)

            saliency_rows.append({
                "image_id": image_id,
                "patient_id": patient_id,
                "model_name": model_name,
                "xai_method": "IntegratedGradients",
                "randomization_condition": condition,
                "randomization_description": randomization_description,
                "quality_tertile": row.get("quality_tertile", ""),
                "gon_binary": row.get("gon_binary", np.nan),
                "probability": prob,
                "mask_source": mask_source,
                "mask_source_detail": mask_source_detail,
                **metrics,
                **sim,
                "saliency_npy_path": str(sal_path),
                "error": "",
            })
            array_rows.append({
                "image_id": image_id,
                "patient_id": patient_id,
                "model_name": model_name,
                "xai_method": "IntegratedGradients",
                "randomization_condition": condition,
                "saliency_npy_path": str(sal_path),
            })

        except Exception as exc:
            saliency_rows.append({
                "image_id": image_id,
                "patient_id": patient_id,
                "model_name": model_name,
                "xai_method": "IntegratedGradients",
                "randomization_condition": condition,
                "randomization_description": "",
                "quality_tertile": row.get("quality_tertile", ""),
                "gon_binary": row.get("gon_binary", np.nan),
                "probability": np.nan,
                "mask_source": "",
                "mask_source_detail": "",
                "saliency_mass_od_oc": np.nan,
                "saliency_mass_random": np.nan,
                "od_oc_observed_minus_random": np.nan,
                "od_oc_enrichment_ratio": np.nan,
                "pearson_similarity_vs_original": np.nan,
                "spearman_similarity_vs_original": np.nan,
                "saliency_npy_path": "",
                "error": f"{type(exc).__name__}: {exc}",
            })

        processed_since_save += 1
        if processed_since_save >= SAVE_EVERY_N_TASKS or task_i == len(todo):
            pd.DataFrame(saliency_rows).to_csv(SALIENCY_SANITY_CSV, index=False)
            pd.DataFrame(array_rows).drop_duplicates().to_csv(SALIENCY_ARRAY_INDEX_CSV, index=False)
            elapsed_min = (time.time() - start_time) / 60.0
            print(f"Saved progress after {task_i}/{len(todo)} remaining tasks | elapsed={elapsed_min:.1f} min")
            processed_since_save = 0
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

saliency_sanity = safe_read_csv(SALIENCY_SANITY_CSV)
saliency_array_index = safe_read_csv(SALIENCY_ARRAY_INDEX_CSV)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 6",
    "status": "PASS" if not saliency_sanity.empty else "NO_SALIENCY_ROWS",
    "saliency_sanity_csv": str(SALIENCY_SANITY_CSV),
    "saliency_array_index_csv": str(SALIENCY_ARRAY_INDEX_CSV),
    "n_rows": int(len(saliency_sanity)),
    "n_images": int(saliency_sanity["image_id"].nunique()) if not saliency_sanity.empty else 0,
    "n_patients": int(saliency_sanity["patient_id"].nunique()) if not saliency_sanity.empty else 0,
    "n_models": int(saliency_sanity["model_name"].nunique()) if not saliency_sanity.empty else 0,
    "ig_steps": int(IG_STEPS),
    "audit_cohort_mode": AUDIT_COHORT_MODE,
    "resumable": True,
}
with open(RESULTS_15_DIR / "notebook15_cell06_expanded_saliency_sanity_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saliency sanity preview:")
print(saliency_sanity.head(20).to_string(index=False) if not saliency_sanity.empty else "No rows generated.")
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 7: Saliency sanity summary with image-level, patient-level, bootstrap CIs, and subset stability

SALIENCY_SANITY_SUMMARY_CSV = TABLE_DIR / "notebook15_saliency_sanity_summary.csv"
SALIENCY_SANITY_PATIENT_CSV = TABLE_DIR / "notebook15_saliency_sanity_patient_level.csv"
SALIENCY_SANITY_BOOTSTRAP_CSV = TABLE_DIR / "notebook15_saliency_sanity_bootstrap_ci.csv"
SALIENCY_SANITY_DEGRADATION_CSV = TABLE_DIR / "notebook15_saliency_sanity_randomization_degradation_ci.csv"
SALIENCY_SANITY_STABILITY_CSV = TABLE_DIR / "notebook15_saliency_sanity_subset_stability.csv"
FIG_SALIENCY_SANITY_PNG = FIGURE_DIR / "notebook15_figure_saliency_sanity.png"
FIG_SALIENCY_SANITY_PDF = FIGURE_DIR / "notebook15_figure_saliency_sanity.pdf"

def bootstrap_mean_ci(values, n_boot=BOOTSTRAP_REPEATS, seed=BOOTSTRAP_SEED, alpha=0.05):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(arr) == 0:
        return np.nan, np.nan, np.nan, 0
    if len(arr) == 1:
        return float(arr.mean()), float(arr[0]), float(arr[0]), 1
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(arr), size=(int(n_boot), len(arr)))
    means = arr[idx].mean(axis=1)
    return float(arr.mean()), float(np.quantile(means, alpha/2)), float(np.quantile(means, 1-alpha/2)), int(len(arr))

def summarize_metric_with_ci(df, group_cols, metric_col, level_name, seed_offset=0):
    rows = []
    if df.empty:
        return pd.DataFrame()
    for key, g in df.groupby(group_cols):
        if not isinstance(key, tuple):
            key = (key,)
        mean, lo, hi, n = bootstrap_mean_ci(g[metric_col], seed=BOOTSTRAP_SEED + seed_offset + abs(hash(str(key))) % 100000)
        row = {col: val for col, val in zip(group_cols, key)}
        row.update({
            "level": level_name,
            "metric": metric_col,
            "n_units": int(n),
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "median": float(pd.to_numeric(g[metric_col], errors="coerce").median()) if n else np.nan,
            "std": float(pd.to_numeric(g[metric_col], errors="coerce").std(ddof=1)) if n > 1 else np.nan,
        })
        rows.append(row)
    return pd.DataFrame(rows)

saliency_sanity = safe_read_csv(SALIENCY_SANITY_CSV)
summary_rows = []
patient_rows = []
bootstrap_rows = []
degradation_rows = []
stability_rows = []

if not saliency_sanity.empty:
    for col in ["od_oc_observed_minus_random", "pearson_similarity_vs_original", "spearman_similarity_vs_original"]:
        if col in saliency_sanity.columns:
            saliency_sanity[col] = pd.to_numeric(saliency_sanity[col], errors="coerce")
    valid = saliency_sanity[saliency_sanity.get("error", "").astype(str).isin(["", "nan", "None"])].copy() if "error" in saliency_sanity.columns else saliency_sanity.copy()

    # Image-level summary.
    for (model_name, condition), g in valid.groupby(["model_name", "randomization_condition"]):
        vals = g["od_oc_observed_minus_random"].dropna()
        sims = g["pearson_similarity_vs_original"].dropna()
        summary_rows.append({
            "model_name": model_name,
            "randomization_condition": condition,
            "level": "image",
            "n_images": int(g["image_id"].nunique()),
            "n_patients": int(g["patient_id"].nunique()),
            "mean_od_oc_observed_minus_random": float(vals.mean()) if len(vals) else np.nan,
            "median_od_oc_observed_minus_random": float(vals.median()) if len(vals) else np.nan,
            "std_od_oc_observed_minus_random": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
            "mean_pearson_similarity_vs_original": float(sims.mean()) if len(sims) else np.nan,
            "median_pearson_similarity_vs_original": float(sims.median()) if len(sims) else np.nan,
        })

    # Patient-level aggregation.
    patient = (
        valid.groupby(["model_name", "patient_id", "randomization_condition"])
        .agg(
            gon_binary=("gon_binary", "max"),
            quality_tertile=("quality_tertile", lambda x: ";".join(sorted(set(map(str, x))))),
            od_oc_observed_minus_random=("od_oc_observed_minus_random", "mean"),
            pearson_similarity_vs_original=("pearson_similarity_vs_original", "mean"),
            spearman_similarity_vs_original=("spearman_similarity_vs_original", "mean"),
            n_images=("image_id", "nunique"),
        )
        .reset_index()
    )
    patient.to_csv(SALIENCY_SANITY_PATIENT_CSV, index=False)

    for (model_name, condition), g in patient.groupby(["model_name", "randomization_condition"]):
        vals = g["od_oc_observed_minus_random"].dropna()
        sims = g["pearson_similarity_vs_original"].dropna()
        summary_rows.append({
            "model_name": model_name,
            "randomization_condition": condition,
            "level": "patient",
            "n_images": int(g["n_images"].sum()),
            "n_patients": int(g["patient_id"].nunique()),
            "mean_od_oc_observed_minus_random": float(vals.mean()) if len(vals) else np.nan,
            "median_od_oc_observed_minus_random": float(vals.median()) if len(vals) else np.nan,
            "std_od_oc_observed_minus_random": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
            "mean_pearson_similarity_vs_original": float(sims.mean()) if len(sims) else np.nan,
            "median_pearson_similarity_vs_original": float(sims.median()) if len(sims) else np.nan,
        })

    # Bootstrap CIs for image and patient levels.
    bootstrap_rows.append(summarize_metric_with_ci(valid, ["model_name", "randomization_condition"], "od_oc_observed_minus_random", "image", seed_offset=10))
    bootstrap_rows.append(summarize_metric_with_ci(patient, ["model_name", "randomization_condition"], "od_oc_observed_minus_random", "patient", seed_offset=20))
    bootstrap_df = pd.concat([x for x in bootstrap_rows if not x.empty], ignore_index=True) if bootstrap_rows else pd.DataFrame()

    # Degradation CIs: original enrichment minus randomized enrichment at matched image/patient level.
    for level_name, df_level, id_col in [("image", valid, "image_id"), ("patient", patient, "patient_id")]:
        for model_name, g_model in df_level.groupby("model_name"):
            piv = g_model.pivot_table(index=id_col, columns="randomization_condition", values="od_oc_observed_minus_random", aggfunc="mean")
            for condition in ["randomized_classifier_head", "randomized_last_block_plus_head"]:
                if "original" in piv.columns and condition in piv.columns:
                    diff = piv["original"] - piv[condition]
                    mean, lo, hi, n = bootstrap_mean_ci(diff, seed=BOOTSTRAP_SEED + 100 + abs(hash(model_name + level_name + condition)) % 100000)
                    degradation_rows.append({
                        "model_name": model_name,
                        "level": level_name,
                        "comparison": f"original_minus_{condition}",
                        "n_units": n,
                        "mean_degradation": mean,
                        "ci95_low": lo,
                        "ci95_high": hi,
                        "median_degradation": float(pd.to_numeric(diff, errors="coerce").median()),
                    })

    # Stability over the reviewer-style balanced subsets, computed from all-HYGD outputs.
    stability_subset = safe_read_csv(STABILITY_SUBSET_CSV) if "STABILITY_SUBSET_CSV" in globals() else pd.DataFrame()
    if not stability_subset.empty:
        for seed, seed_df in stability_subset.groupby("subset_seed"):
            seed_images = set(seed_df["image_id"].astype(str))
            block = valid[valid["image_id"].astype(str).isin(seed_images)].copy()
            for (model_name, condition), g in block.groupby(["model_name", "randomization_condition"]):
                mean, lo, hi, n = bootstrap_mean_ci(g["od_oc_observed_minus_random"], seed=BOOTSTRAP_SEED + int(seed))
                stability_rows.append({
                    "subset_seed": int(seed),
                    "model_name": model_name,
                    "randomization_condition": condition,
                    "n_images": int(g["image_id"].nunique()),
                    "n_patients": int(g["patient_id"].nunique()),
                    "mean_od_oc_observed_minus_random": mean,
                    "ci95_low": lo,
                    "ci95_high": hi,
                })

else:
    patient = pd.DataFrame()
    bootstrap_df = pd.DataFrame()

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SALIENCY_SANITY_SUMMARY_CSV, index=False)
bootstrap_df.to_csv(SALIENCY_SANITY_BOOTSTRAP_CSV, index=False)
pd.DataFrame(degradation_rows).to_csv(SALIENCY_SANITY_DEGRADATION_CSV, index=False)
pd.DataFrame(stability_rows).to_csv(SALIENCY_SANITY_STABILITY_CSV, index=False)

figure_generated = False
if not summary_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.5))
    order = RANDOMIZATION_CONDITIONS
    plot_df = summary_df[summary_df["level"].eq("patient")].copy()
    if plot_df.empty:
        plot_df = summary_df[summary_df["level"].eq("image")].copy()

    ax = axes[0]
    for model_name, g in plot_df.groupby("model_name"):
        g = g.set_index("randomization_condition").reindex(order).reset_index()
        ax.plot(g["randomization_condition"], g["mean_od_oc_observed_minus_random"], marker="o", label=model_name)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_ylabel("Mean OD+OC observed - random")
    ax.set_title("Anatomical enrichment sanity")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=7)

    ax = axes[1]
    for model_name, g in plot_df.groupby("model_name"):
        g = g.set_index("randomization_condition").reindex(order).reset_index()
        ax.plot(g["randomization_condition"], g["mean_pearson_similarity_vs_original"], marker="o", label=model_name)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("Mean similarity vs original")
    ax.set_title("Saliency-map similarity")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=7)

    fig.tight_layout()
    fig.savefig(FIG_SALIENCY_SANITY_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(FIG_SALIENCY_SANITY_PDF, bbox_inches="tight")
    plt.close(fig)
    figure_generated = True

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 7",
    "status": "PASS" if not summary_df.empty else "NO_SUMMARY_ROWS",
    "saliency_sanity_summary_csv": str(SALIENCY_SANITY_SUMMARY_CSV),
    "saliency_sanity_patient_csv": str(SALIENCY_SANITY_PATIENT_CSV),
    "saliency_sanity_bootstrap_csv": str(SALIENCY_SANITY_BOOTSTRAP_CSV),
    "saliency_sanity_degradation_csv": str(SALIENCY_SANITY_DEGRADATION_CSV),
    "saliency_sanity_stability_csv": str(SALIENCY_SANITY_STABILITY_CSV),
    "figure_png": str(FIG_SALIENCY_SANITY_PNG) if FIG_SALIENCY_SANITY_PNG.exists() else "",
    "figure_pdf": str(FIG_SALIENCY_SANITY_PDF) if FIG_SALIENCY_SANITY_PDF.exists() else "",
    "figure_generated": bool(figure_generated),
}
with open(RESULTS_15_DIR / "notebook15_cell07_expanded_saliency_sanity_handoff.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(summary_df.to_string(index=False) if not summary_df.empty else "No summary rows.")
print("\nDegradation CIs:")
print(pd.DataFrame(degradation_rows).to_string(index=False) if degradation_rows else "No degradation rows.")
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 8: Expanded deletion and insertion faithfulness analysis
#
# Reviewer concern addressed:
# - Adds insertion curves in addition to deletion.
# - Runs on the expanded HYGD audit cohort using original IG maps from Cell 6.
# - Saves image-level and patient-level curves for bootstrap CI summaries.

FAITHFULNESS_IMAGE_CSV = TABLE_DIR / "notebook15_deletion_insertion_faithfulness_image_level.csv"
FAITHFULNESS_PATIENT_CSV = TABLE_DIR / "notebook15_deletion_insertion_faithfulness_patient_level.csv"
FAITHFULNESS_AUC_IMAGE_CSV = TABLE_DIR / "notebook15_deletion_insertion_auc_image_level.csv"
FAITHFULNESS_AUC_PATIENT_CSV = TABLE_DIR / "notebook15_deletion_insertion_auc_patient_level.csv"

def apply_mask_mean_fill_tensor(x_norm, mask):
    x01 = inverse_normalize_tensor(x_norm)
    mask_t = torch.from_numpy(mask.astype(bool)).to(x01.device).view(1, 1, IMAGE_SIZE, IMAGE_SIZE)
    keep = ~mask_t
    denom = keep.float().sum(dim=(2,3), keepdim=True).clamp_min(1.0)
    mean_val = (x01 * keep.float()).sum(dim=(2,3), keepdim=True) / denom
    out = torch.where(mask_t, mean_val, x01)
    return normalize_tensor_for_model(out)

def apply_mask_gaussian_noise_tensor(x_norm, mask, seed=SEED, noise_std=0.10):
    rng = np.random.default_rng(seed)
    x01 = inverse_normalize_tensor(x_norm)
    arr = x01.detach().cpu().squeeze(0).permute(1,2,0).numpy()
    noise = rng.normal(0, noise_std, size=arr.shape).astype(np.float32)
    arr_new = arr.copy()
    arr_new[mask.astype(bool)] = np.clip(arr_new[mask.astype(bool)] + noise[mask.astype(bool)], 0, 1)
    out = torch.from_numpy(arr_new).permute(2,0,1).float().unsqueeze(0).to(x_norm.device)
    return normalize_tensor_for_model(out)

def globally_blurred_tensor(x_norm, sigma=8):
    full = np.ones((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
    return apply_mask_blur_tensor(x_norm, full, sigma=sigma)

def restore_mask_from_original(base_norm, original_norm, mask):
    base01 = inverse_normalize_tensor(base_norm)
    orig01 = inverse_normalize_tensor(original_norm)
    mask_t = torch.from_numpy(mask.astype(bool)).to(base01.device).view(1,1,IMAGE_SIZE,IMAGE_SIZE)
    out01 = torch.where(mask_t, orig01, base01)
    return normalize_tensor_for_model(out01)

def trapezoid_auc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    order = np.argsort(x)
    if len(x) < 2:
        return np.nan
    return float(np.trapz(y[order], x[order]))

faithfulness_rows = []
subset = safe_read_csv(SUBSET_CSV)
saliency_index = safe_read_csv(SALIENCY_ARRAY_INDEX_CSV)
if not saliency_index.empty:
    saliency_index = saliency_index[saliency_index["randomization_condition"].astype(str).eq("original")].copy()

existing = safe_read_csv(FAITHFULNESS_IMAGE_CSV)
completed = set()
if not existing.empty:
    for _, r in existing.iterrows():
        if str(r.get("error", "")) not in ["", "nan", "None"]:
            continue
        completed.add((str(r.get("model_name", "")), str(r.get("image_id", ""))))
    faithfulness_rows = existing.to_dict("records")

if RUN_DELETION_FAITHFULNESS and not saliency_index.empty:
    for model_name in MODEL_NAMES_EFFECTIVE:
        model = models[model_name]
        sidx_model = saliency_index[saliency_index["model_name"].astype(str).eq(model_name)].copy()
        sal_path_map = dict(zip(sidx_model["image_id"].astype(str), sidx_model["saliency_npy_path"].astype(str)))

        rows_iter = list(subset.iterrows())
        for n_i, (idx, row) in enumerate(rows_iter, start=1):
            image_id = str(row["image_id"])
            if (model_name, image_id) in completed:
                continue
            image_path = Path(str(row["image_path"]))
            sal_path = Path(sal_path_map.get(image_id, ""))
            if not image_path.exists() or not sal_path.exists():
                continue
            try:
                x = load_image_tensor(image_path, IMAGE_SIZE).to(DEVICE)
                sal = np.load(sal_path)
                odoc_mask, mask_source, mask_source_detail = infer_od_oc_mask(row, IMAGE_SIZE)
                p_original = predict_probability(model, x)
                x_baseline = globally_blurred_tensor(x, sigma=8)
                p_baseline = predict_probability(model, x_baseline)

                for frac in DELETION_FRACTIONS:
                    sal_mask = saliency_top_fraction_mask(sal, frac)
                    rand_mask = make_random_mask_like(sal_mask, seed=SEED + int(idx) + int(frac * 10000))
                    odoc_for_frac = odoc_mask if frac > 0 else np.zeros_like(odoc_mask, dtype=bool)
                    masks = {
                        "saliency_guided": sal_mask,
                        "matched_random": rand_mask,
                        "od_oc_proxy_fixed": odoc_for_frac,
                    }
                    for strategy, mask in masks.items():
                        # Deletion: remove evidence from the original image.
                        x_del = apply_mask_blur_tensor(x, mask, sigma=8)
                        p_del = predict_probability(model, x_del)
                        faithfulness_rows.append({
                            "model_name": model_name,
                            "image_id": image_id,
                            "patient_id": row.get("patient_id", ""),
                            "quality_tertile": row.get("quality_tertile", ""),
                            "gon_binary": row.get("gon_binary", np.nan),
                            "xai_method": "IntegratedGradients",
                            "curve_type": "deletion",
                            "strategy": strategy,
                            "fraction": float(frac),
                            "probability_reference": p_original,
                            "probability_modified": p_del,
                            "probability_delta": p_original - p_del,
                            "mask_source": mask_source,
                            "mask_fraction": float(mask.mean()),
                            "error": "",
                        })
                        # Insertion: restore evidence into a globally blurred baseline.
                        x_ins = restore_mask_from_original(x_baseline, x, mask)
                        p_ins = predict_probability(model, x_ins)
                        faithfulness_rows.append({
                            "model_name": model_name,
                            "image_id": image_id,
                            "patient_id": row.get("patient_id", ""),
                            "quality_tertile": row.get("quality_tertile", ""),
                            "gon_binary": row.get("gon_binary", np.nan),
                            "xai_method": "IntegratedGradients",
                            "curve_type": "insertion",
                            "strategy": strategy,
                            "fraction": float(frac),
                            "probability_reference": p_baseline,
                            "probability_modified": p_ins,
                            "probability_delta": p_ins - p_baseline,
                            "mask_source": mask_source,
                            "mask_fraction": float(mask.mean()),
                            "error": "",
                        })
            except Exception as exc:
                faithfulness_rows.append({
                    "model_name": model_name,
                    "image_id": image_id,
                    "patient_id": row.get("patient_id", ""),
                    "quality_tertile": row.get("quality_tertile", ""),
                    "gon_binary": row.get("gon_binary", np.nan),
                    "xai_method": "IntegratedGradients",
                    "curve_type": "error",
                    "strategy": "error",
                    "fraction": np.nan,
                    "probability_reference": np.nan,
                    "probability_modified": np.nan,
                    "probability_delta": np.nan,
                    "mask_source": "",
                    "mask_fraction": np.nan,
                    "error": f"{type(exc).__name__}: {exc}",
                })

            if n_i % SAVE_EVERY_N_TASKS == 0:
                pd.DataFrame(faithfulness_rows).to_csv(FAITHFULNESS_IMAGE_CSV, index=False)
                print(f"Saved faithfulness progress: model={model_name}, image {n_i}/{len(rows_iter)}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

faithfulness = pd.DataFrame(faithfulness_rows)
faithfulness.to_csv(FAITHFULNESS_IMAGE_CSV, index=False)

if not faithfulness.empty:
    valid = faithfulness[faithfulness["curve_type"].isin(["deletion", "insertion"])].copy()
    for col in ["probability_delta", "fraction", "mask_fraction"]:
        valid[col] = pd.to_numeric(valid[col], errors="coerce")
    patient_faithfulness = (
        valid.groupby(["model_name", "patient_id", "curve_type", "strategy", "fraction"])
        .agg(
            gon_binary=("gon_binary", "max"),
            quality_tertile=("quality_tertile", lambda x: ";".join(sorted(set(map(str, x))))),
            probability_reference=("probability_reference", "mean"),
            probability_modified=("probability_modified", "mean"),
            probability_delta=("probability_delta", "mean"),
            mask_fraction=("mask_fraction", "mean"),
            n_images=("image_id", "nunique"),
        )
        .reset_index()
    )
else:
    valid = pd.DataFrame()
    patient_faithfulness = pd.DataFrame()
patient_faithfulness.to_csv(FAITHFULNESS_PATIENT_CSV, index=False)

# AUC summaries per image and patient.
auc_rows = []
if not valid.empty:
    for (model_name, image_id, curve_type, strategy), g in valid.groupby(["model_name", "image_id", "curve_type", "strategy"]):
        auc_rows.append({
            "model_name": model_name,
            "image_id": image_id,
            "patient_id": str(g["patient_id"].iloc[0]),
            "quality_tertile": str(g["quality_tertile"].iloc[0]),
            "gon_binary": g["gon_binary"].iloc[0],
            "curve_type": curve_type,
            "strategy": strategy,
            "auc_delta": trapezoid_auc(g["fraction"], g["probability_delta"]),
            "n_points": int(g["fraction"].nunique()),
        })
auc_image = pd.DataFrame(auc_rows)
auc_image.to_csv(FAITHFULNESS_AUC_IMAGE_CSV, index=False)

auc_patient = (
    auc_image.groupby(["model_name", "patient_id", "curve_type", "strategy"])
    .agg(
        gon_binary=("gon_binary", "max"),
        quality_tertile=("quality_tertile", lambda x: ";".join(sorted(set(map(str, x))))),
        auc_delta=("auc_delta", "mean"),
        n_images=("image_id", "nunique"),
    )
    .reset_index()
) if not auc_image.empty else pd.DataFrame()
auc_patient.to_csv(FAITHFULNESS_AUC_PATIENT_CSV, index=False)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 8",
    "status": "PASS" if not faithfulness.empty else "NO_FAITHFULNESS_ROWS",
    "image_level_csv": str(FAITHFULNESS_IMAGE_CSV),
    "patient_level_csv": str(FAITHFULNESS_PATIENT_CSV),
    "auc_image_level_csv": str(FAITHFULNESS_AUC_IMAGE_CSV),
    "auc_patient_level_csv": str(FAITHFULNESS_AUC_PATIENT_CSV),
    "n_rows": int(len(faithfulness)),
    "n_images": int(faithfulness["image_id"].nunique()) if not faithfulness.empty else 0,
    "n_patients": int(faithfulness["patient_id"].nunique()) if not faithfulness.empty else 0,
    "fractions": DELETION_FRACTIONS,
}
with open(RESULTS_15_DIR / "notebook15_cell08_expanded_faithfulness_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(faithfulness.head(20).to_string(index=False) if not faithfulness.empty else "No rows generated.")
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 9: Deletion/insertion faithfulness summary with bootstrap CIs and subset stability

FAITHFULNESS_SUMMARY_CSV = TABLE_DIR / "notebook15_deletion_insertion_faithfulness_summary.csv"
FAITHFULNESS_BOOTSTRAP_CSV = TABLE_DIR / "notebook15_deletion_insertion_faithfulness_bootstrap_ci.csv"
FAITHFULNESS_AUC_SUMMARY_CSV = TABLE_DIR / "notebook15_deletion_insertion_auc_summary.csv"
FAITHFULNESS_STABILITY_CSV = TABLE_DIR / "notebook15_deletion_insertion_subset_stability.csv"
FIG_FAITHFULNESS_PNG = FIGURE_DIR / "notebook15_figure_deletion_insertion_faithfulness.png"
FIG_FAITHFULNESS_PDF = FIGURE_DIR / "notebook15_figure_deletion_insertion_faithfulness.pdf"

faithfulness = safe_read_csv(FAITHFULNESS_IMAGE_CSV)
patient_faithfulness = safe_read_csv(FAITHFULNESS_PATIENT_CSV)
auc_patient = safe_read_csv(FAITHFULNESS_AUC_PATIENT_CSV)
summary_rows = []
bootstrap_rows = []
auc_summary_rows = []
stability_rows = []

if not faithfulness.empty:
    for col in ["probability_delta", "fraction", "mask_fraction"]:
        faithfulness[col] = pd.to_numeric(faithfulness[col], errors="coerce")
    valid = faithfulness[faithfulness["curve_type"].isin(["deletion", "insertion"])].copy()

    for level_name, df_level, unit_col in [("image", valid, "image_id"), ("patient", patient_faithfulness, "patient_id")]:
        if df_level.empty:
            continue
        for col in ["probability_delta", "fraction"]:
            df_level[col] = pd.to_numeric(df_level[col], errors="coerce")
        for (model_name, curve_type, strategy, frac), g in df_level.groupby(["model_name", "curve_type", "strategy", "fraction"]):
            vals = g["probability_delta"].dropna()
            mean, lo, hi, n = bootstrap_mean_ci(vals, seed=BOOTSTRAP_SEED + abs(hash(str((level_name, model_name, curve_type, strategy, frac)))) % 100000)
            summary_rows.append({
                "model_name": model_name,
                "level": level_name,
                "curve_type": curve_type,
                "strategy": strategy,
                "fraction": float(frac),
                "n_units": int(n),
                "n_images": int(g["image_id"].nunique()) if "image_id" in g.columns else int(g["n_images"].sum()) if "n_images" in g.columns else np.nan,
                "n_patients": int(g["patient_id"].nunique()) if "patient_id" in g.columns else np.nan,
                "mean_probability_delta": mean,
                "ci95_low": lo,
                "ci95_high": hi,
                "median_probability_delta": float(vals.median()) if len(vals) else np.nan,
                "std_probability_delta": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
            })

    # Guided-minus-random difference at matched unit/fraction.
    for level_name, df_level, unit_col in [("image", valid, "image_id"), ("patient", patient_faithfulness, "patient_id")]:
        if df_level.empty:
            continue
        for (model_name, curve_type, frac), g in df_level.groupby(["model_name", "curve_type", "fraction"]):
            piv = g.pivot_table(index=unit_col, columns="strategy", values="probability_delta", aggfunc="mean")
            if {"saliency_guided", "matched_random"}.issubset(set(piv.columns)):
                diff = piv["saliency_guided"] - piv["matched_random"]
                mean, lo, hi, n = bootstrap_mean_ci(diff, seed=BOOTSTRAP_SEED + 777 + abs(hash(str((level_name, model_name, curve_type, frac)))) % 100000)
                bootstrap_rows.append({
                    "model_name": model_name,
                    "level": level_name,
                    "curve_type": curve_type,
                    "comparison": "saliency_guided_minus_matched_random",
                    "fraction": float(frac),
                    "n_units": int(n),
                    "mean_delta": mean,
                    "ci95_low": lo,
                    "ci95_high": hi,
                    "median_delta": float(pd.to_numeric(diff, errors="coerce").median()),
                })

    # AUC summaries at patient level.
    if not auc_patient.empty:
        auc_patient["auc_delta"] = pd.to_numeric(auc_patient["auc_delta"], errors="coerce")
        for (model_name, curve_type, strategy), g in auc_patient.groupby(["model_name", "curve_type", "strategy"]):
            mean, lo, hi, n = bootstrap_mean_ci(g["auc_delta"], seed=BOOTSTRAP_SEED + abs(hash(str((model_name, curve_type, strategy)))) % 100000)
            auc_summary_rows.append({
                "model_name": model_name,
                "level": "patient",
                "curve_type": curve_type,
                "strategy": strategy,
                "n_patients": int(g["patient_id"].nunique()),
                "mean_auc_delta": mean,
                "ci95_low": lo,
                "ci95_high": hi,
                "median_auc_delta": float(g["auc_delta"].median()),
            })

    # Subset stability computed from expanded results.
    stability_subset = safe_read_csv(STABILITY_SUBSET_CSV) if "STABILITY_SUBSET_CSV" in globals() else pd.DataFrame()
    if not stability_subset.empty:
        for seed, seed_df in stability_subset.groupby("subset_seed"):
            seed_images = set(seed_df["image_id"].astype(str))
            block = valid[valid["image_id"].astype(str).isin(seed_images)].copy()
            for (model_name, curve_type, frac), g in block.groupby(["model_name", "curve_type", "fraction"]):
                piv = g.pivot_table(index="image_id", columns="strategy", values="probability_delta", aggfunc="mean")
                if {"saliency_guided", "matched_random"}.issubset(set(piv.columns)):
                    diff = piv["saliency_guided"] - piv["matched_random"]
                    mean, lo, hi, n = bootstrap_mean_ci(diff, seed=BOOTSTRAP_SEED + int(seed))
                    stability_rows.append({
                        "subset_seed": int(seed),
                        "model_name": model_name,
                        "curve_type": curve_type,
                        "fraction": float(frac),
                        "comparison": "saliency_guided_minus_matched_random",
                        "n_images": int(n),
                        "mean_delta": mean,
                        "ci95_low": lo,
                        "ci95_high": hi,
                    })

faithfulness_summary = pd.DataFrame(summary_rows)
faithfulness_bootstrap = pd.DataFrame(bootstrap_rows)
auc_summary = pd.DataFrame(auc_summary_rows)
stability_summary = pd.DataFrame(stability_rows)

faithfulness_summary.to_csv(FAITHFULNESS_SUMMARY_CSV, index=False)
faithfulness_bootstrap.to_csv(FAITHFULNESS_BOOTSTRAP_CSV, index=False)
auc_summary.to_csv(FAITHFULNESS_AUC_SUMMARY_CSV, index=False)
stability_summary.to_csv(FAITHFULNESS_STABILITY_CSV, index=False)

figure_generated = False
plot_df = faithfulness_summary[(faithfulness_summary["level"].eq("patient")) & (faithfulness_summary["strategy"].isin(["saliency_guided", "matched_random"]))].copy()
if not plot_df.empty:
    fig, axes = plt.subplots(len(MODEL_NAMES_EFFECTIVE), 2, figsize=(9.8, 3.2 * len(MODEL_NAMES_EFFECTIVE)), squeeze=False)
    for r, model_name in enumerate(MODEL_NAMES_EFFECTIVE):
        for c, curve_type in enumerate(["deletion", "insertion"]):
            ax = axes[r, c]
            g_model = plot_df[(plot_df["model_name"].eq(model_name)) & (plot_df["curve_type"].eq(curve_type))].copy()
            for strategy, g in g_model.groupby("strategy"):
                g = g.sort_values("fraction")
                ax.plot(g["fraction"], g["mean_probability_delta"], marker="o", label=strategy)
                if {"ci95_low", "ci95_high"}.issubset(g.columns):
                    ax.fill_between(g["fraction"].values.astype(float), g["ci95_low"].values.astype(float), g["ci95_high"].values.astype(float), alpha=0.12)
            ax.axhline(0, linestyle="--", linewidth=1)
            ax.set_title(f"{model_name} — {curve_type}")
            ax.set_xlabel("Fraction")
            ax.set_ylabel("Patient-level probability delta")
            ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIG_FAITHFULNESS_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(FIG_FAITHFULNESS_PDF, bbox_inches="tight")
    plt.close(fig)
    figure_generated = True

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 9",
    "status": "PASS" if not faithfulness_summary.empty else "NO_SUMMARY_ROWS",
    "faithfulness_summary_csv": str(FAITHFULNESS_SUMMARY_CSV),
    "faithfulness_bootstrap_csv": str(FAITHFULNESS_BOOTSTRAP_CSV),
    "faithfulness_auc_summary_csv": str(FAITHFULNESS_AUC_SUMMARY_CSV),
    "faithfulness_stability_csv": str(FAITHFULNESS_STABILITY_CSV),
    "figure_png": str(FIG_FAITHFULNESS_PNG) if FIG_FAITHFULNESS_PNG.exists() else "",
    "figure_pdf": str(FIG_FAITHFULNESS_PDF) if FIG_FAITHFULNESS_PDF.exists() else "",
    "figure_generated": bool(figure_generated),
}
with open(RESULTS_15_DIR / "notebook15_cell09_expanded_faithfulness_handoff.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(faithfulness_summary.head(30).to_string(index=False) if not faithfulness_summary.empty else "No summary rows.")
print("\nGuided-minus-random bootstrap:")
print(faithfulness_bootstrap.head(30).to_string(index=False) if not faithfulness_bootstrap.empty else "No bootstrap rows.")
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 10: Expanded perturbation sensitivity grid with additional perturbation controls
#
# Reviewer concern addressed:
# - Tests whether OD/OC conclusions depend on blur-only perturbation.
# - Adds mean-fill and Gaussian-noise perturbation controls.
# - Runs on the expanded all-HYGD strict-valid cohort.
# - Execution is resumable.

PERTURBATION_GRID_CSV = TABLE_DIR / "notebook15_perturbation_sensitivity_grid.csv"
PERTURBATION_IMAGE_CSV = TABLE_DIR / "notebook15_perturbation_sensitivity_image_level.csv"
PERTURBATION_PATIENT_CSV = TABLE_DIR / "notebook15_perturbation_sensitivity_patient_level.csv"

pd.DataFrame(PERTURBATION_GRID).to_csv(PERTURBATION_GRID_CSV, index=False)

existing = safe_read_csv(PERTURBATION_IMAGE_CSV)
perturb_rows = [] if existing.empty else existing.to_dict("records")
completed = set()
if not existing.empty:
    for _, r in existing.iterrows():
        if str(r.get("error", "")) not in ["", "nan", "None"]:
            continue
        completed.add((
            str(r.get("model_name", "")), str(r.get("image_id", "")), str(r.get("setting_id", "")),
            str(r.get("region_mode", "")), str(r.get("perturbation_method", ""))
        ))

subset = safe_read_csv(SUBSET_CSV)

def apply_perturbation_by_method(x, mask, method, sigma=8, seed=SEED):
    if method == "blur":
        return apply_mask_blur_tensor(x, mask, sigma=sigma)
    if method == "mean_fill":
        return apply_mask_mean_fill_tensor(x, mask)
    if method == "gaussian_noise":
        return apply_mask_gaussian_noise_tensor(x, mask, seed=seed, noise_std=0.10)
    raise ValueError(f"Unsupported perturbation method: {method}")

if RUN_PERTURBATION_SENSITIVITY:
    for model_name in MODEL_NAMES_EFFECTIVE:
        model = models[model_name]
        rows_iter = list(subset.iterrows())
        for n_i, (idx, row) in enumerate(rows_iter, start=1):
            image_id = str(row["image_id"])
            image_path = Path(str(row["image_path"]))
            if not image_path.exists():
                continue
            try:
                x = load_image_tensor(image_path, IMAGE_SIZE).to(DEVICE)
                p_original = predict_probability(model, x)
                odoc_base, mask_source, mask_source_detail = infer_od_oc_mask(row, IMAGE_SIZE)
                for setting in PERTURBATION_GRID:
                    setting_id = setting["setting_id"]
                    sigma = float(setting["sigma"])
                    dilation = int(setting["dilation"])
                    odoc_mask = dilate_mask(odoc_base, dilation)
                    random_mask = make_random_mask_like(odoc_mask, seed=SEED + int(idx) + dilation + int(sigma * 10))
                    outside_mask = ~odoc_mask
                    region_masks = {
                        "od_oc_proxy": odoc_mask,
                        "matched_random_od_oc": random_mask,
                        "outside_od": outside_mask,
                    }
                    for method in PERTURBATION_METHODS:
                        for region_mode, mask in region_masks.items():
                            key = (model_name, image_id, setting_id, region_mode, method)
                            if key in completed:
                                continue
                            x_pert = apply_perturbation_by_method(
                                x, mask, method, sigma=sigma,
                                seed=SEED + int(idx) + int(sigma * 100) + dilation
                            )
                            p_pert = predict_probability(model, x_pert)
                            perturb_rows.append({
                                "model_name": model_name,
                                "image_id": image_id,
                                "patient_id": row.get("patient_id", ""),
                                "quality_tertile": row.get("quality_tertile", ""),
                                "gon_binary": row.get("gon_binary", np.nan),
                                "setting_id": setting_id,
                                "region_mode": region_mode,
                                "perturbation_method": method,
                                "sigma": sigma,
                                "dilation": dilation,
                                "probability_original": p_original,
                                "probability_perturbed": p_pert,
                                "probability_drop": p_original - p_pert,
                                "mask_fraction": float(mask.mean()),
                                "mask_source": mask_source,
                                "error": "",
                            })
            except Exception as exc:
                perturb_rows.append({
                    "model_name": model_name,
                    "image_id": str(row.get("image_id", "")),
                    "patient_id": row.get("patient_id", ""),
                    "quality_tertile": row.get("quality_tertile", ""),
                    "gon_binary": row.get("gon_binary", np.nan),
                    "setting_id": "",
                    "region_mode": "error",
                    "perturbation_method": "error",
                    "sigma": np.nan,
                    "dilation": np.nan,
                    "probability_original": np.nan,
                    "probability_perturbed": np.nan,
                    "probability_drop": np.nan,
                    "mask_fraction": np.nan,
                    "mask_source": "",
                    "error": f"{type(exc).__name__}: {exc}",
                })

            if n_i % SAVE_EVERY_N_TASKS == 0:
                pd.DataFrame(perturb_rows).to_csv(PERTURBATION_IMAGE_CSV, index=False)
                print(f"Saved perturbation progress: model={model_name}, image {n_i}/{len(rows_iter)}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

perturbation = pd.DataFrame(perturb_rows)
perturbation.to_csv(PERTURBATION_IMAGE_CSV, index=False)

if not perturbation.empty:
    valid = perturbation.dropna(subset=["probability_drop"]).copy()
    perturbation_patient = (
        valid.groupby(["model_name", "patient_id", "setting_id", "region_mode", "perturbation_method"])
        .agg(
            gon_binary=("gon_binary", "max"),
            quality_tertile=("quality_tertile", lambda x: ";".join(sorted(set(map(str, x))))),
            sigma=("sigma", "first"),
            dilation=("dilation", "first"),
            probability_original=("probability_original", "mean"),
            probability_perturbed=("probability_perturbed", "mean"),
            probability_drop=("probability_drop", "mean"),
            mask_fraction=("mask_fraction", "mean"),
            n_images=("image_id", "nunique"),
        )
        .reset_index()
    )
else:
    perturbation_patient = pd.DataFrame()
perturbation_patient.to_csv(PERTURBATION_PATIENT_CSV, index=False)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 10",
    "status": "PASS" if not perturbation.empty else "NO_PERTURBATION_ROWS",
    "grid_csv": str(PERTURBATION_GRID_CSV),
    "image_level_csv": str(PERTURBATION_IMAGE_CSV),
    "patient_level_csv": str(PERTURBATION_PATIENT_CSV),
    "n_rows": int(len(perturbation)),
    "n_images": int(perturbation["image_id"].nunique()) if not perturbation.empty else 0,
    "n_patients": int(perturbation["patient_id"].nunique()) if not perturbation.empty else 0,
    "perturbation_methods": PERTURBATION_METHODS,
    "resumable": True,
}
with open(RESULTS_15_DIR / "notebook15_cell10_expanded_perturbation_sensitivity_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(perturbation.head(20).to_string(index=False) if not perturbation.empty else "No rows generated.")
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 11: Expanded perturbation sensitivity summary with bootstrap CIs and subset stability

PERTURBATION_SUMMARY_CSV = TABLE_DIR / "notebook15_perturbation_sensitivity_summary.csv"
PERTURBATION_BOOTSTRAP_CSV = TABLE_DIR / "notebook15_perturbation_sensitivity_bootstrap_ci.csv"
PERTURBATION_COMPARISON_CSV = TABLE_DIR / "notebook15_perturbation_odoc_minus_random_ci.csv"
PERTURBATION_STABILITY_CSV = TABLE_DIR / "notebook15_perturbation_subset_stability.csv"
FIG_PERTURBATION_PNG = FIGURE_DIR / "notebook15_figure_perturbation_sensitivity.png"
FIG_PERTURBATION_PDF = FIGURE_DIR / "notebook15_figure_perturbation_sensitivity.pdf"

perturbation = safe_read_csv(PERTURBATION_IMAGE_CSV)
perturbation_patient = safe_read_csv(PERTURBATION_PATIENT_CSV)
summary_rows = []
bootstrap_rows = []
comparison_rows = []
stability_rows = []

if not perturbation.empty:
    for col in ["probability_drop", "mask_fraction", "sigma", "dilation"]:
        perturbation[col] = pd.to_numeric(perturbation[col], errors="coerce")
    valid = perturbation[perturbation["region_mode"].astype(str).ne("error")].copy()

    for level_name, df_level, unit_col in [("image", valid, "image_id"), ("patient", perturbation_patient, "patient_id")]:
        if df_level.empty:
            continue
        for col in ["probability_drop", "mask_fraction", "sigma", "dilation"]:
            if col in df_level.columns:
                df_level[col] = pd.to_numeric(df_level[col], errors="coerce")
        for (model_name, setting_id, method, region_mode), g in df_level.groupby(["model_name", "setting_id", "perturbation_method", "region_mode"]):
            vals = g["probability_drop"].dropna()
            mean, lo, hi, n = bootstrap_mean_ci(vals, seed=BOOTSTRAP_SEED + abs(hash(str((level_name, model_name, setting_id, method, region_mode)))) % 100000)
            summary_rows.append({
                "model_name": model_name,
                "level": level_name,
                "setting_id": setting_id,
                "perturbation_method": method,
                "region_mode": region_mode,
                "sigma": pd.to_numeric(g["sigma"], errors="coerce").dropna().iloc[0] if g["sigma"].notna().any() else np.nan,
                "dilation": pd.to_numeric(g["dilation"], errors="coerce").dropna().iloc[0] if g["dilation"].notna().any() else np.nan,
                "n_units": int(n),
                "n_images": int(g["image_id"].nunique()) if "image_id" in g.columns else int(g["n_images"].sum()) if "n_images" in g.columns else np.nan,
                "n_patients": int(g["patient_id"].nunique()) if "patient_id" in g.columns else np.nan,
                "mean_probability_drop": mean,
                "ci95_low": lo,
                "ci95_high": hi,
                "median_probability_drop": float(vals.median()) if len(vals) else np.nan,
                "std_probability_drop": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
                "mean_mask_fraction": float(g["mask_fraction"].mean()) if "mask_fraction" in g.columns else np.nan,
            })

        # OD/OC minus matched-random comparisons at matched unit/setting/method.
        for (model_name, setting_id, method), g in df_level.groupby(["model_name", "setting_id", "perturbation_method"]):
            piv = g.pivot_table(index=unit_col, columns="region_mode", values="probability_drop", aggfunc="mean")
            if {"od_oc_proxy", "matched_random_od_oc"}.issubset(set(piv.columns)):
                diff = piv["od_oc_proxy"] - piv["matched_random_od_oc"]
                mean, lo, hi, n = bootstrap_mean_ci(diff, seed=BOOTSTRAP_SEED + 999 + abs(hash(str((level_name, model_name, setting_id, method)))) % 100000)
                comparison_rows.append({
                    "model_name": model_name,
                    "level": level_name,
                    "setting_id": setting_id,
                    "perturbation_method": method,
                    "comparison": "od_oc_proxy_minus_matched_random_od_oc",
                    "n_units": int(n),
                    "mean_delta": mean,
                    "ci95_low": lo,
                    "ci95_high": hi,
                    "median_delta": float(pd.to_numeric(diff, errors="coerce").median()),
                    "n_positive_units": int((pd.to_numeric(diff, errors="coerce") > 0).sum()),
                })

    # Subset stability from all-HYGD outputs.
    stability_subset = safe_read_csv(STABILITY_SUBSET_CSV) if "STABILITY_SUBSET_CSV" in globals() else pd.DataFrame()
    if not stability_subset.empty:
        for seed, seed_df in stability_subset.groupby("subset_seed"):
            seed_images = set(seed_df["image_id"].astype(str))
            block = valid[valid["image_id"].astype(str).isin(seed_images)].copy()
            for (model_name, setting_id, method), g in block.groupby(["model_name", "setting_id", "perturbation_method"]):
                piv = g.pivot_table(index="image_id", columns="region_mode", values="probability_drop", aggfunc="mean")
                if {"od_oc_proxy", "matched_random_od_oc"}.issubset(set(piv.columns)):
                    diff = piv["od_oc_proxy"] - piv["matched_random_od_oc"]
                    mean, lo, hi, n = bootstrap_mean_ci(diff, seed=BOOTSTRAP_SEED + int(seed))
                    stability_rows.append({
                        "subset_seed": int(seed),
                        "model_name": model_name,
                        "setting_id": setting_id,
                        "perturbation_method": method,
                        "comparison": "od_oc_proxy_minus_matched_random_od_oc",
                        "n_images": int(n),
                        "mean_delta": mean,
                        "ci95_low": lo,
                        "ci95_high": hi,
                        "n_positive_images": int((pd.to_numeric(diff, errors="coerce") > 0).sum()),
                    })

perturbation_summary = pd.DataFrame(summary_rows)
comparison_summary = pd.DataFrame(comparison_rows)
stability_summary = pd.DataFrame(stability_rows)

perturbation_summary.to_csv(PERTURBATION_SUMMARY_CSV, index=False)
perturbation_summary.to_csv(PERTURBATION_BOOTSTRAP_CSV, index=False)
comparison_summary.to_csv(PERTURBATION_COMPARISON_CSV, index=False)
stability_summary.to_csv(PERTURBATION_STABILITY_CSV, index=False)

figure_generated = False
plot_df = comparison_summary[(comparison_summary["level"].eq("patient")) & (comparison_summary["perturbation_method"].eq("blur"))].copy()
if not plot_df.empty:
    fig, axes = plt.subplots(1, len(MODEL_NAMES_EFFECTIVE), figsize=(5.8 * len(MODEL_NAMES_EFFECTIVE), 3.8), squeeze=False)
    axes = axes.flatten()
    for ax, model_name in zip(axes, MODEL_NAMES_EFFECTIVE):
        g_model = plot_df[plot_df["model_name"].eq(model_name)].copy().sort_values(["setting_id"])
        ax.errorbar(
            np.arange(len(g_model)),
            g_model["mean_delta"],
            yerr=[g_model["mean_delta"] - g_model["ci95_low"], g_model["ci95_high"] - g_model["mean_delta"]],
            fmt="o-", capsize=3,
        )
        ax.axhline(0, linestyle="--", linewidth=1)
        ax.set_xticks(np.arange(len(g_model)))
        ax.set_xticklabels(g_model["setting_id"], rotation=35, ha="right")
        ax.set_title(model_name)
        ax.set_ylabel("Patient-level OD/OC minus random drop")
        ax.set_xlabel("Blur/dilation setting")
    fig.suptitle("Perturbation sensitivity with bootstrap confidence intervals")
    fig.tight_layout()
    fig.savefig(FIG_PERTURBATION_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(FIG_PERTURBATION_PDF, bbox_inches="tight")
    plt.close(fig)
    figure_generated = True

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 11",
    "status": "PASS" if not perturbation_summary.empty else "NO_SUMMARY_ROWS",
    "perturbation_summary_csv": str(PERTURBATION_SUMMARY_CSV),
    "perturbation_bootstrap_csv": str(PERTURBATION_BOOTSTRAP_CSV),
    "perturbation_comparison_csv": str(PERTURBATION_COMPARISON_CSV),
    "perturbation_stability_csv": str(PERTURBATION_STABILITY_CSV),
    "figure_png": str(FIG_PERTURBATION_PNG) if FIG_PERTURBATION_PNG.exists() else "",
    "figure_pdf": str(FIG_PERTURBATION_PDF) if FIG_PERTURBATION_PDF.exists() else "",
    "figure_generated": bool(figure_generated),
}
with open(RESULTS_15_DIR / "notebook15_cell11_expanded_perturbation_handoff.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(perturbation_summary.head(30).to_string(index=False) if not perturbation_summary.empty else "No summary rows.")
print("\nOD/OC minus random comparison:")
print(comparison_summary.head(30).to_string(index=False) if not comparison_summary.empty else "No comparison rows.")
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 12: Optional conservative outside-OD inference
#
# Guardrail: this is frozen-model inference on conservative outside-OD images/masks.
# It is not equivalent to retraining an outside-OD-only model.

OUTSIDE_OD_INFERENCE_CSV = TABLE_DIR / "notebook15_outside_od_conservative_inference_image_level.csv"
OUTSIDE_OD_METRICS_CSV = TABLE_DIR / "notebook15_outside_od_conservative_inference_metrics.csv"

outside_od_rows = []

if RUN_OUTSIDE_OD_CONSERVATIVE_INFERENCE:
    outside_manifest_path = get_source_path("outside_od_conservative_manifest")
    outside_manifest = safe_read_csv(outside_manifest_path) if outside_manifest_path is not None else pd.DataFrame()

    if outside_manifest.empty:
        print("Outside-OD conservative manifest not found or empty. Skipping optional inference.")
    else:
        image_path_col = find_column(outside_manifest, ["conservative_image_path", "outside_od_image_path", "image_path", "perturbed_image_path"], contains_any=["path"])
        dilation_col = find_column(outside_manifest, ["dilation", "dilation_px", "mask_dilation"], contains_any=["dilation"])
        if image_path_col is None:
            print("No conservative outside-OD image path column was found. Skipping optional inference.")
        else:
            for model_name in MODEL_NAMES_EFFECTIVE:
                model = models[model_name]
                for _, row in outside_manifest.iterrows():
                    path = Path(str(row.get(image_path_col, "")))
                    if not path.exists():
                        continue
                    try:
                        x = load_image_tensor(path, IMAGE_SIZE).to(DEVICE)
                        p = predict_probability(model, x)
                        outside_od_rows.append({
                            "model_name": model_name,
                            "image_id": str(row.get("image_id", "")),
                            "patient_id": str(row.get("patient_id", "")),
                            "gon_binary": row.get("gon_binary", np.nan),
                            "dilation": row.get(dilation_col, np.nan) if dilation_col else np.nan,
                            "outside_od_probability": p,
                            "outside_od_image_path": str(path),
                        })
                    except Exception as exc:
                        outside_od_rows.append({
                            "model_name": model_name,
                            "image_id": str(row.get("image_id", "")),
                            "patient_id": str(row.get("patient_id", "")),
                            "gon_binary": row.get("gon_binary", np.nan),
                            "dilation": row.get(dilation_col, np.nan) if dilation_col else np.nan,
                            "outside_od_probability": np.nan,
                            "outside_od_image_path": str(path),
                            "error": f"{type(exc).__name__}: {exc}",
                        })

outside_inference = pd.DataFrame(outside_od_rows)
outside_inference.to_csv(OUTSIDE_OD_INFERENCE_CSV, index=False)

metric_rows = []
if not outside_inference.empty:
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score, brier_score_loss
    for (model_name, dilation), g in outside_inference.groupby(["model_name", "dilation"]):
        y = pd.to_numeric(g["gon_binary"], errors="coerce")
        p = pd.to_numeric(g["outside_od_probability"], errors="coerce")
        valid = y.notna() & p.notna()
        y = y[valid].astype(int)
        p = p[valid].astype(float)
        row = {
            "model_name": model_name,
            "dilation": dilation,
            "n_images": int(len(y)),
            "n_positive": int((y == 1).sum()) if len(y) else 0,
            "n_negative": int((y == 0).sum()) if len(y) else 0,
            "auroc": np.nan,
            "balanced_accuracy": np.nan,
            "brier": np.nan,
        }
        if len(y) and len(np.unique(y)) == 2:
            row["auroc"] = float(roc_auc_score(y, p))
            row["balanced_accuracy"] = float(balanced_accuracy_score(y, (p >= 0.5).astype(int)))
            row["brier"] = float(brier_score_loss(y, p))
        metric_rows.append(row)

outside_metrics = pd.DataFrame(metric_rows)
outside_metrics.to_csv(OUTSIDE_OD_METRICS_CSV, index=False)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 12",
    "status": "PASS_OPTIONAL_EXECUTED" if not outside_inference.empty else "SKIPPED_OPTIONAL",
    "run_flag": bool(RUN_OUTSIDE_OD_CONSERVATIVE_INFERENCE),
    "image_level_csv": str(OUTSIDE_OD_INFERENCE_CSV),
    "metrics_csv": str(OUTSIDE_OD_METRICS_CSV),
    "n_rows": int(len(outside_inference)),
    "guardrail": "Frozen-model conservative outside-OD inference does not replace retrained outside-OD-only controls.",
}
with open(RESULTS_15_DIR / "notebook15_cell12_outside_od_optional_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(outside_metrics.to_string(index=False) if not outside_metrics.empty else "No optional outside-OD metrics.")
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 13: Final expanded-audit handoff for manuscript revision

FINAL_EVIDENCE_MATRIX_CSV = TABLE_DIR / "notebook15_expanded_core_audit_evidence_matrix.csv"
MANUSCRIPT_SENTENCE_CANDIDATES_CSV = TABLE_DIR / "notebook15_expanded_manuscript_sentence_candidates.csv"
FINAL_HANDOFF_JSON = RESULTS_15_DIR / "notebook15_expanded_final_handoff_summary.json"
FINAL_HANDOFF_TXT = RESULTS_15_DIR / "notebook15_expanded_final_handoff_report.txt"

def path_available(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0

saliency_summary = safe_read_csv(SALIENCY_SANITY_SUMMARY_CSV)
saliency_bootstrap = safe_read_csv(SALIENCY_SANITY_BOOTSTRAP_CSV)
saliency_degradation = safe_read_csv(SALIENCY_SANITY_DEGRADATION_CSV)
faithfulness_summary = safe_read_csv(FAITHFULNESS_SUMMARY_CSV)
faithfulness_bootstrap = safe_read_csv(FAITHFULNESS_BOOTSTRAP_CSV)
faithfulness_auc = safe_read_csv(FAITHFULNESS_AUC_SUMMARY_CSV)
perturbation_summary = safe_read_csv(PERTURBATION_SUMMARY_CSV)
perturbation_comparison = safe_read_csv(PERTURBATION_COMPARISON_CSV)
outside_metrics = safe_read_csv(OUTSIDE_OD_METRICS_CSV) if "OUTSIDE_OD_METRICS_CSV" in globals() else pd.DataFrame()

saliency_available = not saliency_summary.empty and not saliency_bootstrap.empty
faithfulness_available = not faithfulness_summary.empty and not faithfulness_bootstrap.empty
perturbation_available = not perturbation_summary.empty and not perturbation_comparison.empty
outside_optional_available = not outside_metrics.empty

evidence_rows = [
    {
        "reviewer_concern": "Core sanity/deletion/perturbation audit was limited to a 60-image subset.",
        "expanded_audit_status": "resolved" if saliency_available and faithfulness_available and perturbation_available else "not_fully_resolved",
        "evidence_output": f"{SUBSET_CSV}; {SALIENCY_SANITY_SUMMARY_CSV}; {FAITHFULNESS_SUMMARY_CSV}; {PERTURBATION_SUMMARY_CSV}",
        "figure_output": str(FIG_SALIENCY_SANITY_PNG) if path_available(FIG_SALIENCY_SANITY_PNG) else "",
        "claim_boundary": "Primary evidence now uses all eligible HYGD strict-valid images where required inputs are available; balanced subsets are retained only as stability checks.",
    },
    {
        "reviewer_concern": "Limited uncertainty quantification for saliency, deletion, and perturbation endpoints.",
        "expanded_audit_status": "resolved" if saliency_available and faithfulness_available and perturbation_available else "not_fully_resolved",
        "evidence_output": f"{SALIENCY_SANITY_BOOTSTRAP_CSV}; {FAITHFULNESS_BOOTSTRAP_CSV}; {PERTURBATION_COMPARISON_CSV}",
        "figure_output": str(FIG_PERTURBATION_PNG) if path_available(FIG_PERTURBATION_PNG) else "",
        "claim_boundary": "Report image-level and patient-level bootstrap CIs; avoid relying on bar heights alone.",
    },
    {
        "reviewer_concern": "Deletion faithfulness did not include insertion-style controls.",
        "expanded_audit_status": "resolved" if not faithfulness_auc.empty else "not_resolved",
        "evidence_output": f"{FAITHFULNESS_AUC_SUMMARY_CSV}",
        "figure_output": str(FIG_FAITHFULNESS_PNG) if path_available(FIG_FAITHFULNESS_PNG) else "",
        "claim_boundary": "Insertion/deletion AUCs are faithfulness stress tests and can still be affected by perturbation-induced distribution shift.",
    },
    {
        "reviewer_concern": "EfficientNetB0 negative OD/OC blur response may be perturbation-design artifact.",
        "expanded_audit_status": "resolved" if perturbation_available else "not_resolved",
        "evidence_output": f"{PERTURBATION_COMPARISON_CSV}; perturbation methods={PERTURBATION_METHODS}",
        "figure_output": str(FIG_PERTURBATION_PNG) if path_available(FIG_PERTURBATION_PNG) else "",
        "claim_boundary": "Additional perturbation methods test robustness, but do not convert perturbation into causal clinical proof.",
    },
]

evidence_matrix = pd.DataFrame(evidence_rows)
evidence_matrix.to_csv(FINAL_EVIDENCE_MATRIX_CSV, index=False)

sentence_candidates = pd.DataFrame([
    {
        "section": "Methods",
        "candidate_sentence": "The final sanity, insertion/deletion, and perturbation-sensitivity audit was expanded from the initial balanced subset to all eligible HYGD images with strict-valid OD/OC proxies and available image paths; balanced label-by-quality subsets over multiple seeds were retained only as stability checks.",
    },
    {
        "section": "Results",
        "candidate_sentence": "Expanded patient-level bootstrap summaries were used for OD/OC saliency-randomization degradation, saliency-guided versus matched-random deletion/insertion curves, and OD/OC-versus-random perturbation deltas, preventing visual bar-height summaries from serving as the sole quantitative evidence.",
    },
    {
        "section": "Limitations",
        "candidate_sentence": "Insertion/deletion and perturbation analyses remain stress tests because region replacement can induce distribution shift; therefore, AQPR reports convergence or divergence across complementary endpoints rather than treating any single perturbation as causal proof of clinical reasoning.",
    },
])
sentence_candidates.to_csv(MANUSCRIPT_SENTENCE_CANDIDATES_CSV, index=False)

final_summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 13",
    "status": "READY_FOR_NOTEBOOK16_REFRESH" if saliency_available and faithfulness_available and perturbation_available else "RUN_INCOMPLETE_OR_PARTIAL",
    "audit_cohort_mode": AUDIT_COHORT_MODE,
    "evidence_matrix_csv": str(FINAL_EVIDENCE_MATRIX_CSV),
    "sentence_candidates_csv": str(MANUSCRIPT_SENTENCE_CANDIDATES_CSV),
    "expanded_outputs": {
        "saliency_summary": str(SALIENCY_SANITY_SUMMARY_CSV),
        "saliency_bootstrap": str(SALIENCY_SANITY_BOOTSTRAP_CSV),
        "saliency_degradation": str(SALIENCY_SANITY_DEGRADATION_CSV),
        "faithfulness_summary": str(FAITHFULNESS_SUMMARY_CSV),
        "faithfulness_bootstrap": str(FAITHFULNESS_BOOTSTRAP_CSV),
        "faithfulness_auc": str(FAITHFULNESS_AUC_SUMMARY_CSV),
        "perturbation_summary": str(PERTURBATION_SUMMARY_CSV),
        "perturbation_comparison": str(PERTURBATION_COMPARISON_CSV),
        "perturbation_methods": PERTURBATION_METHODS,
    },
}
with open(FINAL_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)

report_lines = [
    "Notebook 15 expanded core AQPR audit handoff",
    "=" * 58,
    f"Status: {final_summary['status']}",
    f"Audit cohort mode: {AUDIT_COHORT_MODE}",
    f"Saliency available: {saliency_available}",
    f"Faithfulness available: {faithfulness_available}",
    f"Perturbation available: {perturbation_available}",
    "",
    "Use these outputs to refresh the manuscript and final consolidation package.",
]
Path(FINAL_HANDOFF_TXT).write_text("\n".join(report_lines), encoding="utf-8")

print(evidence_matrix.to_string(index=False))
print("\nFinal summary:")
print(json.dumps(final_summary, indent=2))


In [ ]:
# Cell 15: Label- and quality-stratified expanded endpoint summaries
#
# Purpose:
# - Convert the expanded all-HYGD audit into reviewer-facing stratified evidence.
# - No model inference is performed here.
# - Uses already generated Cell 6-11 CSV outputs.
# - Produces image-level and patient-level summaries stratified by:
#     overall
#     GON label
#     HYGD quality tertile
#     GON label x HYGD quality tertile
#
# Reviewer concerns addressed:
# - "bar heights without detailed statistics"
# - "limited uncertainty quantification"
# - "quality stratification lacks methodological detail"
# - "provide distributions across images/patients"
# - "clarify whether conclusions hold by label/quality strata"

import json
from pathlib import Path
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Output paths
# -------------------------------------------------------------------------

STRAT_SALIENCY_SUMMARY_CSV = TABLE_DIR / "notebook15_expanded_stratified_saliency_summary.csv"
STRAT_SALIENCY_DEGRADATION_CSV = TABLE_DIR / "notebook15_expanded_stratified_saliency_degradation_ci.csv"

STRAT_FAITHFULNESS_SUMMARY_CSV = TABLE_DIR / "notebook15_expanded_stratified_deletion_insertion_summary.csv"
STRAT_FAITHFULNESS_GUIDED_MINUS_RANDOM_CSV = TABLE_DIR / "notebook15_expanded_stratified_deletion_insertion_guided_minus_random_ci.csv"
STRAT_FAITHFULNESS_AUC_SUMMARY_CSV = TABLE_DIR / "notebook15_expanded_stratified_deletion_insertion_auc_summary.csv"
STRAT_FAITHFULNESS_AUC_GUIDED_MINUS_RANDOM_CSV = TABLE_DIR / "notebook15_expanded_stratified_deletion_insertion_auc_guided_minus_random_ci.csv"

STRAT_PERTURBATION_SUMMARY_CSV = TABLE_DIR / "notebook15_expanded_stratified_perturbation_summary.csv"
STRAT_PERTURBATION_ODOC_MINUS_RANDOM_CSV = TABLE_DIR / "notebook15_expanded_stratified_perturbation_odoc_minus_random_ci.csv"
STRAT_PERTURBATION_METHOD_STABILITY_CSV = TABLE_DIR / "notebook15_expanded_stratified_perturbation_method_stability.csv"

STRAT_CORE_ENDPOINT_SUMMARY_CSV = TABLE_DIR / "notebook15_expanded_stratified_core_endpoint_summary.csv"
STRAT_CELL15_HANDOFF_JSON = RESULTS_15_DIR / "notebook15_cell15_stratified_endpoint_handoff.json"

# -------------------------------------------------------------------------
# Fallback-safe helpers
# -------------------------------------------------------------------------

def _stable_seed(*parts, modulo=100_000):
    text = "||".join(map(str, parts))
    return int(hashlib.md5(text.encode("utf-8")).hexdigest()[:10], 16) % modulo

def _ci(values, seed_offset=0):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan, 0
    if "bootstrap_mean_ci" in globals():
        try:
            return bootstrap_mean_ci(vals, seed=int(BOOTSTRAP_SEED + seed_offset))
        except Exception:
            pass
    rng = np.random.default_rng(int(BOOTSTRAP_SEED + seed_offset))
    mean = float(np.mean(vals))
    if len(vals) == 1:
        return mean, mean, mean, 1
    boot = rng.choice(vals, size=(int(BOOTSTRAP_REPEATS), len(vals)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return mean, float(lo), float(hi), int(len(vals))

def _norm_quality(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    mapping = {
        "0": "low",
        "1": "medium",
        "2": "high",
        "q1": "low",
        "q2": "medium",
        "q3": "high",
        "low_quality": "low",
        "medium_quality": "medium",
        "high_quality": "high",
    }
    return mapping.get(s, s if s not in ["", "nan", "none"] else "unknown")

def _norm_label(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    if s in ["1", "1.0", "true", "gon+", "positive", "glaucoma", "rg"]:
        return "GON+"
    if s in ["0", "0.0", "false", "gon-", "negative", "normal", "nrg"]:
        return "GON-"
    try:
        return "GON+" if float(s) >= 0.5 else "GON-"
    except Exception:
        return str(x)

def _read_or_empty(path):
    try:
        return safe_read_csv(path)
    except Exception:
        path = Path(path)
        if path.exists() and path.stat().st_size > 0:
            return pd.read_csv(path, low_memory=False)
        return pd.DataFrame()

def _attach_cohort_metadata(df):
    """Attach label/quality from the all-HYGD cohort when a result table is missing metadata."""
    if df is None or df.empty:
        return pd.DataFrame()

    out = df.copy()
    cohort = _read_or_empty(SUBSET_CSV)
    if not cohort.empty and "image_id" in out.columns and "image_id" in cohort.columns:
        meta_cols = [c for c in ["image_id", "patient_id", "gon_binary", "quality_tertile"] if c in cohort.columns]
        meta = cohort[meta_cols].drop_duplicates(subset=["image_id"]).copy()
        out["image_id"] = out["image_id"].astype(str)
        meta["image_id"] = meta["image_id"].astype(str)

        for c in ["patient_id", "gon_binary", "quality_tertile"]:
            if c in meta.columns and c not in out.columns:
                out = out.merge(meta[["image_id", c]], on="image_id", how="left")
            elif c in meta.columns and c in out.columns:
                tmp = meta[["image_id", c]].rename(columns={c: f"{c}_cohort"})
                out = out.merge(tmp, on="image_id", how="left")
                out[c] = out[c].where(out[c].notna() & out[c].astype(str).str.lower().ne("nan"), out[f"{c}_cohort"])
                out = out.drop(columns=[f"{c}_cohort"])

    if "gon_binary" in out.columns:
        out["gon_label"] = out["gon_binary"].apply(_norm_label)
    elif "gon_label" not in out.columns:
        out["gon_label"] = "unknown"

    if "quality_tertile" in out.columns:
        out["quality_group"] = out["quality_tertile"].apply(_norm_quality)
    elif "quality_group" not in out.columns:
        out["quality_group"] = "unknown"

    if "patient_id" in out.columns:
        out["patient_id"] = out["patient_id"].astype(str)
    if "image_id" in out.columns:
        out["image_id"] = out["image_id"].astype(str)

    return out

def _valid_rows(df, metric_col):
    if df is None or df.empty or metric_col not in df.columns:
        return pd.DataFrame()
    out = df.copy()
    if "error" in out.columns:
        out = out[out["error"].astype(str).str.strip().isin(["", "nan", "None", "none"])]
    out[metric_col] = pd.to_numeric(out[metric_col], errors="coerce")
    out = out.dropna(subset=[metric_col])
    return out

STRATA_SPECS = {
    "overall": [],
    "label": ["gon_label"],
    "quality": ["quality_group"],
    "label_quality": ["gon_label", "quality_group"],
}

def _make_stratum_label(row_or_dict, cols):
    if not cols:
        return "overall"
    return " | ".join([f"{c}={row_or_dict.get(c, 'unknown')}" for c in cols])

def _unit_level(df, metric_col, level, parameter_cols, strata_cols):
    """Return image-level rows or patient-level rows averaged within each patient and stratum."""
    cols_needed = [c for c in parameter_cols + strata_cols if c in df.columns]
    base_cols = cols_needed + [metric_col]
    if level == "image":
        extra = [c for c in ["image_id", "patient_id"] if c in df.columns]
        return df[extra + base_cols].copy(), "image_id"

    if "patient_id" not in df.columns:
        return pd.DataFrame(), "patient_id"

    group_cols = cols_needed + ["patient_id"]
    agg_dict = {metric_col: "mean"}
    if "image_id" in df.columns:
        agg_dict["image_id"] = "nunique"
    patient = df.groupby(group_cols, dropna=False).agg(agg_dict).reset_index()
    if "image_id" in patient.columns:
        patient = patient.rename(columns={"image_id": "n_images"})
    return patient, "patient_id"

def _summarize_metric(df, endpoint, metric_col, parameter_cols, levels=("image", "patient")):
    rows = []
    df = _valid_rows(_attach_cohort_metadata(df), metric_col)
    if df.empty:
        return pd.DataFrame()

    for stratum_type, strata_cols in STRATA_SPECS.items():
        for level in levels:
            unit_df, unit_col = _unit_level(df, metric_col, level, parameter_cols, strata_cols)
            if unit_df.empty or unit_col not in unit_df.columns:
                continue

            group_cols = [c for c in parameter_cols + strata_cols if c in unit_df.columns]
            if not group_cols:
                grouped = [((), unit_df)]
            else:
                grouped = unit_df.groupby(group_cols, dropna=False)

            for key, g in grouped:
                key_tuple = key if isinstance(key, tuple) else (key,)
                key_map = dict(zip(group_cols, key_tuple))
                vals = pd.to_numeric(g[metric_col], errors="coerce").dropna()
                seed_offset = _stable_seed(endpoint, metric_col, level, stratum_type, key_map)
                mean, lo, hi, n = _ci(vals, seed_offset)

                row = {
                    "endpoint": endpoint,
                    "metric": metric_col,
                    "level": level,
                    "stratum_type": stratum_type,
                    "stratum_label": _make_stratum_label(key_map, strata_cols),
                    "n_units": int(n),
                    "n_images": int(g["image_id"].nunique()) if "image_id" in g.columns else int(g["n_images"].sum()) if "n_images" in g.columns else np.nan,
                    "n_patients": int(g["patient_id"].nunique()) if "patient_id" in g.columns else int(g[unit_col].nunique()),
                    "mean": mean,
                    "median": float(vals.median()) if len(vals) else np.nan,
                    "std": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
                    "ci95_low": lo,
                    "ci95_high": hi,
                }
                row.update(key_map)
                rows.append(row)

    return pd.DataFrame(rows)

def _summarize_pair_delta(
    df,
    endpoint,
    metric_col,
    parameter_cols,
    pair_col,
    positive_col_value,
    reference_col_value,
    comparison_name,
    levels=("image", "patient"),
):
    rows = []
    df = _valid_rows(_attach_cohort_metadata(df), metric_col)
    if df.empty or pair_col not in df.columns:
        return pd.DataFrame()

    for stratum_type, strata_cols in STRATA_SPECS.items():
        for level in levels:
            unit_df, unit_col = _unit_level(df, metric_col, level, parameter_cols + [pair_col], strata_cols)
            if unit_df.empty or unit_col not in unit_df.columns:
                continue

            base_group_cols = [c for c in parameter_cols + strata_cols if c in unit_df.columns]
            index_cols = base_group_cols + [unit_col]
            piv = unit_df.pivot_table(
                index=index_cols,
                columns=pair_col,
                values=metric_col,
                aggfunc="mean",
            ).reset_index()

            if positive_col_value not in piv.columns or reference_col_value not in piv.columns:
                continue

            piv["delta"] = pd.to_numeric(piv[positive_col_value], errors="coerce") - pd.to_numeric(piv[reference_col_value], errors="coerce")
            if not base_group_cols:
                grouped = [((), piv)]
            else:
                grouped = piv.groupby(base_group_cols, dropna=False)

            for key, g in grouped:
                key_tuple = key if isinstance(key, tuple) else (key,)
                key_map = dict(zip(base_group_cols, key_tuple))
                vals = pd.to_numeric(g["delta"], errors="coerce").dropna()
                seed_offset = _stable_seed(endpoint, comparison_name, level, stratum_type, key_map)
                mean, lo, hi, n = _ci(vals, seed_offset)

                row = {
                    "endpoint": endpoint,
                    "comparison": comparison_name,
                    "metric": f"{positive_col_value}_minus_{reference_col_value}",
                    "level": level,
                    "stratum_type": stratum_type,
                    "stratum_label": _make_stratum_label(key_map, strata_cols),
                    "n_units": int(n),
                    "n_positive_units": int((vals > 0).sum()) if len(vals) else 0,
                    "positive_unit_fraction": float((vals > 0).mean()) if len(vals) else np.nan,
                    "mean_delta": mean,
                    "median_delta": float(vals.median()) if len(vals) else np.nan,
                    "std_delta": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
                    "ci95_low": lo,
                    "ci95_high": hi,
                    "support_status": "supported" if (pd.notna(mean) and pd.notna(lo) and mean > 0 and lo > 0) else "directional_only" if (pd.notna(mean) and mean > 0) else "not_supported",
                }
                row.update(key_map)
                rows.append(row)

    return pd.DataFrame(rows)

# -------------------------------------------------------------------------
# Load expanded result tables
# -------------------------------------------------------------------------

saliency_image = _read_or_empty(SALIENCY_SANITY_CSV)
faithfulness_image = _read_or_empty(FAITHFULNESS_IMAGE_CSV)
faithfulness_auc_image = _read_or_empty(FAITHFULNESS_AUC_IMAGE_CSV)
perturbation_image = _read_or_empty(PERTURBATION_IMAGE_CSV)

# -------------------------------------------------------------------------
# Saliency: stratified summaries and randomization degradation CIs
# -------------------------------------------------------------------------

saliency_summary = _summarize_metric(
    saliency_image,
    endpoint="saliency_sanity",
    metric_col="od_oc_observed_minus_random",
    parameter_cols=["model_name", "randomization_condition"],
)

saliency_degradation_rows = []
if not saliency_image.empty:
    for condition in ["randomized_classifier_head", "randomized_last_block_plus_head"]:
        block = saliency_image[
            saliency_image["randomization_condition"].astype(str).isin(["original", condition])
        ].copy()
        tmp = _summarize_pair_delta(
            block,
            endpoint="saliency_sanity",
            metric_col="od_oc_observed_minus_random",
            parameter_cols=["model_name"],
            pair_col="randomization_condition",
            positive_col_value="original",
            reference_col_value=condition,
            comparison_name=f"original_minus_{condition}",
        )
        saliency_degradation_rows.append(tmp)

saliency_degradation = pd.concat(
    [x for x in saliency_degradation_rows if x is not None and not x.empty],
    ignore_index=True,
) if saliency_degradation_rows else pd.DataFrame()

saliency_summary.to_csv(STRAT_SALIENCY_SUMMARY_CSV, index=False)
saliency_degradation.to_csv(STRAT_SALIENCY_DEGRADATION_CSV, index=False)

# -------------------------------------------------------------------------
# Faithfulness: deletion/insertion summaries, guided-minus-random, and AUC
# -------------------------------------------------------------------------

faithfulness_summary = _summarize_metric(
    faithfulness_image,
    endpoint="deletion_insertion_faithfulness",
    metric_col="probability_delta",
    parameter_cols=["model_name", "curve_type", "strategy", "fraction"],
)

faithfulness_guided_minus_random = _summarize_pair_delta(
    faithfulness_image,
    endpoint="deletion_insertion_faithfulness",
    metric_col="probability_delta",
    parameter_cols=["model_name", "curve_type", "fraction"],
    pair_col="strategy",
    positive_col_value="saliency_guided",
    reference_col_value="matched_random",
    comparison_name="saliency_guided_minus_matched_random",
)

faithfulness_auc_summary = _summarize_metric(
    faithfulness_auc_image,
    endpoint="deletion_insertion_auc",
    metric_col="auc_delta",
    parameter_cols=["model_name", "curve_type", "strategy"],
)

faithfulness_auc_guided_minus_random = _summarize_pair_delta(
    faithfulness_auc_image,
    endpoint="deletion_insertion_auc",
    metric_col="auc_delta",
    parameter_cols=["model_name", "curve_type"],
    pair_col="strategy",
    positive_col_value="saliency_guided",
    reference_col_value="matched_random",
    comparison_name="auc_saliency_guided_minus_matched_random",
)

faithfulness_summary.to_csv(STRAT_FAITHFULNESS_SUMMARY_CSV, index=False)
faithfulness_guided_minus_random.to_csv(STRAT_FAITHFULNESS_GUIDED_MINUS_RANDOM_CSV, index=False)
faithfulness_auc_summary.to_csv(STRAT_FAITHFULNESS_AUC_SUMMARY_CSV, index=False)
faithfulness_auc_guided_minus_random.to_csv(STRAT_FAITHFULNESS_AUC_GUIDED_MINUS_RANDOM_CSV, index=False)

# -------------------------------------------------------------------------
# Perturbation: probability-drop summaries and OD/OC-minus-random CIs
# -------------------------------------------------------------------------

perturbation_summary = _summarize_metric(
    perturbation_image,
    endpoint="perturbation_sensitivity",
    metric_col="probability_drop",
    parameter_cols=["model_name", "setting_id", "perturbation_method", "region_mode", "sigma", "dilation"],
)

perturbation_odoc_minus_random = _summarize_pair_delta(
    perturbation_image,
    endpoint="perturbation_sensitivity",
    metric_col="probability_drop",
    parameter_cols=["model_name", "setting_id", "perturbation_method", "sigma", "dilation"],
    pair_col="region_mode",
    positive_col_value="od_oc_proxy",
    reference_col_value="matched_random_od_oc",
    comparison_name="od_oc_proxy_minus_matched_random_od_oc",
)

perturbation_summary.to_csv(STRAT_PERTURBATION_SUMMARY_CSV, index=False)
perturbation_odoc_minus_random.to_csv(STRAT_PERTURBATION_ODOC_MINUS_RANDOM_CSV, index=False)

# -------------------------------------------------------------------------
# Perturbation method-stability table
# -------------------------------------------------------------------------

method_stability_rows = []
if not perturbation_odoc_minus_random.empty:
    block = perturbation_odoc_minus_random[
        perturbation_odoc_minus_random["level"].astype(str).eq("patient")
    ].copy()

    for stratum_type, strata_cols in STRATA_SPECS.items():
        base_cols = ["model_name", "perturbation_method", "stratum_type", "stratum_label"]
        if stratum_type != "overall":
            base_cols += [c for c in strata_cols if c in block.columns]

        if block.empty:
            continue

        grouped = block[block["stratum_type"].eq(stratum_type)].groupby(base_cols, dropna=False)
        for key, g in grouped:
            key_tuple = key if isinstance(key, tuple) else (key,)
            key_map = dict(zip(base_cols, key_tuple))

            n_settings = int(len(g))
            n_supported = int((g["support_status"].astype(str).eq("supported")).sum())
            n_directional = int((pd.to_numeric(g["mean_delta"], errors="coerce") > 0).sum())
            n_not_supported = int((g["support_status"].astype(str).eq("not_supported")).sum())

            method_stability_rows.append({
                **key_map,
                "level": "patient",
                "n_settings": n_settings,
                "n_supported_settings": n_supported,
                "n_directional_settings": n_directional,
                "n_not_supported_settings": n_not_supported,
                "supported_setting_fraction": n_supported / n_settings if n_settings else np.nan,
                "directional_setting_fraction": n_directional / n_settings if n_settings else np.nan,
            })

method_stability = pd.DataFrame(method_stability_rows)
method_stability.to_csv(STRAT_PERTURBATION_METHOD_STABILITY_CSV, index=False)

# -------------------------------------------------------------------------
# Unified core endpoint summary for manuscript/supplement handoff
# -------------------------------------------------------------------------

core_blocks = []

if not saliency_degradation.empty:
    core_blocks.append(
        saliency_degradation.assign(
            core_endpoint="saliency_randomization_degradation",
            reviewer_use="Report patient-level CIs by model; use label/quality strata as supplement."
        )
    )

if not faithfulness_guided_minus_random.empty:
    core_blocks.append(
        faithfulness_guided_minus_random.assign(
            core_endpoint="deletion_insertion_guided_minus_random",
            reviewer_use="Report patient-level guided-minus-random CIs and distributions."
        )
    )

if not faithfulness_auc_guided_minus_random.empty:
    core_blocks.append(
        faithfulness_auc_guided_minus_random.assign(
            core_endpoint="deletion_insertion_auc_guided_minus_random",
            reviewer_use="Use as compact AUC-level faithfulness check."
        )
    )

if not perturbation_odoc_minus_random.empty:
    core_blocks.append(
        perturbation_odoc_minus_random.assign(
            core_endpoint="perturbation_odoc_minus_random",
            reviewer_use="Report blur-based support and alternative-method sensitivity."
        )
    )

core_endpoint_summary = pd.concat(core_blocks, ignore_index=True, sort=False) if core_blocks else pd.DataFrame()
core_endpoint_summary.to_csv(STRAT_CORE_ENDPOINT_SUMMARY_CSV, index=False)

# -------------------------------------------------------------------------
# Compact console report
# -------------------------------------------------------------------------

def _compact_patient_overall(df, title, max_rows=30):
    if df is None or df.empty:
        print(f"\n{title}: empty")
        return
    block = df[
        df["level"].astype(str).eq("patient")
        & df["stratum_type"].astype(str).eq("overall")
    ].copy()
    if block.empty:
        print(f"\n{title}: no patient-level overall rows")
        return
    cols = [
        c for c in [
            "core_endpoint",
            "endpoint",
            "comparison",
            "model_name",
            "curve_type",
            "fraction",
            "perturbation_method",
            "sigma",
            "dilation",
            "n_units",
            "mean_delta",
            "ci95_low",
            "ci95_high",
            "support_status",
        ] if c in block.columns
    ]
    print(f"\n{title}")
    print(block[cols].head(max_rows).to_string(index=False))

_compact_patient_overall(saliency_degradation, "Patient-level saliency randomization degradation, overall")
_compact_patient_overall(faithfulness_guided_minus_random, "Patient-level deletion/insertion guided-minus-random, overall")
_compact_patient_overall(faithfulness_auc_guided_minus_random, "Patient-level insertion/deletion AUC guided-minus-random, overall")
_compact_patient_overall(perturbation_odoc_minus_random, "Patient-level perturbation OD/OC-minus-random, overall")

if not method_stability.empty:
    print("\nPatient-level perturbation method-stability, overall")
    cols = [
        "model_name",
        "perturbation_method",
        "stratum_type",
        "stratum_label",
        "n_settings",
        "n_supported_settings",
        "n_directional_settings",
        "supported_setting_fraction",
        "directional_setting_fraction",
    ]
    print(
        method_stability[
            method_stability["stratum_type"].astype(str).eq("overall")
        ][[c for c in cols if c in method_stability.columns]].to_string(index=False)
    )

# -------------------------------------------------------------------------
# Handoff JSON
# -------------------------------------------------------------------------

handoff = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 15",
    "status": "PASS",
    "purpose": "Label- and quality-stratified expanded all-HYGD endpoint summaries.",
    "input_tables": {
        "saliency_image": str(SALIENCY_SANITY_CSV),
        "faithfulness_image": str(FAITHFULNESS_IMAGE_CSV),
        "faithfulness_auc_image": str(FAITHFULNESS_AUC_IMAGE_CSV),
        "perturbation_image": str(PERTURBATION_IMAGE_CSV),
        "audit_cohort": str(SUBSET_CSV),
    },
    "output_tables": {
        "stratified_saliency_summary": str(STRAT_SALIENCY_SUMMARY_CSV),
        "stratified_saliency_degradation": str(STRAT_SALIENCY_DEGRADATION_CSV),
        "stratified_faithfulness_summary": str(STRAT_FAITHFULNESS_SUMMARY_CSV),
        "stratified_faithfulness_guided_minus_random": str(STRAT_FAITHFULNESS_GUIDED_MINUS_RANDOM_CSV),
        "stratified_faithfulness_auc_summary": str(STRAT_FAITHFULNESS_AUC_SUMMARY_CSV),
        "stratified_faithfulness_auc_guided_minus_random": str(STRAT_FAITHFULNESS_AUC_GUIDED_MINUS_RANDOM_CSV),
        "stratified_perturbation_summary": str(STRAT_PERTURBATION_SUMMARY_CSV),
        "stratified_perturbation_odoc_minus_random": str(STRAT_PERTURBATION_ODOC_MINUS_RANDOM_CSV),
        "stratified_perturbation_method_stability": str(STRAT_PERTURBATION_METHOD_STABILITY_CSV),
        "core_endpoint_summary": str(STRAT_CORE_ENDPOINT_SUMMARY_CSV),
    },
    "n_rows": {
        "saliency_summary": int(len(saliency_summary)),
        "saliency_degradation": int(len(saliency_degradation)),
        "faithfulness_summary": int(len(faithfulness_summary)),
        "faithfulness_guided_minus_random": int(len(faithfulness_guided_minus_random)),
        "faithfulness_auc_summary": int(len(faithfulness_auc_summary)),
        "faithfulness_auc_guided_minus_random": int(len(faithfulness_auc_guided_minus_random)),
        "perturbation_summary": int(len(perturbation_summary)),
        "perturbation_odoc_minus_random": int(len(perturbation_odoc_minus_random)),
        "perturbation_method_stability": int(len(method_stability)),
        "core_endpoint_summary": int(len(core_endpoint_summary)),
    },
    "interpretation_guardrail": (
        "Stratified rows are post-hoc summaries of the expanded frozen-model audit. "
        "They support robustness and transparency analyses, not new model selection."
    ),
}

with open(STRAT_CELL15_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(handoff, f, indent=2)

print("\nCell 15 completed.")
print(json.dumps(handoff, indent=2))

In [ ]:
# Cell 14: Expanded AQPR claim-boundary audit
#
# Purpose:
# - Replace the earlier 60-image claim-boundary logic with expanded all-HYGD evidence.
# - Use patient-level bootstrap CIs, insertion/deletion AUC, and perturbation method-stability.
# - Avoid overclaiming universal perturbation stability when method-dependent effects are present.
#
# Main interpretation:
# - EfficientNetB0:
#     saliency sanity supported;
#     deletion/insertion faithfulness mixed or partially supported;
#     perturbation response method-dependent rather than stable anatomical dependence.
# - ConvNeXtTiny:
#     saliency sanity partially supported;
#     deletion/insertion faithfulness supported;
#     blur-based OD/OC perturbation supported, but replacement/noise controls reveal method dependence;
#     therefore strongest bounded AQPR convergence, not universal anatomical faithfulness.

import json
from pathlib import Path
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------

EXPANDED_SALIENCY_DEGRADATION_CSV = TABLE_DIR / "notebook15_expanded_stratified_saliency_degradation_ci.csv"
EXPANDED_AUC_GUIDED_MINUS_RANDOM_CSV = TABLE_DIR / "notebook15_expanded_stratified_deletion_insertion_auc_guided_minus_random_ci.csv"
EXPANDED_PERT_METHOD_STABILITY_CSV = TABLE_DIR / "notebook15_expanded_stratified_perturbation_method_stability.csv"
EXPANDED_PERT_ODOC_MINUS_RANDOM_CSV = TABLE_DIR / "notebook15_expanded_stratified_perturbation_odoc_minus_random_ci.csv"

# Fallbacks from earlier cells, in case Cell 15 has not been run yet.
FALLBACK_SALIENCY_DEGRADATION_CSV = TABLE_DIR / "notebook15_saliency_sanity_randomization_degradation_ci.csv"
FALLBACK_AUC_GUIDED_MINUS_RANDOM_CSV = TABLE_DIR / "notebook15_deletion_insertion_auc_guided_minus_random_ci.csv"
FALLBACK_PERT_ODOC_MINUS_RANDOM_CSV = TABLE_DIR / "notebook15_perturbation_odoc_minus_random_ci.csv"

CLAIM_BOUNDARY_CSV = TABLE_DIR / "notebook15_expanded_model_claim_boundary_full.csv"
CLAIM_BOUNDARY_COMPAT_CSV = TABLE_DIR / "notebook15_model_claim_boundary_full.csv"
CLAIM_BOUNDARY_SHORT_CSV = TABLE_DIR / "notebook15_expanded_model_claim_boundary_short.csv"
CLAIM_BOUNDARY_JSON = RESULTS_15_DIR / "notebook15_expanded_claim_boundary_handoff.json"

# -------------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------------

def _read_existing_csv(*paths):
    for path in paths:
        path = Path(path)
        if path.exists() and path.stat().st_size > 0:
            return pd.read_csv(path, low_memory=False), str(path)
    return pd.DataFrame(), None

def _as_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def _norm_status(x):
    s = str(x).strip().lower()
    if s in ["supported", "support", "yes", "true"]:
        return "supported"
    if s in ["directional_only", "directional", "partial", "partially_supported"]:
        return "directional_only"
    if s in ["mixed_or_partially_supported", "mixed", "method_dependent_partial"]:
        return "mixed_or_partially_supported"
    return "not_supported"

def _is_supported(row):
    if "support_status" in row:
        return _norm_status(row["support_status"]) == "supported"
    lo = _as_float(row.get("ci95_low", np.nan))
    mean = _as_float(row.get("mean_delta", row.get("mean", np.nan)))
    return bool(pd.notna(mean) and pd.notna(lo) and mean > 0 and lo > 0)

def _is_directional(row):
    mean = _as_float(row.get("mean_delta", row.get("mean", np.nan)))
    return bool(pd.notna(mean) and mean > 0)

def _fmt_ci(mean, lo, hi, digits=4):
    if pd.isna(mean):
        return "NA"
    return f"{mean:.{digits}f} [{lo:.{digits}f}, {hi:.{digits}f}]"

def _patient_overall(df):
    if df.empty:
        return df
    out = df.copy()
    if "level" in out.columns:
        out = out[out["level"].astype(str).eq("patient")]
    if "stratum_type" in out.columns:
        out = out[out["stratum_type"].astype(str).eq("overall")]
    return out.copy()

def _get_model_names(*dfs):
    names = set()
    for df in dfs:
        if df is not None and not df.empty and "model_name" in df.columns:
            names.update(df["model_name"].dropna().astype(str).unique().tolist())
    return sorted(names)

def _get_delta_row(df, model_name, **filters):
    if df.empty:
        return None
    block = df[df["model_name"].astype(str).eq(str(model_name))].copy()
    for col, val in filters.items():
        if col in block.columns:
            block = block[block[col].astype(str).eq(str(val))]
    if block.empty:
        return None
    return block.iloc[0].to_dict()

# -------------------------------------------------------------------------
# Load evidence tables
# -------------------------------------------------------------------------

saliency_deg, saliency_source = _read_existing_csv(
    EXPANDED_SALIENCY_DEGRADATION_CSV,
    FALLBACK_SALIENCY_DEGRADATION_CSV,
)

auc_delta, auc_source = _read_existing_csv(
    EXPANDED_AUC_GUIDED_MINUS_RANDOM_CSV,
    FALLBACK_AUC_GUIDED_MINUS_RANDOM_CSV,
)

pert_method_stability, pert_method_source = _read_existing_csv(
    EXPANDED_PERT_METHOD_STABILITY_CSV,
)

pert_delta, pert_delta_source = _read_existing_csv(
    EXPANDED_PERT_ODOC_MINUS_RANDOM_CSV,
    FALLBACK_PERT_ODOC_MINUS_RANDOM_CSV,
)

saliency_deg = _patient_overall(saliency_deg)
auc_delta = _patient_overall(auc_delta)
pert_delta = _patient_overall(pert_delta)

# If Cell 15 method-stability table is absent, derive method stability from perturbation deltas.
if pert_method_stability.empty and not pert_delta.empty:
    rows = []
    block = pert_delta.copy()
    if "perturbation_method" not in block.columns:
        block["perturbation_method"] = "unknown"
    for (model_name, method), g in block.groupby(["model_name", "perturbation_method"], dropna=False):
        support = g["support_status"].astype(str).map(_norm_status) if "support_status" in g.columns else pd.Series(["not_supported"] * len(g))
        mean_vals = pd.to_numeric(g.get("mean_delta", g.get("mean", np.nan)), errors="coerce")
        rows.append({
            "model_name": model_name,
            "perturbation_method": method,
            "stratum_type": "overall",
            "stratum_label": "overall",
            "level": "patient",
            "n_settings": int(len(g)),
            "n_supported_settings": int((support == "supported").sum()),
            "n_directional_settings": int((mean_vals > 0).sum()),
            "n_not_supported_settings": int((support == "not_supported").sum()),
            "supported_setting_fraction": float((support == "supported").mean()) if len(g) else np.nan,
            "directional_setting_fraction": float((mean_vals > 0).mean()) if len(g) else np.nan,
        })
    pert_method_stability = pd.DataFrame(rows)
    pert_method_source = "derived_from_perturbation_delta_table"

if pert_method_stability is not None and not pert_method_stability.empty:
    if "level" in pert_method_stability.columns:
        pert_method_stability = pert_method_stability[
            pert_method_stability["level"].astype(str).eq("patient")
        ].copy()
    if "stratum_type" in pert_method_stability.columns:
        pert_method_stability = pert_method_stability[
            pert_method_stability["stratum_type"].astype(str).eq("overall")
        ].copy()

model_names = _get_model_names(saliency_deg, auc_delta, pert_method_stability, pert_delta)

if not model_names:
    raise RuntimeError(
        "No model-level evidence tables were found. "
        "Run Cells 6-11 and preferably Cell 15 before this claim-boundary cell."
    )

# -------------------------------------------------------------------------
# Status rules
# -------------------------------------------------------------------------

def classify_saliency(model_name):
    head_row = _get_delta_row(
        saliency_deg,
        model_name,
        comparison="original_minus_randomized_classifier_head",
    )
    last_row = _get_delta_row(
        saliency_deg,
        model_name,
        comparison="original_minus_randomized_last_block_plus_head",
    )

    head_supported = _is_supported(head_row) if head_row is not None else False
    last_supported = _is_supported(last_row) if last_row is not None else False
    head_directional = _is_directional(head_row) if head_row is not None else False
    last_directional = _is_directional(last_row) if last_row is not None else False

    if head_supported and last_supported:
        status = "supported"
    elif last_supported or head_supported:
        status = "partially_supported"
    elif head_directional or last_directional:
        status = "directional_only"
    else:
        status = "not_supported"

    return {
        "saliency_sanity_status": status,
        "saliency_head_delta_mean": _as_float(head_row.get("mean_delta", np.nan)) if head_row else np.nan,
        "saliency_head_delta_ci95_low": _as_float(head_row.get("ci95_low", np.nan)) if head_row else np.nan,
        "saliency_head_delta_ci95_high": _as_float(head_row.get("ci95_high", np.nan)) if head_row else np.nan,
        "saliency_last_block_delta_mean": _as_float(last_row.get("mean_delta", np.nan)) if last_row else np.nan,
        "saliency_last_block_delta_ci95_low": _as_float(last_row.get("ci95_low", np.nan)) if last_row else np.nan,
        "saliency_last_block_delta_ci95_high": _as_float(last_row.get("ci95_high", np.nan)) if last_row else np.nan,
        "saliency_interpretation": (
            "Both classifier-head and last-block randomization reduced OD/OC enrichment."
            if status == "supported"
            else "OD/OC enrichment was reduced mainly after deeper randomization; head randomization alone was insufficient."
            if status == "partially_supported"
            else "Randomization effects were weak or not consistently positive."
        ),
    }

def classify_faithfulness(model_name):
    deletion_row = _get_delta_row(
        auc_delta,
        model_name,
        curve_type="deletion",
    )
    insertion_row = _get_delta_row(
        auc_delta,
        model_name,
        curve_type="insertion",
    )

    deletion_supported = _is_supported(deletion_row) if deletion_row is not None else False
    insertion_supported = _is_supported(insertion_row) if insertion_row is not None else False
    deletion_directional = _is_directional(deletion_row) if deletion_row is not None else False
    insertion_directional = _is_directional(insertion_row) if insertion_row is not None else False

    if deletion_supported and insertion_supported:
        status = "supported"
    elif deletion_supported or insertion_supported or deletion_directional or insertion_directional:
        status = "mixed_or_partially_supported"
    else:
        status = "not_supported"

    return {
        "deletion_insertion_faithfulness_status": status,
        "deletion_auc_delta_mean": _as_float(deletion_row.get("mean_delta", np.nan)) if deletion_row else np.nan,
        "deletion_auc_delta_ci95_low": _as_float(deletion_row.get("ci95_low", np.nan)) if deletion_row else np.nan,
        "deletion_auc_delta_ci95_high": _as_float(deletion_row.get("ci95_high", np.nan)) if deletion_row else np.nan,
        "insertion_auc_delta_mean": _as_float(insertion_row.get("mean_delta", np.nan)) if insertion_row else np.nan,
        "insertion_auc_delta_ci95_low": _as_float(insertion_row.get("ci95_low", np.nan)) if insertion_row else np.nan,
        "insertion_auc_delta_ci95_high": _as_float(insertion_row.get("ci95_high", np.nan)) if insertion_row else np.nan,
        "faithfulness_interpretation": (
            "Saliency-guided deletion and insertion exceeded matched-random controls at the AUC level."
            if status == "supported"
            else "Faithfulness evidence was present for only part of the deletion/insertion protocol."
            if status == "mixed_or_partially_supported"
            else "Saliency-guided deletion/insertion did not consistently exceed matched-random controls."
        ),
    }

def _method_row(model_name, method):
    if pert_method_stability.empty:
        return None
    block = pert_method_stability[
        pert_method_stability["model_name"].astype(str).eq(str(model_name))
        & pert_method_stability["perturbation_method"].astype(str).eq(str(method))
    ]
    if block.empty:
        return None
    return block.iloc[0].to_dict()

def _method_stats(row):
    if row is None:
        return {
            "n_settings": 0,
            "n_supported": 0,
            "n_directional": 0,
            "supported_fraction": np.nan,
            "directional_fraction": np.nan,
        }
    return {
        "n_settings": int(_as_float(row.get("n_settings", 0), 0)),
        "n_supported": int(_as_float(row.get("n_supported_settings", 0), 0)),
        "n_directional": int(_as_float(row.get("n_directional_settings", 0), 0)),
        "supported_fraction": _as_float(row.get("supported_setting_fraction", np.nan)),
        "directional_fraction": _as_float(row.get("directional_setting_fraction", np.nan)),
    }

def classify_perturbation(model_name):
    blur = _method_stats(_method_row(model_name, "blur"))
    mean_fill = _method_stats(_method_row(model_name, "mean_fill"))
    noise = _method_stats(_method_row(model_name, "gaussian_noise"))

    methods = [blur, mean_fill, noise]
    available_methods = [m for m in methods if m["n_settings"] > 0]

    blur_all_supported = blur["n_settings"] > 0 and blur["n_supported"] == blur["n_settings"]
    mean_fill_all_supported = mean_fill["n_settings"] > 0 and mean_fill["n_supported"] == mean_fill["n_settings"]
    noise_all_supported = noise["n_settings"] > 0 and noise["n_supported"] == noise["n_settings"]

    all_available_supported = (
        len(available_methods) > 0
        and all(m["n_supported"] == m["n_settings"] for m in available_methods)
    )

    any_supported = any(m["n_supported"] > 0 for m in available_methods)
    any_directional = any(m["n_directional"] > 0 for m in available_methods)

    if all_available_supported:
        status = "method_invariant_supported"
    elif blur_all_supported and not (mean_fill_all_supported and noise_all_supported):
        status = "blur_supported_method_dependent"
    elif mean_fill_all_supported and not blur_all_supported:
        status = "replacement_dependent_not_stable"
    elif any_supported or any_directional:
        status = "method_dependent_partial"
    else:
        status = "not_supported"

    # Summarize mean deltas by method across patient-level overall settings.
    method_delta_summary = {}
    if not pert_delta.empty and "perturbation_method" in pert_delta.columns:
        block = pert_delta[pert_delta["model_name"].astype(str).eq(str(model_name))].copy()
        for method in ["blur", "mean_fill", "gaussian_noise"]:
            g = block[block["perturbation_method"].astype(str).eq(method)]
            if g.empty:
                method_delta_summary[f"{method}_mean_delta_across_settings"] = np.nan
                method_delta_summary[f"{method}_min_ci95_low_across_settings"] = np.nan
                method_delta_summary[f"{method}_max_ci95_high_across_settings"] = np.nan
            else:
                method_delta_summary[f"{method}_mean_delta_across_settings"] = float(pd.to_numeric(g["mean_delta"], errors="coerce").mean())
                method_delta_summary[f"{method}_min_ci95_low_across_settings"] = float(pd.to_numeric(g["ci95_low"], errors="coerce").min())
                method_delta_summary[f"{method}_max_ci95_high_across_settings"] = float(pd.to_numeric(g["ci95_high"], errors="coerce").max())

    out = {
        "perturbation_stability_status": status,
        "blur_supported_settings": blur["n_supported"],
        "blur_total_settings": blur["n_settings"],
        "blur_directional_settings": blur["n_directional"],
        "mean_fill_supported_settings": mean_fill["n_supported"],
        "mean_fill_total_settings": mean_fill["n_settings"],
        "mean_fill_directional_settings": mean_fill["n_directional"],
        "gaussian_noise_supported_settings": noise["n_supported"],
        "gaussian_noise_total_settings": noise["n_settings"],
        "gaussian_noise_directional_settings": noise["n_directional"],
        "perturbation_interpretation": (
            "OD/OC perturbation was supported across all tested perturbation methods/settings."
            if status == "method_invariant_supported"
            else "OD/OC perturbation was stable under blur but sensitive to replacement/noise perturbation design."
            if status == "blur_supported_method_dependent"
            else "OD/OC perturbation was driven mainly by replacement sensitivity and was not stable across perturbation methods."
            if status == "replacement_dependent_not_stable"
            else "OD/OC perturbation showed partial method-dependent evidence."
            if status == "method_dependent_partial"
            else "OD/OC perturbation did not exceed matched-random controls in a stable way."
        ),
    }
    out.update(method_delta_summary)
    return out

def classify_overall(saliency_status, faithfulness_status, perturbation_status):
    if (
        faithfulness_status == "supported"
        and perturbation_status in ["method_invariant_supported", "blur_supported_method_dependent"]
        and saliency_status in ["supported", "partially_supported"]
    ):
        return "strongest_bounded_aqpr_convergence"

    if (
        saliency_status == "supported"
        and faithfulness_status in ["mixed_or_partially_supported", "supported"]
        and perturbation_status in ["replacement_dependent_not_stable", "method_dependent_partial", "not_supported"]
    ):
        return "saliency_supported_but_functional_dependence_mixed"

    if saliency_status in ["supported", "partially_supported"] and faithfulness_status == "not_supported":
        return "saliency_only_or_functional_divergence"

    if faithfulness_status == "supported" or perturbation_status in ["method_invariant_supported", "blur_supported_method_dependent"]:
        return "partial_bounded_aqpr_evidence"

    return "insufficient_or_divergent_aqpr_evidence"

# -------------------------------------------------------------------------
# Build claim-boundary table
# -------------------------------------------------------------------------

rows = []

for model_name in model_names:
    sal = classify_saliency(model_name)
    fai = classify_faithfulness(model_name)
    per = classify_perturbation(model_name)

    overall_status = classify_overall(
        sal["saliency_sanity_status"],
        fai["deletion_insertion_faithfulness_status"],
        per["perturbation_stability_status"],
    )

    if overall_status == "strongest_bounded_aqpr_convergence":
        manuscript_claim = (
            f"{model_name} showed the strongest bounded AQPR profile: deletion/insertion "
            "faithfulness was supported and blur-based OD/OC perturbation was stable, "
            "but saliency randomization and alternative perturbation controls require a bounded, "
            "not universal, anatomical-reliability claim."
        )
    elif overall_status == "saliency_supported_but_functional_dependence_mixed":
        manuscript_claim = (
            f"{model_name} showed model-dependent OD/OC saliency evidence, but functional "
            "dependence was mixed: deletion/insertion evidence was partial and perturbation "
            "responses were method-dependent rather than stably anatomy-specific."
        )
    elif overall_status == "saliency_only_or_functional_divergence":
        manuscript_claim = (
            f"{model_name} showed saliency localization evidence without consistent functional "
            "faithfulness or perturbation support."
        )
    else:
        manuscript_claim = (
            f"{model_name} showed partial or insufficient convergence across AQPR endpoints; "
            "claims should remain endpoint-specific."
        )

    row = {
        "model_name": model_name,
        "audit_cohort": "all_hygd_strict_valid",
        "n_images": int(globals().get("audit_df", pd.DataFrame()).shape[0]) if "audit_df" in globals() else np.nan,
        "n_patients": int(globals().get("audit_df", pd.DataFrame())["patient_id"].nunique()) if "audit_df" in globals() and "patient_id" in globals().get("audit_df", pd.DataFrame()).columns else 288,
        **sal,
        **fai,
        **per,
        "overall_aqpr_convergence_status": overall_status,
        "manuscript_claim_boundary": manuscript_claim,
    }
    rows.append(row)

claim_df = pd.DataFrame(rows)

# Stable ordering for manuscript interpretation.
preferred_order = ["ConvNeXtTiny", "EfficientNetB0"]
claim_df["model_order"] = claim_df["model_name"].apply(
    lambda x: preferred_order.index(x) if x in preferred_order else 99
)
claim_df = claim_df.sort_values(["model_order", "model_name"]).drop(columns=["model_order"]).reset_index(drop=True)

# Compact version for manuscript/supplement display.
short_cols = [
    "model_name",
    "saliency_sanity_status",
    "deletion_insertion_faithfulness_status",
    "perturbation_stability_status",
    "overall_aqpr_convergence_status",
    "manuscript_claim_boundary",
]
claim_short = claim_df[[c for c in short_cols if c in claim_df.columns]].copy()

# Save both expanded and compatibility filenames.
claim_df.to_csv(CLAIM_BOUNDARY_CSV, index=False)
claim_df.to_csv(CLAIM_BOUNDARY_COMPAT_CSV, index=False)
claim_short.to_csv(CLAIM_BOUNDARY_SHORT_CSV, index=False)

# -------------------------------------------------------------------------
# Handoff
# -------------------------------------------------------------------------

handoff = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 14",
    "status": "PASS",
    "purpose": "Expanded all-HYGD AQPR claim-boundary audit using patient-level CIs and perturbation method-stability.",
    "input_tables": {
        "saliency_degradation": saliency_source,
        "deletion_insertion_auc_guided_minus_random": auc_source,
        "perturbation_method_stability": pert_method_source,
        "perturbation_odoc_minus_random": pert_delta_source,
    },
    "output_tables": {
        "expanded_claim_boundary": str(CLAIM_BOUNDARY_CSV),
        "compatibility_claim_boundary": str(CLAIM_BOUNDARY_COMPAT_CSV),
        "short_claim_boundary": str(CLAIM_BOUNDARY_SHORT_CSV),
    },
    "claim_status_counts": claim_df["overall_aqpr_convergence_status"].value_counts(dropna=False).to_dict(),
    "interpretation_guardrail": (
        "ConvNeXtTiny should be described as the strongest bounded AQPR profile, not as universally faithful. "
        "EfficientNetB0 should be described as saliency-supported but functionally mixed/method-dependent."
    ),
}

with open(CLAIM_BOUNDARY_JSON, "w", encoding="utf-8") as f:
    json.dump(handoff, f, indent=2)

print("\nExpanded AQPR claim-boundary table")
display(claim_short)

print("\nNumerical anchors")
anchor_cols = [
    "model_name",
    "saliency_head_delta_mean",
    "saliency_head_delta_ci95_low",
    "saliency_head_delta_ci95_high",
    "saliency_last_block_delta_mean",
    "saliency_last_block_delta_ci95_low",
    "saliency_last_block_delta_ci95_high",
    "deletion_auc_delta_mean",
    "deletion_auc_delta_ci95_low",
    "deletion_auc_delta_ci95_high",
    "insertion_auc_delta_mean",
    "insertion_auc_delta_ci95_low",
    "insertion_auc_delta_ci95_high",
    "blur_supported_settings",
    "blur_total_settings",
    "mean_fill_supported_settings",
    "mean_fill_total_settings",
    "gaussian_noise_supported_settings",
    "gaussian_noise_total_settings",
]
display(claim_df[[c for c in anchor_cols if c in claim_df.columns]])

print("\nCell 14 completed.")
print(json.dumps(handoff, indent=2))

## v3.6 reviewer robustness additions

Run these cells after the expanded audit outputs exist. For the full robustness update, run Cells 1–5, then Cells 16–19. If Cells 6–15 outputs are missing, run the notebook linearly once.

In [ ]:

# Cell 16: Endpoint definitions, exact insertion/deletion curve export, and reviewer parameter table
#
# Purpose:
# - Make endpoint definitions reproducible for the manuscript/supplement.
# - Export full insertion/deletion curve points with explicit AUC-delta interpretation.
# - Does not run model inference.
#
# Reviewer concerns addressed:
# - "How exactly is AUC delta computed?"
# - "Could you report full curves in the supplement?"
# - "Important hyperparameters are under-specified."

import json
from pathlib import Path
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Paths
# -------------------------------------------------------------------------

FAITHFULNESS_IMAGE_CSV = TABLE_DIR / "notebook15_deletion_insertion_faithfulness_image_level.csv"
FAITHFULNESS_PATIENT_CSV = TABLE_DIR / "notebook15_deletion_insertion_faithfulness_patient_level.csv"
FAITHFULNESS_AUC_IMAGE_CSV = TABLE_DIR / "notebook15_deletion_insertion_auc_image_level.csv"
FAITHFULNESS_AUC_PATIENT_CSV = TABLE_DIR / "notebook15_deletion_insertion_auc_patient_level.csv"
PERTURBATION_GRID_CSV = TABLE_DIR / "notebook15_perturbation_sensitivity_grid.csv"

FULL_CURVES_IMAGE_CSV = TABLE_DIR / "notebook15_full_insertion_deletion_curve_points_image_level.csv"
FULL_CURVES_PATIENT_CSV = TABLE_DIR / "notebook15_full_insertion_deletion_curve_points_patient_level.csv"
AUC_DELTA_FORMULA_CSV = TABLE_DIR / "notebook15_insertion_deletion_auc_delta_formula_table.csv"
ENDPOINT_PARAMETER_TABLE_CSV = TABLE_DIR / "notebook15_aqpr_endpoint_parameter_table.csv"
ENDPOINT_PARAMETER_TABLE_MD = TABLE_DIR / "notebook15_aqpr_endpoint_parameter_table.md"
CELL16_HANDOFF_JSON = RESULTS_15_DIR / "notebook15_cell16_endpoint_definition_handoff.json"

def _read_csv_if_exists(path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path, low_memory=False)
    return pd.DataFrame()

def _safe_to_markdown(df, path):
    try:
        df.to_markdown(path, index=False)
    except Exception:
        with open(path, "w", encoding="utf-8") as f:
            f.write(df.to_string(index=False))

# -------------------------------------------------------------------------
# Full insertion/deletion curves
# -------------------------------------------------------------------------

faith_image = _read_csv_if_exists(FAITHFULNESS_IMAGE_CSV)
faith_patient = _read_csv_if_exists(FAITHFULNESS_PATIENT_CSV)
faith_auc_image = _read_csv_if_exists(FAITHFULNESS_AUC_IMAGE_CSV)
faith_auc_patient = _read_csv_if_exists(FAITHFULNESS_AUC_PATIENT_CSV)

if faith_image.empty:
    raise FileNotFoundError(
        f"Missing {FAITHFULNESS_IMAGE_CSV}. Run Cell 8 before Cell 16."
    )

# Keep all curve points, including baseline fraction 0.00, because reviewers asked for full curves.
curve_cols = [
    c for c in [
        "model_name", "image_id", "patient_id", "gon_binary", "quality_tertile",
        "curve_type", "strategy", "fraction", "probability", "probability_delta",
        "p_original", "p_modified", "error"
    ] if c in faith_image.columns
]
full_curves_image = faith_image[curve_cols].copy()
full_curves_image.to_csv(FULL_CURVES_IMAGE_CSV, index=False)

if not faith_patient.empty:
    patient_cols = [
        c for c in [
            "model_name", "patient_id", "gon_binary", "quality_tertile",
            "curve_type", "strategy", "fraction", "probability_delta",
            "n_images", "error"
        ] if c in faith_patient.columns
    ]
    full_curves_patient = faith_patient[patient_cols].copy()
else:
    # Recreate a patient-level curve from image-level rows.
    group_cols = [c for c in ["model_name", "patient_id", "gon_binary", "quality_tertile", "curve_type", "strategy", "fraction"] if c in full_curves_image.columns]
    full_curves_patient = (
        full_curves_image
        .dropna(subset=["probability_delta"])
        .groupby(group_cols, dropna=False)
        .agg(
            probability_delta=("probability_delta", "mean"),
            n_images=("image_id", "nunique") if "image_id" in full_curves_image.columns else ("probability_delta", "size")
        )
        .reset_index()
    )

full_curves_patient.to_csv(FULL_CURVES_PATIENT_CSV, index=False)

# -------------------------------------------------------------------------
# Exact formula / endpoint definitions
# -------------------------------------------------------------------------

formula_rows = [
    {
        "endpoint": "positive heatmap energy",
        "symbol": "e(H,M)",
        "definition": (
            "Sum of positive normalized attribution values inside mask M divided by the "
            "sum of positive normalized attribution values over the image. In the expanded "
            "audit, Integrated Gradients absolute attributions are min-max normalized before "
            "energy computation."
        ),
        "implementation_note": "saliency = abs(IG).sum(channel); saliency = (saliency-min)/(max-min+eps); e(H,M)=sum(H[M])/(sum(H)+eps).",
    },
    {
        "endpoint": "OD/OC saliency enrichment",
        "symbol": "E_i",
        "definition": "Observed OD/OC positive heatmap energy minus the mean positive heatmap energy inside matched-random control masks.",
        "implementation_note": "E_i = e(H_i, M_i) - mean_k e(H_i, R_ik).",
    },
    {
        "endpoint": "perturbation delta",
        "symbol": "Delta_i",
        "definition": "Probability drop after OD/OC perturbation minus the mean probability drop after matched-random perturbations.",
        "implementation_note": "Delta_i = [p(x)-p(x^M)] - mean_k[p(x)-p(x^R_k)].",
    },
    {
        "endpoint": "insertion/deletion curve",
        "symbol": "C(f)",
        "definition": (
            "Probability-change curve evaluated on fraction grid f. Deletion removes "
            "saliency-ranked pixels from the original image; insertion restores saliency-ranked "
            "pixels into a globally perturbed baseline."
        ),
        "implementation_note": f"Fractions: {DELETION_FRACTIONS}. Ranking: descending Integrated Gradients attribution.",
    },
    {
        "endpoint": "AUC delta",
        "symbol": "AUC_delta",
        "definition": (
            "Trapezoidal area under the patient/image-level probability-delta curve for "
            "saliency-guided ranking minus the corresponding matched-random ranking curve."
        ),
        "implementation_note": "AUC_delta = trapz(delta_guided(f), f) - trapz(delta_random(f), f), with fractions sorted ascending.",
    },
]

formula_table = pd.DataFrame(formula_rows)
formula_table.to_csv(AUC_DELTA_FORMULA_CSV, index=False)

parameter_rows = [
    {
        "audit_component": "Primary attribution",
        "parameter": "method",
        "value": "Integrated Gradients",
        "rationale_or_guardrail": "Primary pixel-ranking method for expanded all-HYGD sanity and insertion/deletion analyses.",
    },
    {
        "audit_component": "Attribution localization checks",
        "parameter": "secondary methods",
        "value": "Grad-CAM/Grad-CAM++ where available; occlusion-based secondary check in v3.6",
        "rationale_or_guardrail": "Used only to assess method sensitivity, not to tune models.",
    },
    {
        "audit_component": "Integrated Gradients",
        "parameter": "steps",
        "value": str(IG_STEPS),
        "rationale_or_guardrail": "Fixed before external audit.",
    },
    {
        "audit_component": "Matched-random controls",
        "parameter": "primary K",
        "value": "128",
        "rationale_or_guardrail": "Shape-preserving random-control ablation added for K=32/64/128.",
    },
    {
        "audit_component": "Matched-random controls",
        "parameter": "rejection sampling",
        "value": "low target overlap, FOV containment, maximum attempts recorded",
        "rationale_or_guardrail": "Avoids trivial overlap with OD/OC while preserving mask geometry.",
    },
    {
        "audit_component": "Insertion/deletion",
        "parameter": "fractions",
        "value": ", ".join([f"{x:.2f}" for x in DELETION_FRACTIONS]),
        "rationale_or_guardrail": "Fixed curve grid; full curve points exported.",
    },
    {
        "audit_component": "Insertion/deletion",
        "parameter": "AUC integration",
        "value": "trapezoidal rule over sorted fractions",
        "rationale_or_guardrail": "No interpolation beyond observed fraction grid.",
    },
    {
        "audit_component": "Perturbation sensitivity",
        "parameter": "methods",
        "value": ", ".join(PERTURBATION_METHODS),
        "rationale_or_guardrail": "Blur, mean-fill, and Gaussian noise bound replacement dependence.",
    },
    {
        "audit_component": "Perturbation sensitivity",
        "parameter": "sigma/dilation grid",
        "value": "; ".join([f"sigma={g['sigma']}, dilation={g['dilation']}" for g in PERTURBATION_GRID]),
        "rationale_or_guardrail": "Fixed before external audit.",
    },
    {
        "audit_component": "Bootstrap",
        "parameter": "replicates",
        "value": str(BOOTSTRAP_REPEATS),
        "rationale_or_guardrail": "Patient-level bootstrap CIs; no model selection on HYGD.",
    },
    {
        "audit_component": "Random seed",
        "parameter": "seed",
        "value": str(SEED),
        "rationale_or_guardrail": "Used for reproducible control generation and subset-stability assignments.",
    },
]

parameter_table = pd.DataFrame(parameter_rows)
parameter_table.to_csv(ENDPOINT_PARAMETER_TABLE_CSV, index=False)
_safe_to_markdown(parameter_table, ENDPOINT_PARAMETER_TABLE_MD)

handoff = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 16",
    "status": "PASS",
    "purpose": "Endpoint definitions and full insertion/deletion curve export for reviewer reproducibility.",
    "output_tables": {
        "full_curves_image_level": str(FULL_CURVES_IMAGE_CSV),
        "full_curves_patient_level": str(FULL_CURVES_PATIENT_CSV),
        "auc_delta_formula": str(AUC_DELTA_FORMULA_CSV),
        "endpoint_parameter_table": str(ENDPOINT_PARAMETER_TABLE_CSV),
    },
    "n_full_curve_image_rows": int(len(full_curves_image)),
    "n_full_curve_patient_rows": int(len(full_curves_patient)),
    "n_formula_rows": int(len(formula_table)),
    "n_parameter_rows": int(len(parameter_table)),
}

with open(CELL16_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(handoff, f, indent=2)

print("\nEndpoint parameter table")
display(parameter_table)

print("\nAUC delta / endpoint formula table")
display(formula_table)

print("\nCell 16 completed.")
print(json.dumps(handoff, indent=2))


In [ ]:
# Cell 17: Matched-random control robustness ablation
#
# Purpose:
# - Provide the reviewer-requested small but meaningful ablation of matched-random settings.
# - Saliency enrichment ablation is run on all available strict-valid HYGD images with cached IG maps.
# - Perturbation random-control ablation is run on a balanced sensitivity subset by default because it requires model inference.
#
# Reviewer concerns addressed:
# - "Can you provide a quantitative ablation on the matched-random mask generator varying K, IoU threshold, FOV containment?"
# - "How robust are enrichment and perturbation deltas to K and matched-random constraints?"

import json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

# -------------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------------

MATCHED_RANDOM_K_VALUES = [32, 64, 128]
MATCHED_RANDOM_MAX_IOU_VALUES = [0.01, 0.03, 0.05]
MATCHED_RANDOM_FOV_CONTAINMENT = 0.985
MATCHED_RANDOM_MAX_ATTEMPTS = 5000

RUN_MATCHED_RANDOM_SALIENCY_ABLATION = True
RUN_MATCHED_RANDOM_PERTURBATION_ABLATION = True

# Perturbation ablation uses a balanced subset by default because it performs additional model inference.
# Set to None to use all eligible HYGD images.
PERTURBATION_ABLATION_IMAGES_PER_LABEL_QUALITY = 20
PERTURBATION_ABLATION_K_VALUES = [8, 16, 32]
PERTURBATION_ABLATION_MAX_IOU_VALUES = [0.01, 0.03, 0.05]
PERTURBATION_ABLATION_SETTING = {"sigma": 8, "dilation": 7, "perturbation_method": "blur"}

SUBSET_CSV = TABLE_DIR / "notebook15_hygd_all_strict_valid_core_audit_cohort.csv"
STABILITY_SUBSET_CSV = TABLE_DIR / "notebook15_balanced_subset_stability_assignments.csv"
SALIENCY_ARRAY_INDEX_CSV = TABLE_DIR / "notebook15_saliency_array_index.csv"

MR_ABLATION_SALIENCY_IMAGE_CSV = TABLE_DIR / "notebook15_matched_random_ablation_saliency_image_level.csv"
MR_ABLATION_SALIENCY_PATIENT_CSV = TABLE_DIR / "notebook15_matched_random_ablation_saliency_patient_level.csv"
MR_ABLATION_SALIENCY_SUMMARY_CSV = TABLE_DIR / "notebook15_matched_random_ablation_saliency_summary.csv"

MR_ABLATION_PERT_IMAGE_CSV = TABLE_DIR / "notebook15_matched_random_ablation_perturbation_image_level.csv"
MR_ABLATION_PERT_PATIENT_CSV = TABLE_DIR / "notebook15_matched_random_ablation_perturbation_patient_level.csv"
MR_ABLATION_PERT_SUMMARY_CSV = TABLE_DIR / "notebook15_matched_random_ablation_perturbation_summary.csv"

MR_ABLATION_HANDOFF_JSON = RESULTS_15_DIR / "notebook15_cell17_matched_random_ablation_handoff.json"

# -------------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------------

def _read_csv(path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path, low_memory=False)
    return pd.DataFrame()

def _bootstrap_ci(values, n_boot=5000, seed=411):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(np.mean(vals))
    if len(vals) == 1:
        return mean, mean, mean, 1
    rng = np.random.default_rng(seed)
    boot = rng.choice(vals, size=(int(n_boot), len(vals)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return mean, float(lo), float(hi), int(len(vals))

def _stable_seed(*parts, modulo=2_000_000_000):
    return int(hashlib.md5("||".join(map(str, parts)).encode("utf-8")).hexdigest()[:12], 16) % modulo

def _load_mask_array_local(path, image_size=IMAGE_SIZE):
    if path is None or str(path) == "" or str(path).lower() in ["nan", "none"]:
        return None
    p = Path(str(path))
    if not p.exists():
        return None
    try:
        if p.suffix.lower() == ".npy":
            arr = np.load(p)
        else:
            arr = np.asarray(Image.open(p).convert("L"))
        if arr.shape[:2] != (image_size, image_size):
            arr = np.asarray(Image.fromarray(arr.astype(np.uint8)).resize((image_size, image_size), resample=Image.NEAREST))
        return arr > 0
    except Exception:
        return None

def _infer_odoc_mask_local(row, image_size=IMAGE_SIZE):
    if "infer_od_oc_mask" in globals():
        try:
            m, src, detail = infer_od_oc_mask(row, image_size=image_size)
            return m.astype(bool), src, detail
        except Exception:
            pass

    masks = []
    details = []
    for col in ["od_oc_mask_path", "od_mask_path", "oc_mask_path"]:
        if col in row.index:
            m = _load_mask_array_local(row.get(col), image_size=image_size)
            if m is not None:
                masks.append(m)
                details.append(str(row.get(col)))
    if masks:
        out = np.zeros((image_size, image_size), dtype=bool)
        for m in masks:
            out |= m.astype(bool)
        return out, "mask_path_local", ";".join(details)

    yy, xx = np.mgrid[:image_size, :image_size]
    cx, cy = image_size // 2, image_size // 2
    radius = int(image_size * 0.13)
    fallback = ((xx - cx) ** 2 + (yy - cy) ** 2) <= radius ** 2
    return fallback, "center_circle_fallback", ""

def _image_fov_mask(image_path, image_size=IMAGE_SIZE):
    p = Path(str(image_path))
    if not p.exists():
        return np.ones((image_size, image_size), dtype=bool)
    try:
        img = Image.open(p).convert("RGB").resize((image_size, image_size))
        arr = np.asarray(img).astype(np.float32) / 255.0
        gray = arr.mean(axis=2)
        # Conservative fundus field estimate. If it fails, use full image to avoid excluding cases.
        mask = gray > 0.03
        if mask.mean() < 0.25:
            return np.ones((image_size, image_size), dtype=bool)
        return mask
    except Exception:
        return np.ones((image_size, image_size), dtype=bool)

def _iou(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / union) if union > 0 else 0.0

def _random_translated_mask(mask, fov_mask, seed, max_iou=0.03, fov_containment=0.985, max_attempts=5000):
    mask = mask.astype(bool)
    fov_mask = fov_mask.astype(bool)
    h, w = mask.shape
    coords = np.argwhere(mask)
    n = len(coords)
    if n == 0:
        return None, {"accepted": False, "reason": "empty_target", "attempts": 0}

    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0)
    bh = int(y1 - y0 + 1)
    bw = int(x1 - x0 + 1)
    rel = coords - np.array([y0, x0])

    rng = np.random.default_rng(seed)
    best_mask = None
    best_score = -np.inf
    best_diag = None

    for attempt in range(1, int(max_attempts) + 1):
        ny0 = int(rng.integers(0, max(1, h - bh + 1)))
        nx0 = int(rng.integers(0, max(1, w - bw + 1)))
        new_coords = rel + np.array([ny0, nx0])
        out = np.zeros_like(mask, dtype=bool)
        out[new_coords[:, 0], new_coords[:, 1]] = True

        containment = float(out[fov_mask].sum() / max(1, out.sum()))
        overlap_iou = _iou(mask, out)

        # Track nearest valid candidate for diagnostic fallback.
        score = containment - 5.0 * max(0.0, overlap_iou - max_iou)
        if score > best_score:
            best_score = score
            best_mask = out
            best_diag = {
                "accepted": False,
                "reason": "best_fallback_after_rejection",
                "attempts": attempt,
                "fov_containment": containment,
                "target_iou": overlap_iou,
            }

        if containment >= fov_containment and overlap_iou <= max_iou:
            return out, {
                "accepted": True,
                "reason": "accepted",
                "attempts": attempt,
                "fov_containment": containment,
                "target_iou": overlap_iou,
            }

    return best_mask, best_diag

def _random_mask_coord_list(target_mask, fov_mask, k, max_iou, seed):
    masks = []
    diagnostics = []
    for kk in range(int(k)):
        m, diag = _random_translated_mask(
            target_mask,
            fov_mask,
            seed=_stable_seed(seed, kk, max_iou, k),
            max_iou=max_iou,
            fov_containment=MATCHED_RANDOM_FOV_CONTAINMENT,
            max_attempts=MATCHED_RANDOM_MAX_ATTEMPTS,
        )
        if m is not None:
            masks.append(np.argwhere(m))
        diagnostics.append(diag)
    return masks, diagnostics

def _summarize_group(df, value_col, group_cols, summary_name):
    rows = []
    if df.empty or value_col not in df.columns:
        return pd.DataFrame()
    for key, g in df.groupby(group_cols, dropna=False):
        key_tuple = key if isinstance(key, tuple) else (key,)
        vals = pd.to_numeric(g[value_col], errors="coerce").dropna()
        seed = _stable_seed(summary_name, key_tuple, value_col, modulo=1_000_000)
        mean, lo, hi, n = _bootstrap_ci(vals, n_boot=BOOTSTRAP_REPEATS, seed=BOOTSTRAP_SEED + seed)
        row = dict(zip(group_cols, key_tuple))
        row.update({
            "summary_name": summary_name,
            "metric": value_col,
            "n_units": int(n),
            "mean": mean,
            "median": float(vals.median()) if len(vals) else np.nan,
            "ci95_low": lo,
            "ci95_high": hi,
            "support_status": "supported" if pd.notna(lo) and lo > 0 else "directional_only" if pd.notna(mean) and mean > 0 else "not_supported",
        })
        rows.append(row)
    return pd.DataFrame(rows)


# -------------------------------------------------------------------------
# Local perturbation/insertion helpers for reviewer-robustness cells
# -------------------------------------------------------------------------
# These fallbacks make Cells 17-18 independent from the heavier earlier
# deletion/perturbation cells. If the original notebook helpers already
# exist, they are preserved.

if "apply_mask_mean_fill_tensor" not in globals():
    def apply_mask_mean_fill_tensor(x_norm, mask):
        mask_np = np.asarray(mask).astype(bool)
        x01 = inverse_normalize_tensor(x_norm)
        mask_t = torch.as_tensor(mask_np, device=x_norm.device, dtype=torch.bool).view(1, 1, mask_np.shape[0], mask_np.shape[1])
        keep = ~mask_t
        denom = keep.float().sum(dim=(2, 3), keepdim=True).clamp_min(1.0)
        mean_val = (x01 * keep.float()).sum(dim=(2, 3), keepdim=True) / denom
        out = torch.where(mask_t.expand_as(x01), mean_val.expand_as(x01), x01)
        return normalize_tensor_for_model(out)

if "restore_mask_from_original" not in globals():
    def restore_mask_from_original(x_base_norm, x_original_norm, mask):
        mask_np = np.asarray(mask).astype(bool)
        base01 = inverse_normalize_tensor(x_base_norm)
        orig01 = inverse_normalize_tensor(x_original_norm)
        mask_t = torch.as_tensor(mask_np, device=x_original_norm.device, dtype=torch.bool).view(1, 1, mask_np.shape[0], mask_np.shape[1])
        out = torch.where(mask_t.expand_as(orig01), orig01, base01)
        return normalize_tensor_for_model(out)

if "apply_perturbation_by_method" not in globals():
    def apply_perturbation_by_method(x_norm, mask, method="blur", sigma=8, seed=411):
        method = str(method).lower()
        mask_np = np.asarray(mask).astype(bool)
        if method in ["blur", "gaussian_blur"]:
            if "apply_mask_blur_tensor" in globals():
                return apply_mask_blur_tensor(x_norm, mask_np, sigma=float(sigma))
            return apply_mask_mean_fill_tensor(x_norm, mask_np)
        if method in ["mean", "mean_fill", "mean-fill"]:
            return apply_mask_mean_fill_tensor(x_norm, mask_np)
        if method in ["gaussian_noise", "noise"]:
            x01 = inverse_normalize_tensor(x_norm)
            rng = np.random.default_rng(int(seed))
            noise = rng.normal(loc=0.5, scale=0.20, size=tuple(x01.shape)).astype(np.float32)
            noise_t = torch.from_numpy(noise).to(x_norm.device).clamp(0.0, 1.0)
            mask_t = torch.as_tensor(mask_np, device=x_norm.device, dtype=torch.bool).view(1, 1, mask_np.shape[0], mask_np.shape[1])
            out = torch.where(mask_t.expand_as(x01), noise_t, x01)
            return normalize_tensor_for_model(out)
        raise ValueError(f"Unsupported perturbation method: {method}")

# -------------------------------------------------------------------------
# Saliency enrichment ablation using cached original IG maps
# -------------------------------------------------------------------------

subset = _read_csv(SUBSET_CSV)
saliency_index = _read_csv(SALIENCY_ARRAY_INDEX_CSV)

if subset.empty:
    raise FileNotFoundError(f"Missing expanded audit cohort: {SUBSET_CSV}. Run Cell 3 first.")
if saliency_index.empty:
    raise FileNotFoundError(f"Missing saliency array index: {SALIENCY_ARRAY_INDEX_CSV}. Run Cell 6 first.")

saliency_index = saliency_index[
    saliency_index["randomization_condition"].astype(str).eq("original")
].copy()

saliency_path_map = {
    (str(r["model_name"]), str(r["image_id"])): str(r["saliency_npy_path"])
    for _, r in saliency_index.iterrows()
    if str(r.get("saliency_npy_path", "")) not in ["", "nan", "None"]
}

saliency_rows = []
if RUN_MATCHED_RANDOM_SALIENCY_ABLATION:
    print("Running matched-random saliency ablation on cached IG maps...")
    for row_i, row in subset.iterrows():
        image_id = str(row["image_id"])
        patient_id = str(row["patient_id"])
        gon_binary = row.get("gon_binary", np.nan)
        quality = row.get("quality_tertile", "")
        image_path = row.get("image_path", "")

        target_mask, mask_source, mask_detail = _infer_odoc_mask_local(row, IMAGE_SIZE)
        fov_mask = _image_fov_mask(image_path, IMAGE_SIZE)

        for max_iou in MATCHED_RANDOM_MAX_IOU_VALUES:
            # Generate maximum K once, then reuse prefixes for K=32/64/128.
            max_k = max(MATCHED_RANDOM_K_VALUES)
            random_coords, diagnostics = _random_mask_coord_list(
                target_mask,
                fov_mask,
                k=max_k,
                max_iou=max_iou,
                seed=_stable_seed("saliency_ablation", image_id, max_iou),
            )
            accepted_fraction = float(np.mean([d.get("accepted", False) for d in diagnostics])) if diagnostics else np.nan
            mean_attempts = float(np.mean([d.get("attempts", np.nan) for d in diagnostics])) if diagnostics else np.nan
            mean_iou = float(np.nanmean([d.get("target_iou", np.nan) for d in diagnostics])) if diagnostics else np.nan
            mean_fov = float(np.nanmean([d.get("fov_containment", np.nan) for d in diagnostics])) if diagnostics else np.nan

            for model_name in MODEL_NAMES_EFFECTIVE if "MODEL_NAMES_EFFECTIVE" in globals() else MODEL_NAMES:
                sal_path = saliency_path_map.get((str(model_name), image_id), "")
                if not sal_path or not Path(sal_path).exists():
                    continue
                sal = np.load(sal_path).astype(np.float32)
                sal = np.maximum(sal, 0)
                sal = sal / (sal.sum() + 1e-8)
                observed = float(sal[target_mask].sum()) if target_mask.any() else np.nan

                # Precompute random masses for max K.
                random_masses = []
                for coords in random_coords:
                    if coords is None or len(coords) == 0:
                        continue
                    random_masses.append(float(sal[coords[:, 0], coords[:, 1]].sum()))
                random_masses = np.asarray(random_masses, dtype=float)

                for k in MATCHED_RANDOM_K_VALUES:
                    used = random_masses[:int(k)]
                    random_mean = float(np.mean(used)) if len(used) else np.nan
                    random_std = float(np.std(used, ddof=1)) if len(used) > 1 else np.nan
                    delta = observed - random_mean if np.isfinite(observed) and np.isfinite(random_mean) else np.nan
                    saliency_rows.append({
                        "model_name": model_name,
                        "image_id": image_id,
                        "patient_id": patient_id,
                        "gon_binary": gon_binary,
                        "quality_tertile": quality,
                        "xai_method": "IntegratedGradients",
                        "k_random_masks": int(k),
                        "max_target_iou": float(max_iou),
                        "fov_containment_threshold": float(MATCHED_RANDOM_FOV_CONTAINMENT),
                        "observed_odoc_energy": observed,
                        "matched_random_energy_mean": random_mean,
                        "matched_random_energy_std": random_std,
                        "odoc_minus_random_energy": delta,
                        "mask_source": mask_source,
                        "mask_source_detail": mask_detail,
                        "accepted_random_mask_fraction": accepted_fraction,
                        "mean_rejection_attempts": mean_attempts,
                        "mean_generated_iou": mean_iou,
                        "mean_generated_fov_containment": mean_fov,
                    })

saliency_ablation_image = pd.DataFrame(saliency_rows)
saliency_ablation_image.to_csv(MR_ABLATION_SALIENCY_IMAGE_CSV, index=False)

if not saliency_ablation_image.empty:
    patient_cols = ["model_name", "patient_id", "gon_binary", "quality_tertile", "xai_method", "k_random_masks", "max_target_iou"]
    saliency_ablation_patient = (
        saliency_ablation_image
        .groupby(patient_cols, dropna=False)
        .agg(
            odoc_minus_random_energy=("odoc_minus_random_energy", "mean"),
            observed_odoc_energy=("observed_odoc_energy", "mean"),
            matched_random_energy_mean=("matched_random_energy_mean", "mean"),
            n_images=("image_id", "nunique"),
            accepted_random_mask_fraction=("accepted_random_mask_fraction", "mean"),
            mean_generated_iou=("mean_generated_iou", "mean"),
            mean_generated_fov_containment=("mean_generated_fov_containment", "mean"),
        )
        .reset_index()
    )
else:
    saliency_ablation_patient = pd.DataFrame()

saliency_ablation_patient.to_csv(MR_ABLATION_SALIENCY_PATIENT_CSV, index=False)

sal_summary = _summarize_group(
    saliency_ablation_patient,
    value_col="odoc_minus_random_energy",
    group_cols=["model_name", "xai_method", "k_random_masks", "max_target_iou"],
    summary_name="patient_level_saliency_matched_random_ablation",
)
sal_summary.to_csv(MR_ABLATION_SALIENCY_SUMMARY_CSV, index=False)

# -------------------------------------------------------------------------
# Perturbation matched-random ablation on a balanced sensitivity subset
# -------------------------------------------------------------------------

pert_rows = []

can_run_perturbation = (
    RUN_MATCHED_RANDOM_PERTURBATION_ABLATION
    and "models" in globals()
    and "predict_probability" in globals()
    and "load_image_tensor" in globals()
    and "apply_perturbation_by_method" in globals()
)

if can_run_perturbation:
    # Use balanced subset for robustness ablation; primary perturbation remains all-HYGD from Cell 10.
    if PERTURBATION_ABLATION_IMAGES_PER_LABEL_QUALITY is None:
        pert_subset = subset.copy()
        subset_policy = "all_hygd_strict_valid"
    else:
        chosen = []
        for (q, y), g in subset.groupby(["quality_tertile", "gon_binary"], dropna=False):
            chosen.append(
                g.sample(
                    n=min(int(PERTURBATION_ABLATION_IMAGES_PER_LABEL_QUALITY), len(g)),
                    random_state=_stable_seed("perturb_subset", q, y, SEED, modulo=1_000_000),
                )
            )
        pert_subset = pd.concat(chosen, ignore_index=True).drop_duplicates(subset=["image_id"])
        subset_policy = f"balanced_{PERTURBATION_ABLATION_IMAGES_PER_LABEL_QUALITY}_per_label_quality"

    print(f"Running perturbation matched-random ablation on {len(pert_subset)} images ({subset_policy}).")

    sigma = float(PERTURBATION_ABLATION_SETTING["sigma"])
    dilation = int(PERTURBATION_ABLATION_SETTING["dilation"])
    method = str(PERTURBATION_ABLATION_SETTING["perturbation_method"])

    for _, row in pert_subset.iterrows():
        image_id = str(row["image_id"])
        patient_id = str(row["patient_id"])
        image_path = Path(str(row["image_path"]))
        if not image_path.exists():
            continue

        target_base, mask_source, mask_detail = _infer_odoc_mask_local(row, IMAGE_SIZE)
        target_mask = dilate_mask(target_base, dilation) if "dilate_mask" in globals() else target_base
        fov_mask = _image_fov_mask(image_path, IMAGE_SIZE)

        try:
            x = load_image_tensor(image_path, IMAGE_SIZE).to(DEVICE)
        except Exception as exc:
            print(f"Skipping image {image_id}: {type(exc).__name__}: {exc}")
            continue

        for max_iou in PERTURBATION_ABLATION_MAX_IOU_VALUES:
            max_k = max(PERTURBATION_ABLATION_K_VALUES)
            random_coords, diagnostics = _random_mask_coord_list(
                target_mask,
                fov_mask,
                k=max_k,
                max_iou=max_iou,
                seed=_stable_seed("pert_ablation", image_id, max_iou, sigma, dilation),
            )
            random_masks = []
            for coords in random_coords:
                m = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
                if coords is not None and len(coords) > 0:
                    m[coords[:, 0], coords[:, 1]] = True
                random_masks.append(m)

            accepted_fraction = float(np.mean([d.get("accepted", False) for d in diagnostics])) if diagnostics else np.nan

            for model_name in MODEL_NAMES_EFFECTIVE if "MODEL_NAMES_EFFECTIVE" in globals() else MODEL_NAMES:
                if model_name not in models:
                    continue
                model = models[model_name]
                try:
                    p_original = predict_probability(model, x)
                    x_target = apply_perturbation_by_method(
                        x,
                        target_mask,
                        method=method,
                        sigma=sigma,
                        seed=_stable_seed("target", image_id, model_name, sigma, dilation),
                    )
                    p_target = predict_probability(model, x_target)
                    target_drop = p_original - p_target

                    random_drops = []
                    for rm_i, rm in enumerate(random_masks):
                        x_random = apply_perturbation_by_method(
                            x,
                            rm,
                            method=method,
                            sigma=sigma,
                            seed=_stable_seed("random", image_id, model_name, rm_i, sigma, dilation),
                        )
                        p_random = predict_probability(model, x_random)
                        random_drops.append(float(p_original - p_random))
                    random_drops = np.asarray(random_drops, dtype=float)

                    for k in PERTURBATION_ABLATION_K_VALUES:
                        used = random_drops[:int(k)]
                        random_mean = float(np.mean(used)) if len(used) else np.nan
                        delta = float(target_drop - random_mean) if np.isfinite(target_drop) and np.isfinite(random_mean) else np.nan
                        pert_rows.append({
                            "model_name": model_name,
                            "image_id": image_id,
                            "patient_id": patient_id,
                            "gon_binary": row.get("gon_binary", np.nan),
                            "quality_tertile": row.get("quality_tertile", ""),
                            "subset_policy": subset_policy,
                            "perturbation_method": method,
                            "sigma": sigma,
                            "dilation": dilation,
                            "k_random_masks": int(k),
                            "max_target_iou": float(max_iou),
                            "fov_containment_threshold": float(MATCHED_RANDOM_FOV_CONTAINMENT),
                            "p_original": float(p_original),
                            "target_drop": float(target_drop),
                            "matched_random_drop_mean": random_mean,
                            "odoc_minus_random_drop": delta,
                            "accepted_random_mask_fraction": accepted_fraction,
                            "mask_source": mask_source,
                            "mask_source_detail": mask_detail,
                        })
                except Exception as exc:
                    pert_rows.append({
                        "model_name": model_name,
                        "image_id": image_id,
                        "patient_id": patient_id,
                        "gon_binary": row.get("gon_binary", np.nan),
                        "quality_tertile": row.get("quality_tertile", ""),
                        "subset_policy": subset_policy,
                        "perturbation_method": method,
                        "sigma": sigma,
                        "dilation": dilation,
                        "error": f"{type(exc).__name__}: {exc}",
                    })
else:
    print("Perturbation matched-random ablation skipped. Run Cells 4-5 before this cell to enable model inference.")

pert_ablation_image = pd.DataFrame(pert_rows)
pert_ablation_image.to_csv(MR_ABLATION_PERT_IMAGE_CSV, index=False)

if not pert_ablation_image.empty and "error" in pert_ablation_image.columns:
    pert_valid = pert_ablation_image[pert_ablation_image["error"].isna() | pert_ablation_image["error"].astype(str).isin(["", "nan", "None"])].copy()
else:
    pert_valid = pert_ablation_image.copy()

if not pert_valid.empty:
    patient_cols = ["model_name", "patient_id", "gon_binary", "quality_tertile", "subset_policy", "perturbation_method", "sigma", "dilation", "k_random_masks", "max_target_iou"]
    pert_ablation_patient = (
        pert_valid
        .groupby(patient_cols, dropna=False)
        .agg(
            odoc_minus_random_drop=("odoc_minus_random_drop", "mean"),
            target_drop=("target_drop", "mean"),
            matched_random_drop_mean=("matched_random_drop_mean", "mean"),
            n_images=("image_id", "nunique"),
            accepted_random_mask_fraction=("accepted_random_mask_fraction", "mean"),
        )
        .reset_index()
    )
else:
    pert_ablation_patient = pd.DataFrame()

pert_ablation_patient.to_csv(MR_ABLATION_PERT_PATIENT_CSV, index=False)

pert_summary = _summarize_group(
    pert_ablation_patient,
    value_col="odoc_minus_random_drop",
    group_cols=["model_name", "perturbation_method", "sigma", "dilation", "k_random_masks", "max_target_iou"],
    summary_name="patient_level_perturbation_matched_random_ablation",
)
pert_summary.to_csv(MR_ABLATION_PERT_SUMMARY_CSV, index=False)

handoff = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 17",
    "status": "PASS",
    "purpose": "Matched-random control robustness ablation for saliency enrichment and perturbation deltas.",
    "configuration": {
        "saliency_k_values": MATCHED_RANDOM_K_VALUES,
        "saliency_max_iou_values": MATCHED_RANDOM_MAX_IOU_VALUES,
        "fov_containment_threshold": MATCHED_RANDOM_FOV_CONTAINMENT,
        "max_attempts": MATCHED_RANDOM_MAX_ATTEMPTS,
        "perturbation_k_values": PERTURBATION_ABLATION_K_VALUES,
        "perturbation_max_iou_values": PERTURBATION_ABLATION_MAX_IOU_VALUES,
        "perturbation_setting": PERTURBATION_ABLATION_SETTING,
        "perturbation_subset_policy": "skipped" if pert_ablation_image.empty else str(pert_ablation_image.get("subset_policy", pd.Series(["unknown"])).iloc[0]),
    },
    "output_tables": {
        "saliency_image_level": str(MR_ABLATION_SALIENCY_IMAGE_CSV),
        "saliency_patient_level": str(MR_ABLATION_SALIENCY_PATIENT_CSV),
        "saliency_summary": str(MR_ABLATION_SALIENCY_SUMMARY_CSV),
        "perturbation_image_level": str(MR_ABLATION_PERT_IMAGE_CSV),
        "perturbation_patient_level": str(MR_ABLATION_PERT_PATIENT_CSV),
        "perturbation_summary": str(MR_ABLATION_PERT_SUMMARY_CSV),
    },
    "n_saliency_image_rows": int(len(saliency_ablation_image)),
    "n_saliency_patient_rows": int(len(saliency_ablation_patient)),
    "n_perturbation_image_rows": int(len(pert_ablation_image)),
    "n_perturbation_patient_rows": int(len(pert_ablation_patient)),
    "interpretation_guardrail": (
        "This is a robustness sensitivity analysis for the matched-random generator. "
        "The primary all-HYGD endpoint estimates remain those from the fixed preregistered settings."
    ),
}

with open(MR_ABLATION_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(handoff, f, indent=2)

print("\nMatched-random saliency ablation summary")
display(sal_summary)

print("\nMatched-random perturbation ablation summary")
display(pert_summary)

print("\nCell 17 completed.")
print(json.dumps(handoff, indent=2))


In [ ]:
# Cell 18: Secondary occlusion-based attribution robustness check
#
# Purpose:
# - Add a small additional attribution-method check without requiring RISE.
# - Uses patch-occlusion sensitivity as a secondary, model-agnostic ranking signal.
# - Compares OD/OC enrichment and insertion/deletion AUC direction with IG on a balanced HYGD subset.
#
# Reviewer concern addressed:
# - "How sensitive are conclusions to the attribution method?"
# - "Do occlusion-based rankings produce similar insertion/deletion outcomes as IG?"

import json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

RUN_OCCLUSION_SECONDARY_ATTRIBUTION = True
OCCLUSION_IMAGES_PER_LABEL_QUALITY = 30
OCCLUSION_PATCH_GRID = 14
OCCLUSION_DELETION_FRACTIONS = [0.00, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
OCCLUSION_RANDOM_K = 32
OCCLUSION_MAX_IOU = 0.03

SUBSET_CSV = TABLE_DIR / "notebook15_hygd_all_strict_valid_core_audit_cohort.csv"
SALIENCY_ARRAY_INDEX_CSV = TABLE_DIR / "notebook15_saliency_array_index.csv"

OCCLUSION_ATTR_IMAGE_CSV = TABLE_DIR / "notebook15_secondary_occlusion_attribution_image_level.csv"
OCCLUSION_ATTR_PATIENT_CSV = TABLE_DIR / "notebook15_secondary_occlusion_attribution_patient_level.csv"
OCCLUSION_ATTR_SUMMARY_CSV = TABLE_DIR / "notebook15_secondary_occlusion_attribution_summary.csv"
OCCLUSION_CURVE_IMAGE_CSV = TABLE_DIR / "notebook15_secondary_occlusion_deletion_insertion_curve_image_level.csv"
OCCLUSION_CURVE_PATIENT_CSV = TABLE_DIR / "notebook15_secondary_occlusion_deletion_insertion_curve_patient_level.csv"
OCCLUSION_AUC_IMAGE_CSV = TABLE_DIR / "notebook15_secondary_occlusion_deletion_insertion_auc_image_level.csv"
OCCLUSION_AUC_PATIENT_CSV = TABLE_DIR / "notebook15_secondary_occlusion_deletion_insertion_auc_patient_level.csv"
OCCLUSION_AUC_SUMMARY_CSV = TABLE_DIR / "notebook15_secondary_occlusion_deletion_insertion_auc_summary.csv"
OCCLUSION_HANDOFF_JSON = RESULTS_15_DIR / "notebook15_cell18_secondary_occlusion_handoff.json"

def _read_csv(path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path, low_memory=False)
    return pd.DataFrame()

def _stable_seed(*parts, modulo=2_000_000_000):
    return int(hashlib.md5("||".join(map(str, parts)).encode("utf-8")).hexdigest()[:12], 16) % modulo

def _bootstrap_ci(values, n_boot=5000, seed=411):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(np.mean(vals))
    if len(vals) == 1:
        return mean, mean, mean, 1
    rng = np.random.default_rng(seed)
    boot = rng.choice(vals, size=(int(n_boot), len(vals)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return mean, float(lo), float(hi), int(len(vals))

def _trapezoid_auc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    order = np.argsort(x)
    if len(x) < 2:
        return np.nan
    return float(np.trapz(y[order], x[order]))


# -------------------------------------------------------------------------
# Local perturbation/insertion helpers for reviewer-robustness cells
# -------------------------------------------------------------------------
# These fallbacks make Cells 17-18 independent from the heavier earlier
# deletion/perturbation cells. If the original notebook helpers already
# exist, they are preserved.

if "apply_mask_mean_fill_tensor" not in globals():
    def apply_mask_mean_fill_tensor(x_norm, mask):
        mask_np = np.asarray(mask).astype(bool)
        x01 = inverse_normalize_tensor(x_norm)
        mask_t = torch.as_tensor(mask_np, device=x_norm.device, dtype=torch.bool).view(1, 1, mask_np.shape[0], mask_np.shape[1])
        keep = ~mask_t
        denom = keep.float().sum(dim=(2, 3), keepdim=True).clamp_min(1.0)
        mean_val = (x01 * keep.float()).sum(dim=(2, 3), keepdim=True) / denom
        out = torch.where(mask_t.expand_as(x01), mean_val.expand_as(x01), x01)
        return normalize_tensor_for_model(out)

if "restore_mask_from_original" not in globals():
    def restore_mask_from_original(x_base_norm, x_original_norm, mask):
        mask_np = np.asarray(mask).astype(bool)
        base01 = inverse_normalize_tensor(x_base_norm)
        orig01 = inverse_normalize_tensor(x_original_norm)
        mask_t = torch.as_tensor(mask_np, device=x_original_norm.device, dtype=torch.bool).view(1, 1, mask_np.shape[0], mask_np.shape[1])
        out = torch.where(mask_t.expand_as(orig01), orig01, base01)
        return normalize_tensor_for_model(out)

if "apply_perturbation_by_method" not in globals():
    def apply_perturbation_by_method(x_norm, mask, method="blur", sigma=8, seed=411):
        method = str(method).lower()
        mask_np = np.asarray(mask).astype(bool)
        if method in ["blur", "gaussian_blur"]:
            if "apply_mask_blur_tensor" in globals():
                return apply_mask_blur_tensor(x_norm, mask_np, sigma=float(sigma))
            return apply_mask_mean_fill_tensor(x_norm, mask_np)
        if method in ["mean", "mean_fill", "mean-fill"]:
            return apply_mask_mean_fill_tensor(x_norm, mask_np)
        if method in ["gaussian_noise", "noise"]:
            x01 = inverse_normalize_tensor(x_norm)
            rng = np.random.default_rng(int(seed))
            noise = rng.normal(loc=0.5, scale=0.20, size=tuple(x01.shape)).astype(np.float32)
            noise_t = torch.from_numpy(noise).to(x_norm.device).clamp(0.0, 1.0)
            mask_t = torch.as_tensor(mask_np, device=x_norm.device, dtype=torch.bool).view(1, 1, mask_np.shape[0], mask_np.shape[1])
            out = torch.where(mask_t.expand_as(x01), noise_t, x01)
            return normalize_tensor_for_model(out)
        raise ValueError(f"Unsupported perturbation method: {method}")

def _mean_fill_patch_tensor(x_norm, y0, y1, x0, x1):
    # Requires inverse_normalize_tensor and normalize_tensor_for_model from Cell 4.
    x01 = inverse_normalize_tensor(x_norm)
    out = x01.clone()
    keep = torch.ones_like(out[:, :1, :, :], dtype=torch.bool)
    keep[:, :, y0:y1, x0:x1] = False
    denom = keep.float().sum(dim=(2,3), keepdim=True).clamp_min(1.0)
    mean_val = (x01 * keep.float()).sum(dim=(2,3), keepdim=True) / denom
    out[:, :, y0:y1, x0:x1] = mean_val
    return normalize_tensor_for_model(out)

def _occlusion_saliency_map(model, x, grid=14):
    h = x.shape[-2]
    w = x.shape[-1]
    p0 = predict_probability(model, x)
    sal = np.zeros((h, w), dtype=np.float32)
    for gy in range(grid):
        y0 = int(round(gy * h / grid))
        y1 = int(round((gy + 1) * h / grid))
        for gx in range(grid):
            x0 = int(round(gx * w / grid))
            x1 = int(round((gx + 1) * w / grid))
            xp = _mean_fill_patch_tensor(x, y0, y1, x0, x1)
            pp = predict_probability(model, xp)
            # Positive values mean the occluded patch reduced the positive GON+ score.
            sal[y0:y1, x0:x1] = max(0.0, float(p0 - pp))
    sal = sal - sal.min()
    sal = sal / (sal.max() + 1e-8)
    return sal.astype(np.float32), float(p0)

def _top_fraction_mask(saliency, fraction):
    sal = np.asarray(saliency, dtype=np.float32)
    if fraction <= 0:
        return np.zeros_like(sal, dtype=bool)
    n = int(round(float(fraction) * sal.size))
    n = max(1, min(n, sal.size))
    thr = np.partition(sal.reshape(-1), -n)[-n]
    return sal >= thr

def _curve_auc_from_rows(rows, curve_type, strategy):
    block = pd.DataFrame(rows)
    block = block[(block["curve_type"] == curve_type) & (block["strategy"] == strategy)]
    return _trapezoid_auc(block["fraction"], block["probability_delta"])

def _summarize(df, value_col, group_cols, summary_name):
    rows = []
    if df.empty:
        return pd.DataFrame()
    for key, g in df.groupby(group_cols, dropna=False):
        key_tuple = key if isinstance(key, tuple) else (key,)
        vals = pd.to_numeric(g[value_col], errors="coerce").dropna()
        seed = _stable_seed(summary_name, key_tuple, value_col, modulo=1_000_000)
        mean, lo, hi, n = _bootstrap_ci(vals, n_boot=BOOTSTRAP_REPEATS, seed=BOOTSTRAP_SEED + seed)
        row = dict(zip(group_cols, key_tuple))
        row.update({
            "summary_name": summary_name,
            "metric": value_col,
            "n_units": int(n),
            "mean": mean,
            "median": float(vals.median()) if len(vals) else np.nan,
            "ci95_low": lo,
            "ci95_high": hi,
            "support_status": "supported" if pd.notna(lo) and lo > 0 else "directional_only" if pd.notna(mean) and mean > 0 else "not_supported",
        })
        rows.append(row)
    return pd.DataFrame(rows)

if not RUN_OCCLUSION_SECONDARY_ATTRIBUTION:
    print("Secondary occlusion attribution check disabled.")
    occ_attr_image = pd.DataFrame()
    occ_curve_image = pd.DataFrame()
    occ_auc_image = pd.DataFrame()
else:
    required = ["models", "predict_probability", "load_image_tensor", "inverse_normalize_tensor", "normalize_tensor_for_model"]
    missing = [x for x in required if x not in globals()]
    if missing:
        raise RuntimeError(f"Run Cells 4-5 before Cell 18. Missing: {missing}")

    subset = _read_csv(SUBSET_CSV)
    if subset.empty:
        raise FileNotFoundError(f"Missing {SUBSET_CSV}. Run Cell 3 first.")

    # Balanced sensitivity subset: class x quality. This is not the primary all-HYGD result.
    chosen = []
    for (q, y), g in subset.groupby(["quality_tertile", "gon_binary"], dropna=False):
        chosen.append(
            g.sample(
                n=min(int(OCCLUSION_IMAGES_PER_LABEL_QUALITY), len(g)),
                random_state=_stable_seed("occlusion_subset", q, y, SEED, modulo=1_000_000),
            )
        )
    occ_subset = pd.concat(chosen, ignore_index=True).drop_duplicates(subset=["image_id"])

    occ_attr_rows = []
    occ_curve_rows = []
    occ_auc_rows = []

    print(f"Running secondary occlusion attribution check on {len(occ_subset)} images.")

    for _, row in occ_subset.iterrows():
        image_id = str(row["image_id"])
        patient_id = str(row["patient_id"])
        image_path = Path(str(row["image_path"]))
        if not image_path.exists():
            continue

        target_mask, mask_source, mask_detail = _infer_odoc_mask_local(row, IMAGE_SIZE) if "_infer_odoc_mask_local" in globals() else infer_od_oc_mask(row, IMAGE_SIZE)
        if isinstance(target_mask, tuple):
            target_mask = target_mask[0]
        fov_mask = _image_fov_mask(image_path, IMAGE_SIZE) if "_image_fov_mask" in globals() else np.ones((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)

        random_coords, diagnostics = _random_mask_coord_list(
            target_mask,
            fov_mask,
            k=OCCLUSION_RANDOM_K,
            max_iou=OCCLUSION_MAX_IOU,
            seed=_stable_seed("occlusion_random", image_id),
        ) if "_random_mask_coord_list" in globals() else ([], [])

        random_masks = []
        for coords in random_coords:
            m = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
            if coords is not None and len(coords) > 0:
                m[coords[:, 0], coords[:, 1]] = True
            random_masks.append(m)

        try:
            x = load_image_tensor(image_path, IMAGE_SIZE).to(DEVICE)
        except Exception as exc:
            print(f"Skipping image {image_id}: {type(exc).__name__}: {exc}")
            continue

        for model_name in MODEL_NAMES_EFFECTIVE if "MODEL_NAMES_EFFECTIVE" in globals() else MODEL_NAMES:
            if model_name not in models:
                continue
            model = models[model_name]

            try:
                occ_sal, p_original = _occlusion_saliency_map(model, x, grid=OCCLUSION_PATCH_GRID)
                occ_sal = np.maximum(occ_sal, 0)
                occ_sal = occ_sal / (occ_sal.sum() + 1e-8)
                observed = float(occ_sal[target_mask].sum()) if target_mask.any() else np.nan
                random_energy = [float(occ_sal[m].sum()) for m in random_masks if m.any()]
                random_mean = float(np.mean(random_energy)) if random_energy else np.nan
                enrichment_delta = observed - random_mean if np.isfinite(observed) and np.isfinite(random_mean) else np.nan

                occ_attr_rows.append({
                    "model_name": model_name,
                    "image_id": image_id,
                    "patient_id": patient_id,
                    "gon_binary": row.get("gon_binary", np.nan),
                    "quality_tertile": row.get("quality_tertile", ""),
                    "xai_method": "PatchOcclusion",
                    "patch_grid": int(OCCLUSION_PATCH_GRID),
                    "observed_odoc_energy": observed,
                    "matched_random_energy_mean": random_mean,
                    "odoc_minus_random_energy": enrichment_delta,
                    "p_original": p_original,
                    "mask_source": mask_source if isinstance(mask_source, str) else "",
                    "mask_source_detail": mask_detail if isinstance(mask_detail, str) else "",
                })

                # Deletion/insertion curves using occlusion ranking.
                curve_rows_one = []
                for curve_type in ["deletion", "insertion"]:
                    for strategy in ["saliency_guided", "matched_random"]:
                        for fraction in OCCLUSION_DELETION_FRACTIONS:
                            if strategy == "saliency_guided":
                                mask = _top_fraction_mask(occ_sal, fraction)
                            else:
                                if random_masks:
                                    # Random-control mask with area matched to top fraction, not OD/OC.
                                    rng = np.random.default_rng(_stable_seed("curve_random", image_id, model_name, curve_type, fraction))
                                    flat = np.zeros(IMAGE_SIZE * IMAGE_SIZE, dtype=bool)
                                    n = int(round(float(fraction) * flat.size))
                                    if n > 0:
                                        idx = rng.choice(flat.size, size=n, replace=False)
                                        flat[idx] = True
                                    mask = flat.reshape((IMAGE_SIZE, IMAGE_SIZE))
                                else:
                                    mask = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)

                            if curve_type == "deletion":
                                x_mod = apply_mask_mean_fill_tensor(x, mask) if "apply_mask_mean_fill_tensor" in globals() else x
                                p_mod = predict_probability(model, x_mod)
                                prob_delta = float(p_original - p_mod)
                            else:
                                x_base = apply_mask_mean_fill_tensor(x, np.ones((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)) if "apply_mask_mean_fill_tensor" in globals() else x
                                x_mod = restore_mask_from_original(x_base, x, mask) if "restore_mask_from_original" in globals() else x
                                p_mod = predict_probability(model, x_mod)
                                prob_delta = float(p_mod - predict_probability(model, x_base))

                            curve_rows_one.append({
                                "model_name": model_name,
                                "image_id": image_id,
                                "patient_id": patient_id,
                                "gon_binary": row.get("gon_binary", np.nan),
                                "quality_tertile": row.get("quality_tertile", ""),
                                "xai_method": "PatchOcclusion",
                                "curve_type": curve_type,
                                "strategy": strategy,
                                "fraction": float(fraction),
                                "probability_delta": prob_delta,
                            })

                occ_curve_rows.extend(curve_rows_one)

                for curve_type in ["deletion", "insertion"]:
                    guided_auc = _curve_auc_from_rows(curve_rows_one, curve_type, "saliency_guided")
                    random_auc = _curve_auc_from_rows(curve_rows_one, curve_type, "matched_random")
                    occ_auc_rows.append({
                        "model_name": model_name,
                        "image_id": image_id,
                        "patient_id": patient_id,
                        "gon_binary": row.get("gon_binary", np.nan),
                        "quality_tertile": row.get("quality_tertile", ""),
                        "xai_method": "PatchOcclusion",
                        "curve_type": curve_type,
                        "guided_auc": guided_auc,
                        "matched_random_auc": random_auc,
                        "auc_delta": guided_auc - random_auc if np.isfinite(guided_auc) and np.isfinite(random_auc) else np.nan,
                    })

            except Exception as exc:
                occ_attr_rows.append({
                    "model_name": model_name,
                    "image_id": image_id,
                    "patient_id": patient_id,
                    "xai_method": "PatchOcclusion",
                    "error": f"{type(exc).__name__}: {exc}",
                })

    occ_attr_image = pd.DataFrame(occ_attr_rows)
    occ_curve_image = pd.DataFrame(occ_curve_rows)
    occ_auc_image = pd.DataFrame(occ_auc_rows)

occ_attr_image.to_csv(OCCLUSION_ATTR_IMAGE_CSV, index=False)
occ_curve_image.to_csv(OCCLUSION_CURVE_IMAGE_CSV, index=False)
occ_auc_image.to_csv(OCCLUSION_AUC_IMAGE_CSV, index=False)

if not occ_attr_image.empty and "patient_id" in occ_attr_image.columns:
    occ_attr_patient = (
        occ_attr_image
        .dropna(subset=["odoc_minus_random_energy"])
        .groupby(["model_name", "patient_id", "gon_binary", "quality_tertile", "xai_method"], dropna=False)
        .agg(
            odoc_minus_random_energy=("odoc_minus_random_energy", "mean"),
            observed_odoc_energy=("observed_odoc_energy", "mean"),
            matched_random_energy_mean=("matched_random_energy_mean", "mean"),
            n_images=("image_id", "nunique")
        )
        .reset_index()
    )
else:
    occ_attr_patient = pd.DataFrame()

occ_attr_patient.to_csv(OCCLUSION_ATTR_PATIENT_CSV, index=False)

if not occ_curve_image.empty and "patient_id" in occ_curve_image.columns:
    occ_curve_patient = (
        occ_curve_image
        .groupby(["model_name", "patient_id", "gon_binary", "quality_tertile", "xai_method", "curve_type", "strategy", "fraction"], dropna=False)
        .agg(probability_delta=("probability_delta", "mean"), n_images=("image_id", "nunique"))
        .reset_index()
    )
else:
    occ_curve_patient = pd.DataFrame()

occ_curve_patient.to_csv(OCCLUSION_CURVE_PATIENT_CSV, index=False)

if not occ_auc_image.empty and "patient_id" in occ_auc_image.columns:
    occ_auc_patient = (
        occ_auc_image
        .groupby(["model_name", "patient_id", "gon_binary", "quality_tertile", "xai_method", "curve_type"], dropna=False)
        .agg(auc_delta=("auc_delta", "mean"), n_images=("image_id", "nunique"))
        .reset_index()
    )
else:
    occ_auc_patient = pd.DataFrame()

occ_auc_patient.to_csv(OCCLUSION_AUC_PATIENT_CSV, index=False)

summary_blocks = []
if not occ_attr_patient.empty:
    summary_blocks.append(
        _summarize(
            occ_attr_patient,
            value_col="odoc_minus_random_energy",
            group_cols=["model_name", "xai_method"],
            summary_name="patient_level_occlusion_odoc_enrichment",
        )
    )
if not occ_auc_patient.empty:
    summary_blocks.append(
        _summarize(
            occ_auc_patient,
            value_col="auc_delta",
            group_cols=["model_name", "xai_method", "curve_type"],
            summary_name="patient_level_occlusion_auc_delta",
        )
    )

occ_summary = pd.concat([x for x in summary_blocks if x is not None and not x.empty], ignore_index=True) if summary_blocks else pd.DataFrame()
occ_summary.to_csv(OCCLUSION_ATTR_SUMMARY_CSV, index=False)
occ_summary.to_csv(OCCLUSION_AUC_SUMMARY_CSV, index=False)

if not occ_auc_patient.empty and "auc_delta" in occ_auc_patient.columns:
    auc_abs_sum = float(pd.to_numeric(occ_auc_patient["auc_delta"], errors="coerce").abs().sum())
    if auc_abs_sum == 0.0:
        print("WARNING: Occlusion insertion/deletion AUC deltas are exactly zero. Check masking helpers before using these AUC results in the manuscript.")


handoff = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 18",
    "status": "PASS",
    "purpose": "Secondary occlusion-based attribution robustness check.",
    "configuration": {
        "xai_method": "PatchOcclusion",
        "images_per_label_quality": OCCLUSION_IMAGES_PER_LABEL_QUALITY,
        "patch_grid": OCCLUSION_PATCH_GRID,
        "fractions": OCCLUSION_DELETION_FRACTIONS,
        "random_k": OCCLUSION_RANDOM_K,
        "max_iou": OCCLUSION_MAX_IOU,
    },
    "output_tables": {
        "occlusion_attr_image": str(OCCLUSION_ATTR_IMAGE_CSV),
        "occlusion_attr_patient": str(OCCLUSION_ATTR_PATIENT_CSV),
        "occlusion_curve_image": str(OCCLUSION_CURVE_IMAGE_CSV),
        "occlusion_curve_patient": str(OCCLUSION_CURVE_PATIENT_CSV),
        "occlusion_auc_image": str(OCCLUSION_AUC_IMAGE_CSV),
        "occlusion_auc_patient": str(OCCLUSION_AUC_PATIENT_CSV),
        "occlusion_summary": str(OCCLUSION_ATTR_SUMMARY_CSV),
    },
    "n_image_rows": int(len(occ_attr_image)),
    "n_patient_rows": int(len(occ_attr_patient)),
    "n_curve_rows": int(len(occ_curve_image)),
    "n_auc_rows": int(len(occ_auc_image)),
    "interpretation_guardrail": (
        "Patch occlusion is a secondary attribution-sensitivity check on a balanced subset; "
        "Integrated Gradients remains the primary all-HYGD ranking method."
    ),
}

with open(OCCLUSION_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(handoff, f, indent=2)

print("\nSecondary occlusion attribution summary")
display(occ_summary)

print("\nCell 18 completed.")
print(json.dumps(handoff, indent=2))


In [ ]:

# Cell 19: Final v3.6 robustness handoff for Notebook 16
#
# Purpose:
# - Consolidate new reviewer-requested robustness outputs from Cells 16-18.
# - Do not recompute anything.
# - Provide a single handoff file for Notebook 16 and manuscript v3.6.

import json
from pathlib import Path
import pandas as pd
import numpy as np

V36_HANDOFF_JSON = RESULTS_15_DIR / "notebook15_v36_robustness_handoff_for_notebook16.json"
V36_ACTION_ITEMS_CSV = TABLE_DIR / "notebook15_v36_outputs_for_notebook16.csv"
V36_REVIEWER_RESPONSE_MATRIX_CSV = TABLE_DIR / "notebook15_v36_reviewer_response_matrix.csv"

def _exists(path):
    p = Path(path)
    return p.exists() and p.stat().st_size > 0

def _csv_has_data_rows(path):
    p = Path(path)
    if not (p.exists() and p.stat().st_size > 0):
        return False
    try:
        return len(pd.read_csv(p, low_memory=False)) > 0
    except Exception:
        return False

def _row(name, path, purpose, ready_required=True, require_data_rows=False):
    exists = bool(_exists(path))
    has_rows = bool(_csv_has_data_rows(path)) if str(path).lower().endswith('.csv') else exists
    ready = has_rows if require_data_rows else (exists if ready_required else True)
    return {
        "output_name": name,
        "path": str(path),
        "exists": exists,
        "has_data_rows": has_rows,
        "purpose": purpose,
        "ready_for_notebook16": bool(ready),
    }

outputs = [
    _row(
        "endpoint_parameter_table",
        TABLE_DIR / "notebook15_aqpr_endpoint_parameter_table.csv",
        "Supplement/main reproducibility table for saliency, matched-random, insertion/deletion, perturbation, and bootstrap parameters.",
    ),
    _row(
        "auc_delta_formula_table",
        TABLE_DIR / "notebook15_insertion_deletion_auc_delta_formula_table.csv",
        "Formal endpoint definitions and AUC-delta computation details.",
    ),
    _row(
        "full_insertion_deletion_curves_patient",
        TABLE_DIR / "notebook15_full_insertion_deletion_curve_points_patient_level.csv",
        "Full patient-level insertion/deletion curve points for supplement.",
    ),
    _row(
        "matched_random_saliency_ablation_summary",
        TABLE_DIR / "notebook15_matched_random_ablation_saliency_summary.csv",
        "K/IoU robustness ablation for OD/OC saliency enrichment.",
    ),
    _row(
        "matched_random_perturbation_ablation_summary",
        TABLE_DIR / "notebook15_matched_random_ablation_perturbation_summary.csv",
        "K/IoU robustness ablation for OD/OC-minus-random perturbation delta.",
        ready_required=True,
        require_data_rows=True,
    ),
    _row(
        "secondary_occlusion_summary",
        TABLE_DIR / "notebook15_secondary_occlusion_attribution_summary.csv",
        "Secondary occlusion-based attribution robustness check.",
        ready_required=True,
        require_data_rows=True,
    ),
    _row(
        "expanded_claim_boundary_full",
        TABLE_DIR / "notebook15_expanded_model_claim_boundary_full.csv",
        "Expanded all-HYGD claim-boundary table.",
    ),
]

outputs_df = pd.DataFrame(outputs)
outputs_df.to_csv(V36_ACTION_ITEMS_CSV, index=False)

reviewer_rows = [
    {
        "reviewer_question": "Ablation on matched-random mask generator K/IoU/FOV.",
        "response_status": "addressed",
        "evidence_output": str(TABLE_DIR / "notebook15_matched_random_ablation_saliency_summary.csv"),
        "manuscript_use": "Briefly report directional stability; place full table in supplement.",
    },
    {
        "reviewer_question": "AUC-delta formula and full insertion/deletion curves.",
        "response_status": "addressed",
        "evidence_output": str(TABLE_DIR / "notebook15_insertion_deletion_auc_delta_formula_table.csv"),
        "manuscript_use": "Add AUC-delta definition in Methods/Supplement; cite full curves in supplement.",
    },
    {
        "reviewer_question": "Sensitivity to attribution method.",
        "response_status": "addressed_with_secondary_subset_check" if _csv_has_data_rows(TABLE_DIR / "notebook15_secondary_occlusion_attribution_summary.csv") else "pending_or_skipped",
        "evidence_output": str(TABLE_DIR / "notebook15_secondary_occlusion_attribution_summary.csv"),
        "manuscript_use": "Report as secondary robustness check, not as replacement for IG primary audit.",
    },
    {
        "reviewer_question": "Perturbation delta robustness to matched-random settings.",
        "response_status": "addressed_with_sensitivity_subset" if _csv_has_data_rows(TABLE_DIR / "notebook15_matched_random_ablation_perturbation_summary.csv") else "pending_or_skipped",
        "evidence_output": str(TABLE_DIR / "notebook15_matched_random_ablation_perturbation_summary.csv"),
        "manuscript_use": "Report as small matched-random sensitivity analysis; main perturbation remains all-HYGD fixed setting.",
    },
    {
        "reviewer_question": "Clarify saliency/perturbation hyperparameters.",
        "response_status": "addressed",
        "evidence_output": str(TABLE_DIR / "notebook15_aqpr_endpoint_parameter_table.csv"),
        "manuscript_use": "Use parameter table in supplement and one concise sentence in main text.",
    },
]

reviewer_matrix = pd.DataFrame(reviewer_rows)
reviewer_matrix.to_csv(V36_REVIEWER_RESPONSE_MATRIX_CSV, index=False)

handoff = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 19",
    "status": "PASS",
    "purpose": "Final v3.6 Notebook 15 robustness handoff for Notebook 16.",
    "output_inventory": outputs,
    "ready_outputs": int(outputs_df["ready_for_notebook16"].sum()),
    "total_outputs": int(len(outputs_df)),
    "reviewer_response_matrix": str(V36_REVIEWER_RESPONSE_MATRIX_CSV),
    "action_items_csv": str(V36_ACTION_ITEMS_CSV),
    "interpretation_guardrails": [
        "Matched-random ablations are sensitivity analyses, not new model selection.",
        "Occlusion is a secondary attribution check on a balanced subset; IG remains the primary all-HYGD ranking method.",
        "Perturbation matched-random ablation may use a sensitivity subset because it requires many additional model inferences.",
    ],
}

with open(V36_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(handoff, f, indent=2)

print("\nNotebook 15 v3.6 robustness outputs for Notebook 16")
display(outputs_df)

print("\nReviewer-response matrix")
display(reviewer_matrix)

print("\nCell 19 completed.")
print(json.dumps(handoff, indent=2))
